# PKG Attrition v13 · Model 4 (customer metrics v3) + Treasury Management briefing

**Payment Knowledge Graph (PKG) · Treasury Management · DSI**

This run is model 4 unchanged, on `pkg_cust_metrics_monthly_v3` (silent month = no payments). §13–14 are new: they compute the
numbers for the business briefing *Deposit Attrition Early Warning: Initial Findings* (balance trajectory of leavers vs comparable
stayers, six signal groups, lead time per relationship, performance at review-list sizes 25 – 1,000, three anonymised examples)
and write it as `PKG_Deposit_Attrition_Early_Warning_Initial_Findings.html` next to the technical report.

Model 3 settled the model: a LightGBM tree beats the simple ridge model by about 0.014 AUC in every test month.
Model 4 keeps the features, folds and models unchanged and fixes what model 3 showed in its examples, calibration
and departure check. It also drafts section 1 of the report.

| | Fix | Why |
|---|---|---|
| 1 | Tree probabilities calibrated on held-out clients from the **3 most recent labelled months**, capped at 95% | calibration on the whole training period did not carry forward (top decile 22.6% predicted vs 17.4% actual; one client at 100%) |
| 2 | **"Moved to another bank"** needs payments to the client's own name at other banks **above its own earlier level** by ≥ 50% of the drop | some clients routinely pay themselves at another bank; a habitual sweeper is not a mover |
| 3 | **Sustained alert** = the first of ≥ 2 consecutive months on the list; one-month blips reported separately | a single month at rank 657 followed by five months off the list is not a warning an RM acts on |
| 4 | Examples need **12 steady months** before the alert; clients whose money arrived recently get their own example, *new money that left* | model 3's "early catch" was USD 3bn that arrived from nothing and left 8 months later; its triggers were log-of-zero artefacts |
| 5 | Example trigger panels show **change features with a stayers' threshold only**; levels stay in the reason text | "payment flow, log = +18.12" means nothing to an RM |
| 6 | Output blocks are excluded from maths rendering | JupyterLab rendered text between two dollar signs as a formula |
| 7 | **Section 1 ("What we found") is drafted** from the results | the numbers are stable across models 2–4 |

Sources: the deposit table and `pkg_cust_metrics_monthly`; no table is written.

### Pre-registered predictions — scored in §12

| # | Prediction | Confidence |
|---|---|---|
| P1 | Calibrated top decile: predicted ÷ actual between 0.8 and 1.25 | 55% |
| P2 | Tree AUC ≥ simple + 0.01 | 80% |
| P3 | With the self-pay baseline, fewer than 12 of the checked departures are "moved to another bank" | 60% |
| P4 | ≥ 30% of reached clients have a sustained alert 3+ months before the money left | 50% |
| P5 | ≥ 15% of reached clients were reached only by one-month blips | 40% |
| P6 | Dollars reached at 1,000 calls, α = 0.5, within ±5% of model 3's USD 11.82bn | 70% |
| P7 | Best month's precision ≤ 2× worst (tree, 1,000 calls, α = 0.5) | 65% |

In [ ]:
# =====================================================================
# 0 · CONFIG
# =====================================================================
DEP_TABLE  = "dsihd01p_dsi.lap_dsi_universe_optimized"          # daily: account × leg × business day
MET_TABLE  = "bdahd01p_dlcdi1_cdi_tm.pkg_cust_metrics_monthly_v3"
DICT_TABLE = "bdahd01p_dlcdi1_cdi_tm.pkg_cust_metrics_dictionary_v3"
OUT_DIR    = "../metrics/attrition/v13/m4_v3_tm"
DATE_START, DATE_END = "2024-01-01", "2026-08-31"                # nothing before 2024 is trusted
SHUFFLE_PARTITIONS = 800

# ── status codes: two vocabularies in one column (settled in v2) ─────
STATUS_CLASS = {"01": "open", "99": "open", "09": "open", "O": "open",
                "03": "inactive", "05": "dormant", "06": "escheat", "12": "closing",
                "07": "closed", "08": "closed", "C": "closed"}
LIVE_CLASSES = {"open", "inactive", "dormant", "escheat", "closing"}
PRICING = ["earnings_credit_rate", "curr_int_rate"]              # used if present

# ── definitions ──────────────────────────────────────────────────────
H          = 6          # horizon: does the money leave within the next 6 months?
CONFIRM_M  = 2          # a full exit is confirmed after 2 observed non-live months
NORM_WIN, NORM_MIN_OBS = 12, 3    # normal balance: median of up to 12 prior live months, needs 3
MIN_HIST   = 3          # months observed before a client-month can be scored
MSH_FRAC, MSH_FLOOR = 0.50, 10_000     # money still here: ≥ half of normal, normal ≥ $10k
BAL_EXIT_FRAC, BAL_EXIT_HOLD = 0.05, 3 # balance exit: < 5% of normal for 3 months, accounts open
CHURN_WARM = 6          # months of payment history before the churn-state metrics are used

# ── training ─────────────────────────────────────────────────────────
NEG_RATE   = 0.15       # share of non-leaving client-months kept for training (weight 1/NEG_RATE)
SEED       = 13
L2_GRID    = [1.0, 10.0, 100.0, 1000.0]   # ridge penalty (summed log-likelihood), chosen on the first fold's validation
CLIP_Q     = (0.005, 0.995)
CORR_MAX   = 0.95       # drop the later of two features correlated above this
TEST_FIRST = "2025-04"  # first test month; the last is the last month with a fully observed outcome
ENGINE     = "auto"     # "auto" | "xgboost" | "lightgbm" | "sklearn" | "off"
HOLD_FRAC  = 0.20       # share of clients held out inside the training window (tuning, early stopping, calibration)
CAL_RECENT = 3          # calibrate the tree on held-out clients from the most recent labelled months only
P_CAP      = 0.95       # calibrated probabilities are capped here
GBM_GRID   = [4, 6, 8]  # tree depth tried on the first fold's validation
GBM_ETA, GBM_MAX_TREES, GBM_EARLY = 0.05, 3000, 100

# ── queue and value ──────────────────────────────────────────────────
CALLS      = [100, 250, 500, 1000, 2000]   # calls per month
ALPHAS     = [0.0, 0.5, 1.0]               # score = p × normal balance^α
SAVE_RATES = [0.01, 0.05, 0.10, 0.20]      # illustrative; no call cost anywhere
K_MAIN, A_MAIN = 1000, 0.5
BANDS      = [(0, 1e5, "< $100k"), (1e5, 1e6, "$100k–1m"), (1e6, 1e7, "$1m–10m"), (1e7, float("inf"), "$10m+")]

# ── signal timing and examples ───────────────────────────────────────
N_TIMING   = 30         # features traced month by month before exit
LIFT_BAR   = 1.5        # a signal "fires" when leavers cross the stayers' 10th/90th percentile ≥ 1.5× as often
EVENT_PRE  = 12
MAX_EVENT_CUST = 15_000 # per group, for the event-time pulls
N_SHAP_CUST = 5_000     # per group, for the model view
LATE_FEATS = ["dep_eom_vs_avg", "dep_d1"]   # drain already in progress: late warnings
N_AUDIT    = 15         # largest departures checked, from the whole window and from the test months each
TOP_EXCL   = [1, 5]     # headline dollars also shown without the largest 1 and 5 departing clients
CAP_CREDIT = 250e6      # headline also shown with each client's credit capped here
N_TRIG     = 4          # trigger features shown per example client
STEADY_M   = 12         # months of steady balance (≥ 10% of normal every month) required before an example's alert

# ── Treasury Management briefing (§13–14) ────────────────────────────
BRIEF_SIZES      = [25, 50, 100, 250, 500, 1000]   # monthly review-list sizes shown (analysis only, not a recommendation)
BRIEF_LEAD_SIZES = (1000, 250)                     # lead-time table: the first is the reference used in the text
TRJ_PRE, TRJ_POST = 12, 2                          # balance trajectory: months before / after the event
TRJ_GAP, TRJ_GAP_PAY = 0.05, 0.10                  # leavers "separate" when ≥ 5% (balance) / 10% (payments) below stayers through month −1

In [ ]:
# =====================================================================
# 1 · SETUP — shims, session, helpers
# =====================================================================
import numpy as np, warnings
for _a, _t in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _a): setattr(np, _a, _t)
if "bool" not in np.__dict__: np.bool = np.bool_
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"): pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):    pd.Series.iteritems = pd.Series.items
for _c in (ResourceWarning, DeprecationWarning, FutureWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=_c)

import os, re, time, json, base64, hashlib, pathlib, html as _html
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")            # features are cast to float before collecting
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
DISK = StorageLevel.DISK_ONLY
OUT = pathlib.Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figdata"; FIG.mkdir(exist_ok=True)
def figdata(name, df):
    """Data behind a figure, so the report can be rebuilt later without rerunning."""
    df.to_csv(FIG / f"{name}.csv", index=False)
T0 = time.time()
R = {}          # every number the report uses
CHARTS = {}     # chart name → file
CHECKS = []

# ── one consolidated output per cell ─────────────────────────────────
def _fmt_col(s, kind):
    if kind == "pct":   return s.map(lambda v: "—" if pd.isna(v) else f"{v:.1%}")
    if kind == "money": return s.map(lambda v: "—" if pd.isna(v) else f"${v:,.0f}")
    if pd.api.types.is_integer_dtype(s): return s.map(lambda v: f"{v:,}")
    if pd.api.types.is_float_dtype(s):   return s.map(lambda v: "—" if pd.isna(v) else f"{v:,.3f}")
    return s.map(lambda v: "—" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))

def fmt_df(df, pct=(), money=()):
    d = df.copy()
    for c in d.columns:
        d[c] = _fmt_col(d[c], "pct" if c in pct else "money" if c in money else None)
    return d

class Block:
    """Collects every table and note of a cell and displays them once, as one image-friendly output."""
    def __init__(self, title):
        # tex2jax_ignore / mathjax_ignore: JupyterLab must not read text between two $ signs as a formula
        self.h = [f"<div class='tex2jax_ignore mathjax_ignore' style='font-family:Arial,sans-serif;font-size:12px'>"
                  f"<h3 style='margin:2px 0 6px'>{title}</h3>"]
    def table(self, df, title, pct=(), money=(), save=None, max_rows=80):
        if save: df.to_csv(OUT / f"{save}.csv", index=False)
        d = fmt_df(df.head(max_rows), pct, money)
        self.h.append(f"<p style='margin:8px 0 2px'><b>{title}</b> <span style='color:#777'>[{len(df):,} rows]</span></p>" +
                      d.to_html(index=False, border=0, classes="t").replace(
                          "<table", "<table style='border-collapse:collapse;font-size:11px' cellpadding='3'"))
        return self
    def kv(self, pairs, title):
        return self.table(pd.DataFrame([{"item": k, "value": v if isinstance(v, str) else
                          f"{v:,}" if isinstance(v, (int, np.integer)) else f"{v:,.4f}" if isinstance(v, float) else str(v)}
                                        for k, v in pairs]), title)
    def note(self, s):
        self.h.append(f"<p style='margin:4px 0;color:#333'>{s}</p>"); return self
    def check(self, item, status, detail):
        CHECKS.append({"item": item, "status": status, "detail": detail})
        col = {"READY": "#2e7d32", "FIX": "#e65100", "BLOCKER": "#b71c1c", "INFO": "#555"}[status]
        return self.note(f"<span style='color:{col};font-weight:bold'>[{status}]</span> {item}: {detail}")
    def show(self, t0=None):
        if t0: self.note(f"<span style='color:#999'>wall {time.time() - t0:,.0f}s</span>")
        display(HTML("".join(self.h) + "</div>"))

def savefig(fig, name):
    p = OUT / f"{name}.png"; fig.savefig(p, dpi=120, bbox_inches="tight"); plt.close(fig); CHARTS[name] = p

def topd(df):
    for c, t in df.dtypes:
        if t.startswith("decimal"): df = df.withColumn(c, F.col(c).cast("double"))
    return df.toPandas()

def nz(c):
    s = F.trim(F.col(c).cast("string")); return F.when(s == "", F.lit(None)).otherwise(s)

# ── months: offset t from the first month only ───────────────────────
def ym_idx(ym): y, m = map(int, ym[:7].split("-")); return y * 12 + m - 1
M0 = ym_idx(DATE_START)
def t_of(ym): return ym_idx(ym) - M0
def ym_of(t): i = M0 + int(t); return f"{i // 12:04d}-{i % 12 + 1:02d}"
def t_col(c): return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - 1 - F.lit(M0)).cast("int")
assert t_of(DATE_START) == 0 and ym_of(0) == DATE_START[:7]

def cid(x):
    """Display id: one-way hash. Full ids are only ever written to *_FULL_IDS.csv."""
    return hashlib.sha1(str(x).encode()).hexdigest()[:10]

def auc(y, s):
    """Rank AUC with ties averaged; y in {0,1}."""
    y = np.asarray(y, float); s = np.asarray(s, float); ok = ~np.isnan(s); y, s = y[ok], s[ok]
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

# ── tree engine ──────────────────────────────────────────────────────
def _engine():
    order = ["xgboost", "lightgbm", "sklearn"] if ENGINE == "auto" else ([] if ENGINE == "off" else [ENGINE])
    for e in order:
        try:
            importlib.import_module(e if e != "sklearn" else "sklearn.ensemble"); return e
        except Exception: pass
    return None
import importlib
GBM = _engine()
print(f"Spark {spark.version} | window {DATE_START} … {DATE_END} | tree engine: {GBM or 'none (simple model only)'} "
      f"| cpus {os.cpu_count()} | outputs → {OUT.resolve()}")

## §2 Deposit data → monthly customer panel, targets, deposit features

Legs are summed per account-day; balances are rolled to customer-day, then month (average, minimum, maximum,
month-end). Every customer gets a complete month grid from its first month, so a month with no rows reads as
non-live with zero balance. All windows look backwards only.

**Target** (defined and argued in the report, §4): *the money leaves within the next 6 months*, the earlier of
a **full exit** (every account non-live, confirmed by 2 observed non-live months) and a **balance exit** (balance
below 5% of normal for 3 months while accounts stay open).

In [ ]:
# =====================================================================
# 2 · DEPOSIT PANEL, TARGETS, DEPOSIT FEATURES                  [OUTPUT BLOCK 2]
# =====================================================================
t0 = time.time(); B = Block("§2 · Deposit data → monthly customer panel")
raw = spark.table(DEP_TABLE)
REQ = ["acct_full_acct_id", "sub_product_cd", "edw_tda_load_dt", "balance", "acct_status", "deposit_family",
       "opened_dt", "closed_dt", "cust_pwr_id", "rltn_pwr_id"]
miss = [c for c in REQ if c not in raw.columns]
if miss: raise RuntimeError(f"deposit table lacks {miss}")
PRC = [c for c in PRICING if c in raw.columns]
TIEB = [c for c in ["bdh_hdfs_load_ts", "cod_hdfs_load_ts"] if c in raw.columns]
SMAP = F.create_map(*[F.lit(x) for kv_ in STATUS_CLASS.items() for x in kv_])
dep = raw.select(*REQ, *PRC, *TIEB)
for c in ["acct_full_acct_id", "sub_product_cd", "cust_pwr_id", "rltn_pwr_id", "acct_status", "deposit_family"]:
    dep = dep.withColumn(c, nz(c))
for c in ["edw_tda_load_dt", "opened_dt", "closed_dt"]: dep = dep.withColumn(c, F.to_date(F.col(c)))
for c in ["balance"] + PRC: dep = dep.withColumn(c, F.col(c).cast("double"))
dep = (dep.filter(F.col("edw_tda_load_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
          .withColumn("leg", F.coalesce(F.col("sub_product_cd"), F.lit("__NA__")))
          .withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM"))
          .withColumn("status_class", F.coalesce(SMAP[F.col("acct_status")], F.lit("UNMAPPED")))
          .withColumn("live_leg", F.col("status_class").isin(*LIVE_CLASSES).cast("int")))

unm = [r[0] for r in dep.filter("status_class = 'UNMAPPED'").select("acct_status").distinct().limit(20).collect()]
if unm: raise RuntimeError(f"unmapped acct_status codes {unm}: add them to STATUS_CLASS")
n_dup = dep.groupBy("acct_full_acct_id", "leg", "edw_tda_load_dt").count().filter("count > 1").count()
if n_dup:
    _o = [F.col(c).desc_nulls_last() for c in TIEB] + [F.col("balance").desc_nulls_last()]
    dep = dep.withColumn("_rn", F.row_number().over(Window.partitionBy("acct_full_acct_id", "leg", "edw_tda_load_dt")
                                                    .orderBy(*_o))).filter("_rn = 1").drop("_rn")
MC = topd(dep.groupBy("ym").agg(F.countDistinct("edw_tda_load_dt").alias("bdays"),
                                F.countDistinct("cust_pwr_id").alias("customers")).orderBy("ym"))
END_M = MC.loc[MC.bdays >= 15, "ym"].max(); END_T = t_of(END_M)
holes = [ym_of(t) for t in range(END_T + 1) if ym_of(t) not in set(MC.loc[MC.bdays >= 15, "ym"])]
if holes: raise RuntimeError(f"deposit months missing or partial inside the window: {holes}")
dep = dep.withColumn("t", t_col("ym")).filter(F.col("t") <= END_T)

# ── customer-month balances (customer-day first, so daily min / max are real) ──
cday = (dep.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "edw_tda_load_dt", "t")
           .agg(F.sum("balance").alias("bal_d")))
cbal = cday.groupBy("cust_pwr_id", "t").agg(F.avg("bal_d").alias("bal_avg"), F.min("bal_d").alias("bal_min"),
                                            F.max("bal_d").alias("bal_max"),
                                            F.expr("max_by(bal_d, edw_tda_load_dt)").alias("bal_eom"))
# ── account-month → customer-month counts, product mix, pricing ─────
am = dep.groupBy("acct_full_acct_id", "t", "ym").agg(
    F.max("live_leg").alias("live"),
    F.expr("max_by(cust_pwr_id, edw_tda_load_dt)").alias("cust_pwr_id"),
    F.expr("max_by(rltn_pwr_id, edw_tda_load_dt)").alias("rltn_pwr_id"),
    F.expr("max_by(deposit_family, abs(coalesce(balance, 0)))").alias("deposit_family"),
    (F.sum("balance") / F.countDistinct("edw_tda_load_dt")).alias("acct_bal"),
    F.min("opened_dt").alias("opened_dt"),
    F.max((F.date_format("closed_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("closed_in_m"),
    *[F.expr(f"max_by({c}, edw_tda_load_dt)").alias(c) for c in PRC])
fam = F.upper(F.coalesce(F.col("deposit_family"), F.lit("")))
pos = F.greatest(F.col("acct_bal"), F.lit(0.0))
fb = lambda cond: F.sum(F.when(cond, pos).otherwise(0.0))
def wavg(c):
    ok = F.col(c).isNotNull() & (F.col("acct_bal") > 0)
    return (F.sum(F.when(ok, F.col(c) * F.col("acct_bal"))) / F.sum(F.when(ok, F.col("acct_bal")))).alias(c)
cacc = am.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "t").agg(
    F.count("*").alias("n_accts"), F.sum("live").alias("n_live"),
    F.sum((F.date_format("opened_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("n_opened"),
    F.sum("closed_in_m").alias("n_closed"),
    fb(fam == "IBDDA").alias("fb_ibdda"), fb(fam == "MMDA").alias("fb_mmda"), fb(fam.contains("SWEEP")).alias("fb_sweep"),
    fb(F.lit(True)).alias("fb_all"), F.max("rltn_pwr_id").alias("rltn_pwr_id"), *[wavg(c) for c in PRC])
cm = cbal.join(cacc, ["cust_pwr_id", "t"], "full")

# ── complete month grid ──────────────────────────────────────────────
first = cm.groupBy("cust_pwr_id").agg(F.min("t").alias("first_t"))
grid = first.crossJoin(F.broadcast(spark.range(0, END_T + 1).select(F.col("id").cast("int").alias("t")))) \
            .filter(F.col("t") >= F.col("first_t"))
X = grid.join(cm, ["cust_pwr_id", "t"], "left")
for c in ["n_accts", "n_live", "n_opened", "n_closed"]: X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
for c in ["bal_avg", "bal_min", "bal_max", "bal_eom", "fb_ibdda", "fb_mmda", "fb_sweep", "fb_all"]:
    X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))
wc = Window.partitionBy("cust_pwr_id"); wo = wc.orderBy("t")
rg = lambda a, b: wo.rangeBetween(a, b)
X = (X.withColumn("rltn_pwr_id", F.last("rltn_pwr_id", ignorenulls=True).over(wo.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("bal", F.greatest(F.col("bal_avg"), F.lit(0.0)))
      .withColumn("live", (F.col("n_live") > 0).cast("int"))
      .withColumn("hist_t", F.col("t") - F.col("first_t") + 1)
      .withColumn("last_live_t", F.max(F.when(F.col("live") == 1, F.col("t"))).over(wc))
      .withColumn("exit_t", F.when(F.col("last_live_t") < END_T, F.col("last_live_t") + 1).cast("int"))
      .withColumn("a_t", F.when(F.lit(END_T) - F.col("exit_t") + 1 >= CONFIRM_M, F.col("exit_t")))
      .withColumn("_bl", F.when(F.col("live") == 1, F.col("bal")))
      .withColumn("bal_norm", F.expr("percentile_approx(_bl, 0.5)").over(rg(-NORM_WIN, -1)))
      .withColumn("_nn", F.count("_bl").over(rg(-NORM_WIN, -1)))
      .withColumn("bal_norm", F.when(F.col("_nn") >= NORM_MIN_OBS, F.col("bal_norm")))
      .withColumn("msh", F.coalesce(((F.col("live") == 1) & (F.col("bal_norm") >= MSH_FLOOR)
                                     & (F.col("bal") >= MSH_FRAC * F.col("bal_norm"))).cast("int"), F.lit(0)))
      .withColumn("capped", F.least(F.col("bal"), F.col("bal_norm"))))
thr = F.lit(BAL_EXIT_FRAC) * F.col("bal_norm")
bc = (F.col("live") == 1) & (F.col("bal_norm") > 0) & (F.col("bal") < thr) & (F.col("t") + BAL_EXIT_HOLD - 1 <= END_T)
for k in range(1, BAL_EXIT_HOLD): bc = bc & (F.lead("bal", k).over(wo) < thr)
X = (X.withColumn("b_t", F.min(F.when(F.coalesce(bc, F.lit(False)), F.col("t"))).over(wc).cast("int"))
      .withColumn("ab_t", F.least("a_t", "b_t"))
      .withColumn("exit_type", F.when(F.col("ab_t").isNull(), F.lit(None))
                                .when(F.col("a_t") == F.col("ab_t"), F.lit("full exit")).otherwise(F.lit("balance exit"))))

# ── deposit features (backward-looking only) ─────────────────────────
def chg(col, a, b, min_obs=2):
    return F.when(F.count(col).over(rg(a, b)) >= min_obs, F.col(col) - F.avg(col).over(rg(a, b)))
X = X.withColumn("_lb", F.log1p("bal")).withColumn("_nl", F.col("n_live").cast("double"))
DEP = {}
def add(name, expr, group, label):
    global X; X = X.withColumn(name, expr); DEP[name] = (group, label)
add("dep_log_bal", F.col("_lb"), "balance level", "balance, log")
add("dep_bal_vs_norm", F.col("_lb") - F.log1p("bal_norm"), "balance level", "balance vs normal balance (log ratio)")
add("dep_d1", F.col("_lb") - F.lag("_lb", 1).over(wo), "balance change", "balance change vs last month")
add("dep_d3", chg("_lb", -3, -1), "balance change", "balance change vs the last 3 months")
add("dep_d6", chg("_lb", -6, -4), "balance change", "balance change vs 4–6 months ago")
add("dep_d12", chg("_lb", -12, -10), "balance change", "balance change vs 10–12 months ago")
add("dep_drawdown", F.col("_lb") - F.max("_lb").over(rg(-12, -1)), "balance change", "balance vs its 12-month peak")
add("dep_vol6", F.when(F.count("_lb").over(rg(-5, 0)) >= 3, F.stddev("_lb").over(rg(-5, 0))), "balance volatility",
    "volatility of balance over 6 months")
add("dep_range", F.log1p(F.greatest("bal_max", F.lit(0.0))) - F.log1p(F.greatest("bal_min", F.lit(0.0))),
    "balance volatility", "within-month range of daily balance")
add("dep_eom_vs_avg", F.log1p(F.greatest("bal_eom", F.lit(0.0))) - F.col("_lb"), "balance volatility",
    "month-end balance vs monthly average")
add("dep_n_accts", F.log1p("n_accts"), "accounts", "number of accounts")
add("dep_n_live", F.log1p("n_live"), "accounts", "number of live accounts")
add("dep_live_d6", chg("_nl", -6, -4), "accounts", "change in live accounts vs 4–6 months ago")
add("dep_opened_3m", F.sum("n_opened").over(rg(-2, 0)), "accounts", "accounts opened in the last 3 months")
add("dep_closed_3m", F.sum("n_closed").over(rg(-2, 0)), "accounts", "accounts closed in the last 3 months")
for c, lab in [("fb_ibdda", "interest-bearing DDA"), ("fb_mmda", "money market"), ("fb_sweep", "sweep")]:
    add(f"dep_share_{c[3:]}", F.when(F.col("fb_all") > 0, F.col(c) / F.col("fb_all")), "product mix", f"share of balance in {lab}")
add("dep_tenure", F.log1p("hist_t"), "tenure", "months observed (log)")
add("dep_at_start", (F.col("first_t") == 0).cast("int"), "tenure", "client already present when the data starts")
for c in PRC:
    add(f"dep_{c}", F.col(c), "pricing", c.replace("_", " "))
    add(f"dep_{c}_d6", chg(c, -6, -4), "pricing", f"change in {c.replace('_', ' ')} vs 4–6 months ago")
rl = X.filter("rltn_pwr_id is not null").groupBy("rltn_pwr_id", "t").agg(F.sum("bal").alias("rl_bal"),
                                                                         F.countDistinct("cust_pwr_id").alias("rl_n"))
X = X.join(rl, ["rltn_pwr_id", "t"], "left").withColumn(
    "_lo", F.when(F.col("rl_n") > 1, F.log1p(F.greatest(F.col("rl_bal") - F.col("bal"), F.lit(0.0)))))
add("dep_siblings", F.log1p(F.coalesce(F.col("rl_n"), F.lit(1)) - 1), "relationship", "other customers in the same relationship")
add("dep_sib_d6", chg("_lo", -6, -4), "relationship", "change in the relationship's other balances vs 4–6 months ago")
DEP_FEATS = list(DEP)
X = X.drop("_bl", "_nn", "_nl", "_lo").persist(DISK)
AMP = am.select("acct_full_acct_id", "t", "cust_pwr_id", "live", "acct_bal").persist(DISK)   # for the §7 checks

# ── summary ──────────────────────────────────────────────────────────
g = X.agg(F.count("*").alias("rows"), F.countDistinct("cust_pwr_id").alias("customers"),
          F.sum("live").alias("live_rows"), F.sum("msh").alias("msh_rows")).first().asDict()
lab = topd(X.groupBy("cust_pwr_id").agg(F.first("exit_type").alias("exit_type"), F.first("exit_t").alias("exit_t"),
                                        F.first("a_t").alias("a_t"), F.first("b_t").alias("b_t"))
            .groupBy("exit_type").agg(F.count("*").alias("customers"),
                                      F.sum(F.col("exit_t").isNotNull().cast("int")).alias("also_full_exit_or_provisional"))
            .orderBy("exit_type"))
# is a full exit absorbing?  P(client comes back | ≥ CONFIRM_M observed non-live months)
nl = X.filter((F.col("live") == 0) & (F.col("t") > F.col("first_t")))
nl = nl.withColumn("_g", F.col("t") - F.row_number().over(Window.partitionBy("cust_pwr_id").orderBy("t")))
runs = nl.groupBy("cust_pwr_id", "_g").agg(F.count("*").alias("len"), F.min("t").alias("s"), F.first("last_live_t").alias("llt"))
rr = runs.filter(F.col("len") >= CONFIRM_M).agg(F.count("*").alias("n"), F.sum((F.col("s") < F.col("llt")).cast("int")).alias("back")).first()
R.update(dep_end=END_M, customers=g["customers"], p_return=(rr["back"] or 0) / max(rr["n"], 1), n_dup_keys=n_dup,
         dep_feats=len(DEP_FEATS), pricing_cols=PRC)
B.kv([("deposit window (full months)", f"{DATE_START[:7]} … {END_M} ({END_T + 1} months)"),
      ("duplicated account × leg × day keys (removed)", n_dup), ("customer-months (complete grid)", g["rows"]),
      ("customers", g["customers"]), ("live customer-months", g["live_rows"]),
      ("live months with money still here", f"{g['msh_rows'] / g['live_rows']:.1%}"),
      (f"P(client returns | ≥ {CONFIRM_M} non-live months)", f"{R['p_return']:.2%}  (runs: {rr['n']:,})"),
      ("deposit features", f"{len(DEP_FEATS)} ({', '.join(sorted(set(v[0] for v in DEP.values())))})")], "2a · Panel")
lab["exit_type"] = lab.exit_type.fillna("no exit in window (or not yet confirmed)")
B.table(lab, "2b · Customers by target event (whole window, before any qualification)", save="m1_2b_events")
B.check("deposit panel", "READY", f"{END_M} is the last full month; exits are absorbing ({R['p_return']:.2%} return)"
        if R["p_return"] <= 0.02 else f"{R['p_return']:.2%} of confirmed exits come back — raise CONFIRM_M")
B.show(t0)

## §3 Payment metrics → payment features

Every numeric column of `pkg_cust_metrics_monthly` except the identity columns becomes three features:

| Feature | Meaning |
|---|---|
| `m` | the level: log1p for dollars and counts (signed for net flows), as stored for shares and ratios |
| `m__d6` | change vs the client's own months −6…−4 (a log ratio for dollars and counts, a difference for shares) |
| `m__d3` | change vs the client's own last 3 months |

A client with several `mdm_id`s under one power id takes the row of its largest entity that month. A month with
no row after the client's first row is **zero activity**: dollars and counts 0, shares empty. Churn-state metrics
(new / reactivated / retained / lost, recurring, retention) are left empty for the first 6 months of 2024, while
their 12-month lookback is too short to mean anything.

In [ ]:
# =====================================================================
# 3 · PAYMENT METRICS → PAYMENT FEATURES                         [OUTPUT BLOCK 3]
# =====================================================================
t0 = time.time(); B = Block("§3 · Payment metrics → payment features")
MET = spark.table(MET_TABLE).filter(F.col("month").between(DATE_START[:7], END_M))
ID_KNOWN = {"mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
            "naics2", "naics_status", "is_tm", "pwr_shared", "family_size", "customer_start_dt", "in_neo4j_customer",
            "in_cust_dim", "month", "history_months"}
try:
    DICT = topd(spark.table(DICT_TABLE))
    ID_KNOWN |= set(DICT.loc[DICT.family == "identity", "column"])
    FAMILY = dict(zip(DICT.column, DICT.family)); DEFN = dict(zip(DICT.column, DICT.definition))
except Exception as e:
    FAMILY, DEFN = {}, {}
    B.check("dictionary", "INFO", f"{DICT_TABLE} unreadable ({type(e).__name__}); families default to 'other'")
NUM = ("int", "bigint", "double", "float", "smallint", "tinyint", "long")
MCOLS = [c for c, t in MET.dtypes if c not in ID_KNOWN and (t in NUM or t.startswith("decimal"))]

# ── every month present and populated? ───────────────────────────────
nn = topd(MET.groupBy("month").agg(F.count("*").alias("rows"), *[F.count(c).alias(c) for c in MCOLS]).orderBy("month"))
want = [ym_of(t) for t in range(END_T + 1)]
absent = [m for m in want if m not in set(nn.month)]
pop = (nn[MCOLS] > 0)
usual = pop.mean() >= 0.5                                         # columns populated in most months
CHURN_RE = re.compile(r"^(new_|reactivated_|retained_|lost_|rec_|fi_rec_|retention|fi_retention)")
nt = nn.month.map(t_of).to_numpy()
empty = ~pop.loc[:, usual.values]
lookback = np.array([bool(CHURN_RE.match(c)) for c in empty.columns])
# lookback-based columns are legitimately empty while their history is short (first months of 2024)
bad = empty.loc[:, ~lookback].any(axis=1).to_numpy() & (nt >= 1) | empty.any(axis=1).to_numpy() & (nt >= CHURN_WARM)
stale = nn.loc[bad, "month"].tolist()
B.table(nn[["month", "rows"]].assign(empty_columns=empty.sum(axis=1).values),
        "3a · Metrics by month (empty_columns = columns populated in most months but empty here)", save="m1_3a_months")
if absent or stale:
    B.check("metrics months", "BLOCKER", f"absent {absent}, empty columns in {stale}")
    raise RuntimeError(f"metrics months not ready: absent {absent}, stale {stale}")
B.check("metrics months", "READY", f"all {len(want)} months present, {len(MCOLS)} metric columns populated throughout")

# ── key: deposit cust_pwr_id ↔ metrics customer_pwr_id (format decided from the data) ──
def pk_expr(c, mode):
    s = F.trim(F.col(c).cast("string")); s = F.when(s == "", F.lit(None)).otherwise(s)
    return F.regexp_replace(s, "^0+(?=.)", "") if mode == "strip0" else s
dk = X.select("cust_pwr_id").distinct()
mk = MET.select("customer_pwr_id").distinct()
rates = {m_: dk.select(pk_expr("cust_pwr_id", m_).alias("k")).join(
            mk.select(pk_expr("customer_pwr_id", m_).alias("k")).distinct().withColumn("h", F.lit(1)), "k", "left")
            .agg(F.avg(F.coalesce("h", F.lit(0)).cast("double"))).first()[0] for m_ in ("raw", "strip0")}
KEY = "strip0" if rates["strip0"] > rates["raw"] + 0.01 else "raw"
R["key_match"] = rates[KEY]
if R["key_match"] < 0.3: raise RuntimeError(f"power-id join broken: {rates}")

# ── one row per power id and month: the largest entity ───────────────
flow = sum(F.coalesce(F.col(c).cast("double"), F.lit(0.0)) for c in ["amt_all_in", "amt_all_out"] if c in MCOLS)
ACT = [c for c in ["ntxn_all_out", "ntxn_all_in", "ntxn_self_out", "ntxn_self_in"] if c in MCOLS]
pm = (MET.select(pk_expr("customer_pwr_id", KEY).alias("pwr"), t_col("month").alias("t"), "mdm_id",
                 *[F.col(c).cast("double").alias(c) for c in MCOLS])
         .filter("pwr is not null")
         .withColumn("_f", flow)
         .withColumn("_r", F.row_number().over(Window.partitionBy("pwr", "t").orderBy(F.desc("_f"), "mdm_id")))
         .withColumn("pay_n_mdm", F.count("*").over(Window.partitionBy("pwr", "t")))
         .filter("_r = 1").drop("_r", "_f", "mdm_id")
         .withColumn("_row", F.lit(1))
         .withColumn("_act", (sum(F.coalesce(F.col(c), F.lit(0.0)) for c in ACT) > 0).cast("int") if ACT else F.lit(1)))

def kind(c):
    if re.match(r"^(amt_|net_flow|avg_ticket|selfpay_amt)", c): return "money"
    if re.search(r"(share|hhi|retention|entropy|coverage|ratio)", c): return "bounded"
    return "count"
KIND = {c: kind(c) for c in MCOLS}

Y = X.withColumn("pwr", pk_expr("cust_pwr_id", KEY)).join(pm, ["pwr", "t"], "left")
wp = Window.partitionBy("cust_pwr_id").orderBy("t")
Y = (Y.withColumn("pay_first_t", F.min(F.when(F.col("_row") == 1, F.col("t"))).over(
            wp.rowsBetween(Window.unboundedPreceding, 0)))                    # first row known by month t
      .withColumn("pay_seen", F.col("pay_first_t").isNotNull().cast("int"))
      .withColumn("pay_silent", ((F.col("pay_seen") == 1) & (F.coalesce(F.col("_act"), F.lit(0)) == 0)).cast("int"))   # v3: no activity
      .withColumn("_last_act", F.max(F.when(F.col("_act") == 1, F.col("t"))).over(wp.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("pay_months_since_act", F.col("t") - F.col("_last_act"))
      .withColumn("pay_n_mdm", F.when(F.col("pay_seen") == 1, F.coalesce("pay_n_mdm", F.lit(0)))))
# size anchor and raw activity kept before the transforms (silent month = 0 once the client has been seen)
_z = lambda c: F.coalesce(F.col(c), F.lit(0.0)) if c in MCOLS else F.lit(0.0)
Y = (Y.withColumn("pay_log_flow", F.when(F.col("pay_seen") == 1, F.log1p(F.greatest(_z("amt_all_in") + _z("amt_all_out"), F.lit(0.0)))))
      .withColumn("raw_ntx", F.when(F.col("pay_seen") == 1, _z("ntxn_all_in") + _z("ntxn_all_out")))
      .withColumn("raw_selfpay_out", F.when(F.col("pay_seen") == 1, _z("selfpay_amt_out"))))
lv = []
for c in MCOLS:
    v = F.col(c)
    if KIND[c] == "money": e = F.signum(v) * F.log1p(F.abs(v))
    elif KIND[c] == "count": e = F.log1p(F.greatest(v, F.lit(0.0)))
    else: e = v
    if KIND[c] != "bounded": e = F.when(F.col("_row").isNull() & (F.col("pay_seen") == 1), F.lit(0.0)).otherwise(e)
    if CHURN_RE.match(c): e = F.when(F.col("t") < CHURN_WARM, F.lit(None)).otherwise(e)
    lv.append(e.alias(c))
Y = Y.select(*[c for c in Y.columns if c not in MCOLS], *lv)
wr = lambda a, b: wp.rangeBetween(a, b)
ch = []
for c in MCOLS:
    for nm, (a, b) in (("d6", (-6, -4)), ("d3", (-3, -1))):
        ch.append(F.when(F.count(c).over(wr(a, b)) >= 2, F.col(c) - F.avg(c).over(wr(a, b))).alias(f"{c}__{nm}"))
FEAT = Y.select("*", *ch).persist(DISK)
PAY_FEATS = [f for c in MCOLS for f in (c, f"{c}__d6", f"{c}__d3")]
STATUS_FEATS = ["pay_seen", "pay_silent", "pay_months_since_act"]

cov = FEAT.filter("live = 1").agg(F.count("*").alias("n"), F.avg("pay_seen").alias("seen"), F.avg("pay_silent").alias("silent"),
                                  F.sum(F.when(F.col("pay_seen") == 1, F.col("bal"))).alias("bal_seen"),
                                  F.sum("bal").alias("bal_all")).first()
R.update(pay_seen=cov["seen"], pay_seen_usd=cov["bal_seen"] / cov["bal_all"], pay_silent=cov["silent"],
         n_metrics=len(MCOLS), pay_feats=len(PAY_FEATS), key_mode=KEY)
fam_tab = pd.Series({c: FAMILY.get(c, "other") for c in MCOLS}).value_counts().rename_axis("family").reset_index(name="metrics")
B.kv([("power-id key match (deposit customers found in the metrics)", f"{R['key_match']:.1%} (mode {KEY})"),
      ("live customer-months with payment history", f"{cov['seen']:.1%} of months, {R['pay_seen_usd']:.1%} of balance"),
      ("…of which silent (no metrics row that month)", f"{cov['silent']:.1%} of live months"),
      ("metric columns → features", f"{len(MCOLS)} → {len(PAY_FEATS)} (+ {len(STATUS_FEATS)} activity-status features)")],
     "3b · Payment features")
B.table(fam_tab, "3c · Metric columns by dictionary family")
B.show(t0)

## §4 Model frame: money-still-here rows, peer gaps, de-duplicated features

**Rows:** live, observed ≥ 3 months, normal balance ≥ USD 10k, current balance ≥ half of normal, money not yet
gone — exactly the clients the list can call. **Label:** the money leaves (full or balance exit) in the next 6
months. **Size peers:** the same month's clients in the same decile of normal balance; each payment level becomes
its gap to the peers' median. **Size anchors:** log balance, log normal balance, log payment flow.

In [ ]:
# =====================================================================
# 4 · MODEL FRAME, TRAINING SAMPLE, FEATURE SETS                  [OUTPUT BLOCK 4]
# =====================================================================
t0 = time.time(); B = Block("§4 · Model frame and feature sets")
LAST_LABEL_T = min(END_T - CONFIRM_M + 1, END_T - BAL_EXIT_HOLD + 1) - H
MF = (FEAT.filter((F.col("live") == 1) & (F.col("hist_t") >= MIN_HIST) & F.col("bal_norm").isNotNull()
                  & (F.col("msh") == 1) & (F.col("ab_t").isNull() | (F.col("t") < F.col("ab_t"))))
          .withColumn("y", F.coalesce(((F.col("ab_t") > F.col("t")) & (F.col("ab_t") <= F.col("t") + H)).cast("int"), F.lit(0)))
          .withColumn("labelled", (F.col("t") <= LAST_LABEL_T).cast("int"))
          .withColumn("dep_log_norm", F.log1p("bal_norm"))
          .withColumn("size_dec", F.least(F.floor(F.percent_rank().over(Window.partitionBy("t").orderBy("bal_norm")) * 10),
                                          F.lit(9)).cast("int")))
wpeer = Window.partitionBy("t", "size_dec")
MF = MF.select("*", *[(F.col(c) - F.expr(f"percentile_approx(`{c}`, 0.5)").over(wpeer)).alias(f"{c}__peer") for c in MCOLS]).persist(DISK)
DEP["dep_log_norm"] = ("size anchor", "normal balance, log"); DEP["dep_log_bal"] = ("size anchor", "balance, log")
ANCHORS = ["dep_log_bal", "dep_log_norm", "pay_log_flow"]
FEATS0 = (ANCHORS + [f for f in DEP_FEATS if f not in ANCHORS] + [f"{c}__peer" for c in MCOLS]
          + [f"{c}__{k}" for c in MCOLS for k in ("d6", "d3")] + STATUS_FEATS)
META = ["cust_pwr_id", "t", "y", "msh", "bal", "bal_norm", "capped", "ab_t", "exit_type", "size_dec"]
keep = F.expr(f"pmod(hash(cust_pwr_id, t, {SEED}), 10000)") < int(NEG_RATE * 10000)
TR = topd(MF.filter("labelled = 1").filter((F.col("y") == 1) | keep)
            .withColumn("w", F.when(F.col("y") == 1, F.lit(1.0)).otherwise(F.lit(1.0 / NEG_RATE)))
            .select(*META, "w", *[F.col(f).cast("float").alias(f) for f in FEATS0]))
TR[FEATS0] = TR[FEATS0].astype("float32")

# ── drop empty / constant, then near-duplicates (keep the better-covered, earlier-listed one) ──
miss0 = TR[FEATS0].isna().mean()
FEATS = [f for f in FEATS0 if miss0[f] < 0.995 and TR[f].nunique(dropna=True) > 1]
S_ = TR[FEATS].sample(min(len(TR), 100_000), random_state=SEED)
S_ = S_.fillna(S_.median()).to_numpy(np.float64)
CM_ = np.nan_to_num(np.corrcoef(S_, rowvar=False))
order = sorted(range(len(FEATS)), key=lambda j: (miss0[FEATS[j]], j))
kept, dupes = [], []
for j in order:
    hit = [i for i in kept if abs(CM_[i, j]) > CORR_MAX]
    if hit: dupes.append({"dropped": FEATS[j], "kept": FEATS[hit[0]], "r": float(CM_[hit[0], j])})
    else: kept.append(j)
FEATS = [FEATS[j] for j in sorted(kept)]
DUP = pd.DataFrame(dupes); DUP.to_csv(OUT / "m4_4_dropped_duplicates.csv", index=False)

def fam_of(f):
    if f in ANCHORS: return "size anchor"
    if f in DEP: return "deposit · " + DEP[f][0]
    if f in STATUS_FEATS: return "payment · activity status"
    if f.startswith("miss·"): return "missing share"
    base = f.split("__")[0]
    return "payment · " + FAMILY.get(base, "other")
def form_of(f):
    if f in ANCHORS: return "size anchor"
    if f.endswith("__peer"): return "gap to size peers"
    if f.endswith("__d6") or f in ("dep_d6", "dep_live_d6", "dep_sib_d6") or f.endswith("_d6"): return "change vs 4–6 months ago"
    if f.endswith("__d3") or f == "dep_d3": return "change vs last 3 months"
    if f in LATE_FEATS: return "drain in progress"
    return "other"
def label(f):
    if f in DEP: return DEP[f][1]
    if f.startswith("miss·"): return f"share of {f[5:]} features not observed"
    st = {"pay_seen": "has payment history", "pay_silent": "no payments this month (silent)",
          "pay_months_since_act": "months since the last payment", "pay_log_flow": "payment flow in + out, log"}
    if f in st: return st[f]
    base, _, tr = f.partition("__")
    d = DEFN.get(base, base.replace("_", " "))
    return d + {"d6": " — change vs 4–6 months ago", "d3": " — change vs the last 3 months",
                "peer": " — gap to size peers", "": ""}[tr]
FAM = {f: fam_of(f) for f in FEATS}
FAM_SETS = sorted(set(v for v in FAM.values() if v.startswith("payment")))
FAM_IDX = {s_: [j for j, f in enumerate(FEATS) if FAM[f] == s_] for s_ in FAM_SETS}
COLS = FEATS + [f"miss·{s_}" for s_ in FAM_SETS]
LABEL = {c: label(c) for c in COLS}; FAMOF = {c: (FAM.get(c) or "missing share") for c in COLS}
FS = pd.DataFrame([{"feature": f, "set": FAM[f], "form": form_of(f), "coverage": 1 - miss0[f]} for f in FEATS])
FST = FS.groupby(["set"]).agg(features=("feature", "size"), median_coverage=("coverage", "median")).reset_index()
FFORM = FS.groupby("form").agg(features=("feature", "size")).reset_index()
FS.assign(label=FS.feature.map(label)).to_csv(OUT / "m4_4_feature_list.csv", index=False)
by_t = topd(MF.groupBy("t").agg(F.count("*").alias("money_still_here"), F.sum("y").alias("leaving_6m"),
                                F.sum(F.when(F.col("y") == 1, F.col("capped"))).alias("leaving_capped"),
                                F.sum("capped").alias("capped")).orderBy("t"))
by_t["month"] = by_t.t.map(ym_of); by_t["rate_6m"] = by_t.leaving_6m / by_t.money_still_here
by_t["labelled"] = by_t.t <= LAST_LABEL_T
R.update(last_label=ym_of(LAST_LABEL_T), train_rows=len(TR), train_pos=int(TR.y.sum()), n_feats=len(FEATS),
         n_dupes=len(DUP), feature_sets=FST, forms=FFORM,
         base_rate=float(by_t.loc[by_t.labelled, "leaving_6m"].sum() / by_t.loc[by_t.labelled, "money_still_here"].sum()))
B.kv([("labelled months", f"… {ym_of(LAST_LABEL_T)} (outcome fully observed)"),
      ("money-still-here client-months, labelled", int(by_t.loc[by_t.labelled, "money_still_here"].sum())),
      ("leave within 6 months (base rate)", f"{R['base_rate']:.2%}"),
      ("training sample", f"{len(TR):,} rows · {R['train_pos']:,} leaving · stayers at {NEG_RATE:.0%}, weight {1 / NEG_RATE:.1f}"),
      ("features", f"{len(FEATS0)} built → {len(FEATS)} kept ({len(DUP)} near-duplicates dropped) + {len(FAM_SETS)} missing shares")],
     "4a · Model frame")
B.table(by_t[["month", "money_still_here", "leaving_6m", "rate_6m", "capped", "leaving_capped", "labelled"]],
        "4b · Money-still-here client-months by month", pct=("rate_6m",), money=("capped", "leaving_capped"), save="m4_4b_by_month")
B.table(FST, "4c · Feature sets", pct=("median_coverage",), save="m4_4c_feature_sets")
B.table(FFORM, "4c · Feature forms")
B.show(t0)

## §5 Rolling monthly test folds — simple model and tree model

For each test month T both models are fitted on months whose outcome was known at T (origins ≤ T − 6). **20% of
clients are held out** (all their months, chosen by a fixed hash): on the first fold they choose the ridge penalty
and the tree depth; on every fold they stop the trees early; those from the **3 most recent labelled months** calibrate the tree's probabilities
(isotonic, capped at 95%).
Held-out clients never share a label window with the fit, so validation is as strict as test. Both models then
score every money-still-here client in T.

In [ ]:
# =====================================================================
# 5 · ROLLING TEST FOLDS — simple (ridge logistic) and tree model   [OUTPUT BLOCK 5]
# =====================================================================
t0 = time.time(); B = Block("§5 · Rolling monthly test folds")
TEST_T = list(range(t_of(TEST_FIRST), LAST_LABEL_T + 1))
if len(TEST_T) < 3: raise RuntimeError(f"only {len(TEST_T)} test months")
FI = [FEATS.index(f) for f in FEATS]

def prep_fit(Xtr):
    s = Xtr[:min(len(Xtr), 200_000)]
    lo, hi = np.nanquantile(s, CLIP_Q[0], axis=0), np.nanquantile(s, CLIP_Q[1], axis=0)
    xc = np.clip(Xtr, lo, hi)
    mu = np.nanmean(xc, axis=0); sd = np.nanstd(xc, axis=0); sd[~(sd > 1e-9)] = 1.0
    mf = np.stack([np.isnan(Xtr[:, FAM_IDX[s_]]).mean(axis=1) for s_ in FAM_SETS], axis=1) if FAM_SETS else np.zeros((len(Xtr), 0))
    return dict(lo=np.nan_to_num(lo), hi=np.nan_to_num(hi), mu=np.nan_to_num(mu), sd=sd, mm=mf.mean(axis=0), ms=mf.std(axis=0) + 1e-9)

def transform(Xr, P):
    z = np.nan_to_num((np.clip(Xr, P["lo"], P["hi"]) - P["mu"]) / P["sd"], nan=0.0)
    mf = np.stack([np.isnan(Xr[:, FAM_IDX[s_]]).mean(axis=1) for s_ in FAM_SETS], axis=1) if FAM_SETS else np.zeros((len(Xr), 0))
    return np.hstack([z, (mf - P["mm"]) / P["ms"]]).astype(np.float32)

def logit_newton(Z, y, w, l2, max_it=60, tol=1e-6):
    """Weighted ridge logistic by DAMPED Newton; intercept unpenalised; chunked."""
    n, k = Z.shape; w = w / w.mean(); CH = 50_000
    pbar = np.average(y, weights=w); b = np.zeros(k + 1); b[0] = np.log(pbar / (1 - pbar))
    def eta(b): return b[0] + np.concatenate([Z[i:i + CH].astype(np.float64) @ b[1:] for i in range(0, n, CH)])
    def obj(e, b): return float(np.sum(w * (np.logaddexp(0, e) - y * e)) + 0.5 * l2 * b[1:] @ b[1:])
    e = eta(b); f = obj(e, b); conv = False
    for it in range(1, max_it + 1):
        p = 1 / (1 + np.exp(-np.clip(e, -35, 35))); r = w * (p - y); v = w * p * (1 - p)
        g = np.empty(k + 1); g[0] = r.sum(); g[1:] = l2 * b[1:]
        Hs = np.zeros((k + 1, k + 1)); Hs[0, 0] = v.sum()
        for i in range(0, n, CH):
            Zc = Z[i:i + CH].astype(np.float64); rc, vc = r[i:i + CH], v[i:i + CH]
            g[1:] += Zc.T @ rc; Hs[1:, 0] += Zc.T @ vc; Hs[1:, 1:] += (Zc * vc[:, None]).T @ Zc
        Hs[0, 1:] = Hs[1:, 0]; Hs[1:, 1:] += l2 * np.eye(k)
        step = np.linalg.solve(Hs, g); s_ = 1.0
        while True:
            bn = b - s_ * step; en = eta(bn); fn = obj(en, bn)
            if fn <= f or s_ < 1e-8: break
            s_ *= 0.5
        b, e, f_old, f = bn, en, f, fn
        if np.max(np.abs(g)) / n < tol or abs(f_old - f) < 1e-10 * max(1.0, abs(f)): conv = True; break
    return b, dict(converged=conv, iters=it)

def lin_score(b, Z): return 1 / (1 + np.exp(-np.clip(b[0] + Z @ b[1:], -35, 35)))

# ── tree engine wrappers: fit(X, y, w, Xv, yv, wv, depth) → model; predict; contributions ──
def gbm_fit(X, y, w, Xv, yv, wv, depth):
    if GBM == "xgboost":
        import xgboost as xgb
        pr = dict(objective="binary:logistic", eval_metric="logloss", eta=GBM_ETA, max_depth=depth, subsample=0.8,
                  colsample_bytree=0.6, min_child_weight=20, reg_lambda=1.0, tree_method="hist", nthread=os.cpu_count(), seed=SEED)
        dt = xgb.DMatrix(X, label=y, weight=w, missing=np.nan); dv = xgb.DMatrix(Xv, label=yv, weight=wv, missing=np.nan)
        m = xgb.train(pr, dt, num_boost_round=GBM_MAX_TREES, evals=[(dv, "val")], early_stopping_rounds=GBM_EARLY, verbose_eval=False)
        return dict(kind="xgboost", m=m, n=int(m.best_iteration) + 1)
    if GBM == "lightgbm":
        import lightgbm as lgb
        pr = dict(objective="binary", learning_rate=GBM_ETA, num_leaves=min(2 ** depth - 1, 255), max_depth=depth,
                  min_data_in_leaf=200, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, seed=SEED, num_threads=os.cpu_count())
        dt = lgb.Dataset(X, y, weight=w); dv = lgb.Dataset(Xv, yv, weight=wv, reference=dt)
        try:    m = lgb.train(pr, dt, GBM_MAX_TREES, valid_sets=[dv], callbacks=[lgb.early_stopping(GBM_EARLY, verbose=False)])
        except (AttributeError, TypeError): m = lgb.train(pr, dt, GBM_MAX_TREES, valid_sets=[dv], early_stopping_rounds=GBM_EARLY, verbose_eval=False)
        return dict(kind="lightgbm", m=m, n=int(m.best_iteration or GBM_MAX_TREES))
    from sklearn.ensemble import HistGradientBoostingClassifier
    m = HistGradientBoostingClassifier(learning_rate=GBM_ETA, max_depth=depth, max_iter=GBM_MAX_TREES, early_stopping=True,
                                       validation_fraction=0.1, n_iter_no_change=GBM_EARLY // 2, l2_regularization=1.0, random_state=SEED)
    m.fit(X, y, sample_weight=w)
    return dict(kind="sklearn", m=m, n=int(m.n_iter_))

def gbm_predict(M, X, contrib=False):
    if M["kind"] == "xgboost":
        import xgboost as xgb
        d = xgb.DMatrix(X, missing=np.nan)
        try:    p = M["m"].predict(d, iteration_range=(0, M["n"]), pred_contribs=contrib)
        except TypeError: p = M["m"].predict(d, ntree_limit=M["n"], pred_contribs=contrib)
        return p[:, :-1] if contrib else p
    if M["kind"] == "lightgbm":
        p = M["m"].predict(X, num_iteration=M["n"], pred_contrib=contrib)
        return p[:, :-1] if contrib else p
    return None if contrib else M["m"].predict_proba(X)[:, 1]

Xall = TR[FEATS].to_numpy(np.float32)
yall, wall, tall = TR.y.to_numpy(float), TR.w.to_numpy(float), TR.t.to_numpy()
HOLD = (pd.util.hash_pandas_object(TR.cust_pwr_id.astype(str), index=False).to_numpy() % 1000) < int(HOLD_FRAC * 1000)
TR = TR[META + ["w"]]
L2, DEPTH, TUNE = None, None, []
SCORES, FOLDS, SAMPLE = [], [], []
ABS_L, ABS_G, N_SC = np.zeros(len(COLS)), np.zeros(len(FEATS)), 0
rng = np.random.default_rng(SEED)
for T in TEST_T:
    tf = time.time()
    m = tall <= T - H; mv = m & HOLD; mf = m & ~HOLD
    if mv.sum() == 0 or yall[mv].sum() == 0: mf, mv = m, m
    # ── simple model ─────────────────────────────────────────────────
    if L2 is None:
        Pv = prep_fit(Xall[mf]); Zf = transform(Xall[mf], Pv); Zv = transform(Xall[mv], Pv)
        for l2 in L2_GRID:
            bb, _ = logit_newton(Zf, yall[mf], wall[mf], l2)
            TUNE.append({"model": "simple", "setting": f"L2 = {l2:g}", "val_auc": auc(yall[mv], lin_score(bb, Zv))})
        L2 = max((r for r in TUNE if r["model"] == "simple"), key=lambda r: r["val_auc"])["setting"]
        L2 = float(L2.split("= ")[1]); del Zf, Zv
    P = prep_fit(Xall[m]); Z = transform(Xall[m], P)
    b, info = logit_newton(Z, yall[m], wall[m], L2); del Z
    # ── tree model ───────────────────────────────────────────────────
    G = None
    if GBM:
        if DEPTH is None:
            for d_ in GBM_GRID:
                Gd = gbm_fit(Xall[mf], yall[mf], wall[mf], Xall[mv], yall[mv], wall[mv], d_)
                TUNE.append({"model": f"tree ({GBM})", "setting": f"depth {d_}, {Gd['n']} trees",
                             "val_auc": auc(yall[mv], gbm_predict(Gd, Xall[mv]))})
            best = max((r for r in TUNE if r["model"].startswith("tree")), key=lambda r: r["val_auc"])
            DEPTH = int(best["setting"].split(",")[0].split()[1])
        G = gbm_fit(Xall[mf], yall[mf], wall[mf], Xall[mv], yall[mv], wall[mv], DEPTH)
        from sklearn.isotonic import IsotonicRegression
        mc = mv & (tall >= T - H - CAL_RECENT + 1)                 # held-out clients, most recent labelled months
        if yall[mc].sum() < 50: mc = mv
        G["iso"] = IsotonicRegression(out_of_bounds="clip", y_min=0.001, y_max=P_CAP).fit(
            gbm_predict(G, Xall[mc]), yall[mc], sample_weight=wall[mc])
    # ── score the test month ─────────────────────────────────────────
    te = topd(MF.filter(F.col("t") == T).select(*META, *[F.col(f).cast("float").alias(f) for f in FEATS]))
    Xt = te[FEATS].to_numpy(np.float32); Zt = transform(Xt, P)
    Cl = Zt * b[1:].astype(np.float32); te["p_lin"] = lin_score(b, Zt)
    ABS_L += np.abs(Cl).sum(axis=0); N_SC += len(te)
    Cg = None
    if G:
        te["p_gbm_raw"] = gbm_predict(G, Xt); te["p_gbm"] = G["iso"].predict(te.p_gbm_raw.to_numpy()); Cg = gbm_predict(G, Xt, contrib=True)
        if Cg is not None: ABS_G += np.abs(Cg).sum(axis=0)
    else:
        te["p_gbm"] = np.nan
    RC, RN = (Cg, FEATS) if Cg is not None else (Cl, COLS)          # reason codes: tree SHAP when available
    top = np.argsort(-RC, axis=1)[:, :3]
    for k_ in range(3):
        te[f"r{k_ + 1}"] = [RN[j] for j in top[:, k_]]
    te["tm"] = T
    si = rng.random(len(te)) < min(1.0, 15_000 / max(len(te), 1))
    SAMPLE.append(pd.DataFrame(Xt[si], columns=FEATS).assign(y=te.y.values[si]))
    SCORES.append(te.drop(columns=FEATS))
    FOLDS.append({"test_month": ym_of(T), "train_rows": int(m.sum()), "train_leaving": int(yall[m].sum()),
                  "simple_converged": info["converged"], "trees": (G or {}).get("n"), "test_rows": len(te),
                  "test_leaving": int(te.y.sum()), "auc_simple": auc(te.y, te.p_lin), "auc_tree": auc(te.y, te.p_gbm) if G else np.nan,
                  "mean_p_simple": float(te.p_lin.mean()), "mean_p_tree": float(te.p_gbm.mean()) if G else np.nan,
                  "actual_rate": float(te.y.mean()), "minutes": (time.time() - tf) / 60})
    MODEL = dict(b=b, P=P, G=G)
    print(f"  {ym_of(T)}: {len(te):,} scored · AUC simple {FOLDS[-1]['auc_simple']:.3f} · tree {FOLDS[-1]['auc_tree']:.3f} · "
          f"{FOLDS[-1]['minutes']:.1f} min")
SC = pd.concat(SCORES, ignore_index=True); SMP = pd.concat(SAMPLE, ignore_index=True); FO = pd.DataFrame(FOLDS)
SC["capped"] = SC.capped.fillna(0.0).clip(lower=0)
PRIMARY = "p_gbm" if GBM else "p_lin"
SC["p"] = SC[PRIMARY]
R.update(test_months=[ym_of(t) for t in TEST_T], n_test=len(TEST_T), folds=FO, tune=pd.DataFrame(TUNE), L2=L2, depth=DEPTH,
         engine=GBM, primary="tree model" if GBM else "simple model",
         auc_lin=auc(SC.y, SC.p_lin), auc_gbm=auc(SC.y, SC.p_gbm) if GBM else np.nan)
B.table(pd.DataFrame(TUNE), "5a · Settings chosen on the held-out clients (first fold)", save="m4_5a_tuning")
B.table(FO, "5b · One fold per test month", pct=("mean_p_simple", "mean_p_tree", "actual_rate"), save="m4_5b_folds")
B.kv([("pooled AUC, simple model", R["auc_lin"]), ("pooled AUC, tree model", R["auc_gbm"]),
      ("tree − simple", R["auc_gbm"] - R["auc_lin"] if GBM else np.nan),
      ("AUC range across months (tree)", f"{FO.auc_tree.min():.3f} – {FO.auc_tree.max():.3f}" if GBM else "—"),
      ("primary model for the list, reasons and examples", R["primary"])], "5c · Discrimination")
if not GBM: B.check("tree model", "FIX", "no tree library in new_default: pip install xgboost (or lightgbm) and rerun")
B.show(t0)

## §6 Predictive power

**In the model:** each feature's share of the average absolute contribution on the test months — exact TreeSHAP
for the tree model, coefficient × standardised value for the simple model. **Alone:** how well the feature ranks
leavers by itself (AUC folded so 0.5 = nothing). Features are also grouped by *form* (size anchor, gap to size
peers, change, drain in progress), which answers whether the model now runs on change rather than size.

In [ ]:
# =====================================================================
# 6 · PREDICTIVE POWER                                              [OUTPUT BLOCK 6]
# =====================================================================
t0 = time.time(); B = Block("§6 · Predictive power")
alone = {}
for f in FEATS:
    a = auc(SMP.y, SMP[f].to_numpy(float))
    alone[f] = (max(a, 1 - a) if pd.notna(a) else np.nan, "higher" if (pd.notna(a) and a >= 0.5) else "lower")
use_g = GBM is not None and ABS_G.sum() > 0
shr = (ABS_G / ABS_G.sum()) if use_g else (ABS_L[:len(FEATS)] / ABS_L.sum())
lin_shr = ABS_L / ABS_L.sum()
IMP = pd.DataFrame([{"feature": f, "label": LABEL[f], "set": FAM[f], "form": form_of(f), "share": shr[j],
                     "share_simple": lin_shr[j], "simple_sign": "risk up when high" if MODEL["b"][1 + j] > 0 else "risk up when low",
                     "alone_auc": alone[f][0], "leavers_sit": alone[f][1]} for j, f in enumerate(FEATS)])
IMP = IMP.sort_values("share", ascending=False).reset_index(drop=True)
IMP.to_csv(OUT / "m4_6_importance_all.csv", index=False)
FAMI = IMP.groupby("set").agg(share=("share", "sum"), share_simple=("share_simple", "sum"), features=("feature", "size"),
                              best_alone=("alone_auc", "max")).sort_values("share", ascending=False).reset_index()
if not use_g: FAMI.loc[len(FAMI)] = ["missing share", float(lin_shr[len(FEATS):].sum()), float(lin_shr[len(FEATS):].sum()), len(FAM_SETS), np.nan]
FORM = IMP.groupby("form").agg(share=("share", "sum"), share_simple=("share_simple", "sum"), features=("feature", "size")) \
          .sort_values("share", ascending=False).reset_index()
R.update(imp=IMP, fam_imp=FAMI, form_imp=FORM, imp_basis="tree SHAP" if use_g else "simple model contributions",
         miss_share_simple=float(lin_shr[len(FEATS):].sum()))
B.table(IMP.head(25), f"6a · Top 25 features ({R['imp_basis']}; share_simple = the simple model's share)",
        pct=("share", "share_simple"), save="m4_6a_top")
B.table(FAMI, "6b · Feature sets", pct=("share", "share_simple"), save="m4_6b_sets")
B.table(FORM, "6c · Feature forms: size, peer gap, change, drain in progress", pct=("share", "share_simple"), save="m4_6c_forms")
B.kv([("missing-share columns in the simple model", f"{R['miss_share_simple']:.1%} of contribution (model 1: 24.5% from 89 flags)")],
     "6d · Missingness")
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
t20 = IMP.head(20)[::-1]
ax[0].barh([l[:60] for l in t20.label], t20.share, color="#1f4e79"); ax[0].set_title(f"Top 20 features ({R['imp_basis']})")
f12 = FAMI.head(12)[::-1]
ax[1].barh(f12.set, f12.share, color="#c55a11"); ax[1].set_title("Feature sets")
for a in ax: a.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m4_6_importance")
B.show(t0)

## §7 Checking the largest departures

A handful of very large clients carry most of the dollars, so each of the largest departures is checked in the
data before its dollars are counted. For every client: did the balance **come back** (≥ 50% of normal after the
exit month)? Did any of its accounts get **re-linked** to another customer id? Did **other customers in the same
relationship** gain what this one lost (money moving inside PNC)? Does the client **keep transacting** here
afterwards? Were payments to **its own name at other banks** visible around the exit? Verdicts:

| Verdict (checked in this order) | Rule | Counted as a loss? |
|---|---|---|
| came back | balance returned to ≥ 50% of normal | no |
| moved to another bank | payments to its own name at other banks (exit month and 3 before) exceed its own level 6–9 months earlier by ≥ 50% of the drop | yes |
| moved within PNC | a re-linked account carries ≥ 50% of the drop afterwards, or siblings gained 50–150% of it | no |
| still transacting | payment activity after the exit ≥ 50% of before | yes, flagged |
| departed | none of the above | yes |

In [ ]:
# =====================================================================
# 7 · CHECKING THE LARGEST DEPARTURES                              [OUTPUT BLOCK 7]
# =====================================================================
t0 = time.time(); B = Block("§7 · Checking the largest departures")
ex = topd(FEAT.filter(F.col("ab_t").isNotNull() & F.col("ab_t").between(MIN_HIST + 1, LAST_LABEL_T + H))
              .groupBy("cust_pwr_id").agg(F.first("ab_t").alias("ab_t"), F.first("exit_type").alias("exit_type"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 1, F.col("bal_norm"))).alias("norm"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("msh"))).alias("msh_m3"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("capped"))).alias("capped_m3")))
ex = ex.dropna(subset=["norm"])
AUD_CACHE = {}
def audit(ids):
    """Evidence and verdict for each departing client id (cached)."""
    new = [c for c in dict.fromkeys(ids) if c not in AUD_CACHE]
    if new:
        P_ = topd(FEAT.filter(F.col("cust_pwr_id").isin(new)).select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t",
                                                                     "exit_type", "first_t", "rltn_pwr_id", "raw_ntx", "raw_selfpay_out"))
        accs = [r[0] for r in AMP.filter(F.col("cust_pwr_id").isin(new)).select("acct_full_acct_id").distinct().collect()]
        RL = topd(AMP.filter(F.col("acct_full_acct_id").isin(accs or ["__none__"]))
                     .select("acct_full_acct_id", "t", "cust_pwr_id", "live", "acct_bal"))
        rls = P_.dropna(subset=["rltn_pwr_id"]).groupby("cust_pwr_id").rltn_pwr_id.last()
        SIB = topd(FEAT.filter(F.col("rltn_pwr_id").isin(list(set(rls.values)) or ["__none__"]))
                       .groupBy("rltn_pwr_id", "t", "cust_pwr_id").agg(F.sum("bal").alias("bal"))) if len(rls) else pd.DataFrame(
                       columns=["rltn_pwr_id", "t", "cust_pwr_id", "bal"])
        for c in new:
            p = P_[P_.cust_pwr_id == c].sort_values("t").set_index("t")
            if not len(p) or p.ab_t.isna().all(): AUD_CACHE[c] = {"verdict": "no exit"}; continue
            e = int(p.ab_t.dropna().iloc[0]); norm = p.bal_norm.get(e - 1, np.nan)
            pre = p.bal.loc[e - 3:e - 1].mean(); post = p.bal.loc[e:e + 2].mean()
            back = p.bal.loc[e + 1:].max() / norm if norm and len(p.bal.loc[e + 1:]) else np.nan
            r_ = RL[RL.acct_full_acct_id.isin(RL.loc[RL.cust_pwr_id == c, "acct_full_acct_id"])]
            relinked = int(r_[(r_.t >= e - 1) & (r_.cust_pwr_id != c)].acct_full_acct_id.nunique())
            rk_ = r_[r_.t.between(e, e + 2) & (r_.cust_pwr_id != c)]
            relink_share = rk_.groupby("t").acct_bal.sum().mean() / (pre - post) if len(rk_) and pre - post > 0 else 0.0
            absorb = np.nan
            if c in rls.index:
                sb = SIB[(SIB.rltn_pwr_id == rls[c]) & (SIB.cust_pwr_id != c)].groupby("t").bal.sum()
                if len(sb) and pre - post > 0:
                    absorb = (sb.loc[e:e + 2].mean() - sb.loc[e - 3:e - 1].mean()) / (pre - post)
            act_b = p.raw_ntx.loc[e - 6:e - 4].mean(); act_a = p.raw_ntx.loc[e:e + 2].mean()
            act = act_a / act_b if act_b and act_b > 0 else np.nan
            selfp = p.raw_selfpay_out.loc[e - 3:e].sum() / (pre - post) if pre - post > 0 else np.nan
            _b = p.raw_selfpay_out.loc[e - 9:e - 4]
            sp_base = float(_b.mean()) * 4 if _b.notna().any() else 0.0          # same 4-month length, 6–9 months earlier
            selfx = (p.raw_selfpay_out.loc[e - 3:e].sum() - sp_base) / (pre - post) if pre - post > 0 else np.nan
            if pd.notna(back) and back >= 0.5: v = "came back"
            elif pd.notna(selfx) and selfx >= 0.5: v = "moved to another bank"
            elif relink_share >= 0.5 or (pd.notna(absorb) and 0.5 <= absorb <= 1.5): v = "moved within PNC"
            elif pd.notna(act) and act >= 0.5: v = "still transacting"
            else: v = "departed"
            AUD_CACHE[c] = {"client": cid(c), "exit_type": p.exit_type.dropna().iloc[0] if p.exit_type.notna().any() else None,
                            "exit_month": ym_of(e), "normal": norm, "months_observed_before": int(e - p.first_t.iloc[0]),
                            "balance_after_vs_normal": back, "accounts_relinked": relinked, "relinked_balance_vs_drop": relink_share, "siblings_absorbed": absorb,
                            "activity_after_vs_before": act, "own_name_out_vs_drop": selfp, "own_name_excess_vs_drop": selfx,
                            "live_accounts_at_end": int(r_[(r_.t == END_T) & (r_.cust_pwr_id == c)].live.sum()),
                            "verdict": v}
    return pd.DataFrame([{"cust_pwr_id": c, **AUD_CACHE[c]} for c in dict.fromkeys(ids)])

pool_all = (SC[SC.y == 1].sort_values("tm").groupby("cust_pwr_id")
              .agg(ab_t=("ab_t", "first"), exit_type=("exit_type", "first"), capped=("capped", "max"),
                   bal_norm=("bal_norm", "first"), tm=("tm", "first")).reset_index())
top_ids = list(ex.sort_values("norm", ascending=False).cust_pwr_id.head(N_AUDIT)) + \
          list(pool_all.sort_values("capped", ascending=False).cust_pwr_id.head(N_AUDIT))
AUD = audit(top_ids)
AUD = AUD.merge(pool_all[["cust_pwr_id", "capped"]].rename(columns={"capped": "test_pool_capped"}), on="cust_pwr_id", how="left")
AUD.to_csv(OUT / "m4_7_audit_FULL_IDS.csv", index=False)
EXCL = set(AUD.loc[AUD.verdict.isin(["came back", "moved within PNC"]), "cust_pwr_id"])
VC = AUD.groupby("verdict").agg(clients=("client", "size"), normal=("normal", "sum")).reset_index()
R.update(largest_excluded=bool(len(AUD) and pool_all.sort_values("capped", ascending=False).cust_pwr_id.iloc[0] in EXCL),
         audit=AUD.drop(columns="cust_pwr_id"), audit_counts=VC, n_not_clean=int((AUD.verdict != "departed").sum()),
         n_audited=len(AUD), excl_ids=EXCL)
B.table(AUD.drop(columns="cust_pwr_id"), f"7a · The largest departures ({len(AUD)} clients: top {N_AUDIT} of the window, top {N_AUDIT} of the test months)",
        pct=("balance_after_vs_normal", "relinked_balance_vs_drop", "siblings_absorbed", "activity_after_vs_before", "own_name_out_vs_drop", "own_name_excess_vs_drop"),
        money=("normal", "test_pool_capped"))
B.table(VC, "7b · Verdicts", money=("normal",))
B.check("largest departures", "INFO", f"{len(EXCL)} of {len(AUD)} are not counted as losses (came back or moved within PNC); "
        "full ids in m4_7_audit_FULL_IDS.csv for an RM check")
B.show(t0)

## §8 How much is at risk; accuracy and savings against calls

The list each month: money-still-here clients ranked by `p × normal balance^α`, top K called. A departing client
is credited once, at its first alert, with its capped balance (the lesser of current and normal). Dollars are
annualised over the test months. Savings = dollars reached × an assumed save rate; there is no call cost.
Headline dollars are also shown **without the largest 1 and 5 departing clients** and **without the clients §7
found were not real losses**. Simple and tree model are compared on the same rows with a paired bootstrap over
departing clients.

In [ ]:
# =====================================================================
# 8 · AT RISK · ACCURACY VS CALLS · SAVINGS · MODEL COMPARISON      [OUTPUT BLOCK 8]
# =====================================================================
t0 = time.time(); B = Block("§8 · At risk, accuracy, savings")
span = (LAST_LABEL_T + H) - (MIN_HIST + 1) + 1
ex["band"] = pd.cut(ex.norm, [b[0] for b in BANDS] + [BANDS[-1][1]], labels=[b[2] for b in BANDS], right=False)
exv = ex[~ex.cust_pwr_id.isin(EXCL)]
AR = exv.groupby(["band", "exit_type"], observed=False).agg(clients=("norm", "size"), normal_balance=("norm", "sum")).reset_index()
AR["per_year"] = AR.normal_balance * 12 / span
R.update(at_risk_year=exv.norm.sum() * 12 / span, exits_per_year=len(exv) * 12 / span, at_risk_tab=AR,
         msh_m3=float(exv.msh_m3.mean()), capped_m4_year=float(exv.capped_m3.fillna(0).sum() * 12 / span),
         bal_exit_share=float(exv.loc[exv.exit_type == "balance exit", "norm"].sum() / exv.norm.sum()),
         top_client_share=float(exv.norm.max() / exv.norm.sum()), excl_normal=float(ex.loc[ex.cust_pwr_id.isin(EXCL), "norm"].sum() * 12 / span))
B.kv([("clients whose money left (per year, checked exclusions removed)", f"{R['exits_per_year']:,.0f}"),
      ("their normal balance (per year)", f"${R['at_risk_year']:,.0f}"),
      ("…of which balance exits (accounts left open)", f"{R['bal_exit_share']:.1%}"),
      ("…still here 3 months before (clients) / capped dollars per year", f"{R['msh_m3']:.1%} / ${R['capped_m4_year']:,.0f}"),
      ("largest single client's share", f"{R['top_client_share']:.1%}"),
      ("removed by §7 (came back / moved within PNC), per year", f"${R['excl_normal']:,.0f}")], "8a · How much is at risk")
B.table(AR, "8a · By size band and exit type (annualised)", money=("normal_balance", "per_year"), save="m4_8a_at_risk")

Q = SC.copy(); Q["norm"] = Q.bal_norm.clip(lower=1.0)
yr = 12 / R["n_test"]
MODELS = [("simple", "p_lin")] + ([("tree", "p_gbm")] if GBM else [])
first_alert, per_month = {}, []
for mn, pc in MODELS:
    for a in ALPHAS:
        Q[f"rank_{mn}_{a}"] = Q.assign(_s=Q[pc] * Q.norm ** a).groupby("tm")["_s"].rank(ascending=False, method="first")
        for K in CALLS:
            on = Q[Q[f"rank_{mn}_{a}"] <= K]
            for T, g in Q.groupby("tm"):
                o = on[on["tm"] == T]; lv = g[g.y == 1]
                per_month.append({"model": mn, "alpha": a, "calls": K, "month": ym_of(T), "precision": o.y.mean(),
                                  "recall_clients": o.y.sum() / max(len(lv), 1),
                                  "recall_dollars": o.loc[o.y == 1, "capped"].sum() / max(lv.capped.sum(), 1)})
            first_alert[(mn, a, K)] = on[on.y == 1].sort_values("tm").drop_duplicates("cust_pwr_id")
PM_ = pd.DataFrame(per_month)
pool = pool_all.copy(); POOL = pool.capped.sum()
top_pool = list(pool.sort_values("capped", ascending=False).cust_pwr_id)
def reached(fa, drop=()):
    f = fa[~fa.cust_pwr_id.isin(drop)]; return f.capped.sum() * yr, len(f)
rows = []
for (mn, a, K), fa in first_alert.items():
    d_, n_ = reached(fa)
    rows.append({"model": mn, "alpha": a, "calls": K, "clients_reached": n_, "share_of_departing": n_ / max(len(pool), 1),
                 "dollars_reached_year": d_, "share_of_dollars": d_ / yr / max(POOL, 1),
                 **{f"save_{int(s * 100)}pct_year": d_ * s for s in SAVE_RATES}})
VAL = pd.DataFrame(rows)
ACC = (PM_.groupby(["model", "alpha", "calls"]).agg(precision_mean=("precision", "mean"), precision_min=("precision", "min"),
                                                    precision_max=("precision", "max"), recall_clients=("recall_clients", "mean"),
                                                    recall_dollars=("recall_dollars", "mean")).reset_index())
MAIN = "tree" if GBM else "simple"
fa0 = first_alert[(MAIN, A_MAIN, K_MAIN)]
variants = [("all departing clients", ()), *[(f"without the largest {k}", tuple(top_pool[:k])) for k in TOP_EXCL],
            ("without clients §7 found were not losses", tuple(EXCL))]
HV = []
for nm, drop in variants:
    d_, n_ = reached(fa0, drop); pl = pool[~pool.cust_pwr_id.isin(drop)].capped.sum() * yr
    HV.append({"dollars": nm, "clients_reached": n_, "reached_per_year": d_, "departing_per_year": pl, "share": d_ / max(pl, 1)})
d_ = float(fa0.capped.clip(upper=CAP_CREDIT).sum() * yr); pl = float(pool.capped.clip(upper=CAP_CREDIT).sum() * yr)
HV.append({"dollars": f"each client's credit capped at ${CAP_CREDIT / 1e6:,.0f}m", "clients_reached": len(fa0),
           "reached_per_year": d_, "departing_per_year": pl, "share": d_ / max(pl, 1)})
HV = pd.DataFrame(HV)
# paired bootstrap over departing clients: tree − simple at the main operating point
val_of = lambda mn: pool.cust_pwr_id.map(first_alert[(mn, A_MAIN, K_MAIN)].set_index("cust_pwr_id").capped).fillna(0.0).to_numpy()
cs = val_of("simple"); ct = val_of("tree") if GBM else cs
bs_t, bs_d = [], []
for _ in range(2000):
    i = rng.integers(0, len(pool), len(pool)); bs_t.append(ct[i].sum() * yr); bs_d.append((ct[i] - cs[i]).sum() * yr)
_pm = PM_[(PM_.model == MAIN) & (PM_.alpha == A_MAIN) & (PM_.calls == K_MAIN)]
R.update(pool_year=POOL * yr, n_departing=len(pool), val=VAL, acc=ACC, per_month=PM_, headline=HV,
         main_reached=float(HV.reached_per_year.iloc[0]), main_ci=(float(np.percentile(bs_t, 2.5)), float(np.percentile(bs_t, 97.5))),
         main_share=float(HV.share.iloc[0]), diff=float((ct - cs).sum() * yr),
         diff_ci=(float(np.percentile(bs_d, 2.5)), float(np.percentile(bs_d, 97.5))),
         prec_ratio=float(_pm.precision.max() / _pm.precision.min()) if _pm.precision.min() > 0 else np.inf,
         drop_top5=1 - float(HV.reached_per_year.iloc[2] / HV.reached_per_year.iloc[0]) if len(HV) > 2 and HV.reached_per_year.iloc[0] else np.nan,
         drop_cap=1 - float(HV.reached_per_year.iloc[-1] / HV.reached_per_year.iloc[0]) if HV.reached_per_year.iloc[0] else np.nan)
for nm_, df_ in (("m4_8_accuracy_all", ACC), ("m4_8_value_all", VAL), ("m4_8_per_month_all", PM_), ("m4_8_headline", HV),
                 ("m4_8_at_risk", AR)): figdata(nm_, df_)
CMP = VAL[(VAL.alpha == A_MAIN)].pivot(index="calls", columns="model", values="dollars_reached_year").reset_index()
CMP = CMP.merge(ACC[ACC.alpha == A_MAIN].pivot(index="calls", columns="model", values="precision_mean").add_prefix("precision_").reset_index(), on="calls")
B.kv([(f"departing clients with money still here, test months", f"{len(pool):,} · ${POOL * yr:,.0f} capped per year"),
      (f"reached at {K_MAIN:,} calls, α = {A_MAIN} ({MAIN} model)",
       f"{len(fa0):,} clients · ${R['main_reached']:,.0f}/yr (95% {R['main_ci'][0]:,.0f} – {R['main_ci'][1]:,.0f})"),
      ("tree − simple, dollars reached per year", f"${R['diff']:,.0f} (95% {R['diff_ci'][0]:,.0f} – {R['diff_ci'][1]:,.0f})" if GBM else "—"),
      ("best ÷ worst month precision", f"{R['prec_ratio']:.2f}×")], "8b · Headline")
B.table(HV, f"8b · Headline dollars at {K_MAIN:,} calls, α = {A_MAIN}, with and without the largest clients",
        pct=("share",), money=("reached_per_year", "departing_per_year"), save="m4_8b_headline")
B.table(CMP, f"8c · Simple vs tree model at α = {A_MAIN}: dollars reached per year and precision",
        pct=tuple(c for c in CMP.columns if c.startswith("precision_")),
        money=tuple(c for c in CMP.columns if c in ("simple", "tree")), save="m4_8c_compare")
B.table(ACC[ACC.model == MAIN], f"8d · Accuracy against calls ({MAIN} model; mean over test months, min/max = spread)",
        pct=("precision_mean", "precision_min", "precision_max", "recall_clients", "recall_dollars"), save="m4_8d_accuracy")
B.table(VAL[VAL.model == MAIN].drop(columns="model"), f"8e · Dollars reached and potential savings per year ({MAIN} model, no call cost)",
        pct=("share_of_departing", "share_of_dollars"),
        money=tuple(["dollars_reached_year"] + [f"save_{int(s * 100)}pct_year" for s in SAVE_RATES]), save="m4_8e_value")
BYM = (PM_[(PM_.calls == K_MAIN) & (PM_.alpha == A_MAIN)].pivot(index="month", columns="model", values="precision")
          .add_prefix("precision_").reset_index().merge(FO[["test_month", "auc_simple", "auc_tree"]], left_on="month",
                                                        right_on="test_month").drop(columns="test_month"))
B.table(BYM, f"8f · By test month at {K_MAIN:,} calls, α = {A_MAIN}", pct=tuple(c for c in BYM.columns if c.startswith("precision_")),
        save="m4_8f_by_month")
SC["bin"] = pd.qcut(SC.p.rank(method="first"), 10, labels=False)
CAL = SC.groupby("bin").agg(mean_predicted=("p", "mean"), actual=("y", "mean"), rows=("y", "size")).reset_index()
CAL["ratio"] = CAL.actual / CAL.mean_predicted
CAL_S = SC.assign(b_=pd.qcut(SC.p_lin.rank(method="first"), 10, labels=False)).groupby("b_").agg(
    mean_predicted=("p_lin", "mean"), actual=("y", "mean")).reset_index().rename(columns={"b_": "bin"})
R.update(cal=CAL, cal_simple=CAL_S, bym=BYM, cmp=CMP,
         top_dec_ratio=float(CAL.mean_predicted.iloc[-1] / CAL.actual.iloc[-1]) if CAL.actual.iloc[-1] else np.nan)
for nm_, df_ in (("m4_8_calibration_primary", CAL), ("m4_8_calibration_simple", CAL_S), ("m4_8_by_month", BYM)): figdata(nm_, df_)
B.table(CAL, f"8g · Calibration ({MAIN} model), predicted vs actual by decile", pct=("mean_predicted", "actual"))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a in ALPHAS:
    d = ACC[(ACC.model == MAIN) & (ACC.alpha == a)]; v = VAL[(VAL.model == MAIN) & (VAL.alpha == a)]
    ax[0].plot(d.calls, d.precision_mean, marker="o", label=f"α={a}"); ax[1].plot(d.calls, d.recall_dollars, marker="o", label=f"α={a}")
    ax[2].plot(v.calls, v.dollars_reached_year / 1e6, marker="o", label=f"α={a}")
for mn, _ in MODELS:
    v = VAL[(VAL.model == mn) & (VAL.alpha == A_MAIN)]
    ax[2].plot(v.calls, v.dollars_reached_year / 1e6, ls=":", color="grey" if mn == "simple" else "black", label=f"{mn}, α={A_MAIN}")
for a_, t_ in zip(ax, ["Precision", "Share of departing dollars reached (per month)", "Dollars reached per year ($m)"]):
    a_.set_title(t_); a_.set_xlabel("calls per month"); a_.legend(fontsize=8)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m4_8_accuracy_vs_calls")
fig, ax = plt.subplots(1, len(SAVE_RATES), figsize=(4 * len(SAVE_RATES), 3.6))
for i, s in enumerate(SAVE_RATES):
    for a in ALPHAS:
        v = VAL[(VAL.model == MAIN) & (VAL.alpha == a)]; ax[i].plot(v.calls, v[f"save_{int(s * 100)}pct_year"] / 1e6, marker="o", label=f"α={a}")
    ax[i].set_title(f"Savings/yr at {s:.0%} save rate ($m)"); ax[i].set_xlabel("calls per month"); ax[i].legend(fontsize=8)
savefig(fig, "m4_8_savings_vs_calls")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for mn, _ in MODELS:
    ax[0].plot(BYM.month, BYM[f"precision_{mn}"], marker="o", label=mn)
    ax[1].plot(FO.test_month, FO[f"auc_{mn}"], marker="o", label=mn)
ax[0].set_title(f"Precision by test month ({K_MAIN:,} calls, α={A_MAIN})"); ax[1].set_title("AUC by test month")
for a_ in ax: a_.legend(); a_.tick_params(axis="x", rotation=90)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m4_8_by_month")
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.plot(CAL.mean_predicted, CAL.actual, marker="o"); mx = max(CAL.mean_predicted.max(), CAL.actual.max())
ax.plot([0, mx], [0, mx], color="grey", lw=0.8); ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Calibration by decile")
savefig(fig, "m4_8_calibration")
B.show(t0)

## §9 When signals fire

**Event time** (describes behaviour; uses the exit date): money-still-here months of leavers aligned on the month
the money left, against stayers aligned on a pseudo-exit month drawn by a fixed hash. Only **change** features are
traced, so a client's size cannot pass for a signal, and every comparison is made **within the same size band**:
thresholds (stayers' 10th/90th percentile) are set per band and the band lifts are averaged with the leavers' band
mix. Onset = the earliest month from which the lift stays ≥ 1.5 through month −1. Drains already in progress are
listed separately as **late warnings**. **Model view:** each feature's contribution to leavers' scores minus
stayers', month by month; the total gap shows whether the score rises toward the exit. **Queue time:** how many
months before the money left a client first reached the list.

In [ ]:
# =====================================================================
# 9 · WHEN SIGNALS FIRE                                              [OUTPUT BLOCK 9]
# =====================================================================
t0 = time.time(); B = Block("§9 · When signals fire")
CHG_FORMS = ("change vs 4–6 months ago", "change vs last 3 months")
CHG = [f for f in IMP.feature if form_of(f) in CHG_FORMS or f in ("dep_bal_vs_norm", "dep_drawdown")]
TIMING = CHG[:N_TIMING] + [f for f in LATE_FEATS if f in FEATS]
span_pe = (LAST_LABEL_T + H) - 12 + 1
ev_base = (MF.withColumn("e", F.when(F.col("ab_t").isNotNull(), F.col("ab_t"))
                              .when(F.col("last_live_t") == END_T, F.lit(12) + F.expr(f"pmod(hash(cust_pwr_id), {span_pe})")))
             .withColumn("grp", F.when(F.col("ab_t").isNotNull(), F.lit("leaver")).otherwise(F.lit("stayer")))
             .filter(F.col("e").isNotNull() & ((F.col("grp") == "stayer") | (F.col("e") <= LAST_LABEL_T + H)))
             .withColumn("rel", F.col("t") - F.col("e")).filter(F.col("rel").between(-EVENT_PRE, -1)))
_cnt = {r["grp"]: r["n"] for r in ev_base.groupBy("grp").agg(F.countDistinct("cust_pwr_id").alias("n")).collect()}
def sub(n_per_group, salt):
    fr = {g: int(1e6 * min(1.0, n_per_group / max(_cnt.get(g, 1), 1))) for g in ("leaver", "stayer")}
    return ev_base.filter(F.expr(f"pmod(hash(cust_pwr_id, {salt}), 1000000)") <
                          F.when(F.col("grp") == "leaver", F.lit(fr["leaver"])).otherwise(F.lit(fr["stayer"])))
bandx = F.when(F.col("bal_norm") < 1e5, 0).when(F.col("bal_norm") < 1e6, 1).when(F.col("bal_norm") < 1e7, 2).otherwise(3)
EV = topd(sub(MAX_EVENT_CUST, SEED + 1).select("cust_pwr_id", "grp", "rel", bandx.alias("band"),
                                               *[F.col(f).cast("double").alias(f) for f in dict.fromkeys(TIMING)]))
lv_rows, st_rows = EV[EV.grp == "leaver"], EV[EV.grp == "stayer"]
rels = list(range(-EVENT_PRE, 0))
wts = lv_rows.groupby(["rel", "band"]).size().unstack(fill_value=0).reindex(rels, fill_value=0)
tim, LIFT = [], {}
for f in dict.fromkeys(TIMING):
    Lb = {"falls": [], "rises": []}
    for bnd in sorted(EV.band.unique()):
        s_ = st_rows[st_rows.band == bnd][f].dropna()
        if len(s_) < 50: continue
        q10, q90 = s_.quantile(0.10), s_.quantile(0.90)
        e_ = EV[EV.band == bnd]; x = e_[f]
        for side, fire in (("falls", (x < q10) & x.notna()), ("rises", (x > q90) & x.notna())):
            r_ = fire.groupby([e_.grp, e_.rel]).mean()
            if "leaver" in r_.index.get_level_values(0) and "stayer" in r_.index.get_level_values(0):
                lift = (r_["leaver"] / r_["stayer"].replace(0, np.nan)).reindex(rels)
                w_ = (wts[bnd] if bnd in wts.columns else pd.Series(0, index=rels)).astype(float)
                Lb[side].append((lift.fillna(0) * w_ * lift.notna(), w_ * lift.notna()))
    comb = {k: (sum(n for n, _ in v) / sum(d for _, d in v).replace(0, np.nan)) if v else pd.Series(np.nan, index=rels)
            for k, v in Lb.items()}
    side = "falls" if np.nanmean(comb["falls"].loc[-3:-1]) >= np.nanmean(comb["rises"].loc[-3:-1]) else "rises"
    Lf = comb[side]; on = None
    for r_ in range(-1, -EVENT_PRE - 1, -1):
        if pd.notna(Lf.get(r_)) and Lf[r_] >= LIFT_BAR: on = r_
        else: break
    LIFT[f] = Lf
    tim.append({"feature": f, "label": LABEL[f], "set": FAM[f], "kind": "late warning" if f in LATE_FEATS else "early signal",
                "leavers": side, "onset_month": on, **{f"lift_{-k}m": Lf.get(k) for k in (-9, -6, -3, -1)},
                "coverage": float(lv_rows[f].notna().mean())})
TIM = pd.DataFrame(tim).sort_values(["kind", "onset_month", "lift_1m"], ascending=[True, True, False], na_position="last").reset_index(drop=True)
R.update(tim=TIM, n_pay_onset6=int(((TIM.onset_month <= -6) & TIM.set.str.startswith("payment")).sum()),
         n_ev_leavers=int(lv_rows.cust_pwr_id.nunique()), n_ev_stayers=int(st_rows.cust_pwr_id.nunique()))
B.table(TIM, "9a · Event time, size-matched: when each change signal fires (lift = leavers ÷ stayers of the same size band)",
        pct=("coverage",), save="m4_9a_onset")
fig, ax = plt.subplots(figsize=(12, 0.32 * len(TIM) + 1.5))
M_ = np.array([LIFT[f].values for f in TIM.feature], dtype=float)
im = ax.imshow(np.clip(M_, 0, 5), aspect="auto", cmap="Reds", vmin=1, vmax=5)
ax.set_yticks(range(len(TIM))); ax.set_yticklabels([f"{l[:62]} ({d})" for l, d in zip(TIM.label, TIM.leavers)], fontsize=8)
ax.set_xticks(range(len(rels))); ax.set_xticklabels(rels); ax.set_xlabel("months before the money left")
ax.set_title("Lift over size-matched stayers (1 = no difference, capped at 5)"); fig.colorbar(im, ax=ax, shrink=0.6)
savefig(fig, "m4_9a_onset_heatmap")
figdata("m4_9a_lift_by_month", pd.DataFrame([{"feature": f, "label": LABEL[f], "rel": r_, "lift": LIFT[f].get(r_)}
                                              for f in TIM.feature for r_ in rels]))

# ── model view: contributions, leavers minus stayers, by month ───────
EVS = topd(sub(N_SHAP_CUST, SEED + 2).select("grp", "rel", *[F.col(f).cast("float").alias(f) for f in FEATS]))
Xe = EVS[FEATS].to_numpy(np.float32)
Ce = gbm_predict(MODEL["G"], Xe, contrib=True) if MODEL["G"] else None
if Ce is None: Ce = (transform(Xe, MODEL["P"]) * MODEL["b"][1:].astype(np.float32))[:, :len(FEATS)]
CEdf = pd.DataFrame(Ce, columns=FEATS).assign(grp=EVS.grp.values, rel=EVS.rel.values)
mg = CEdf.groupby(["grp", "rel"]).mean()
gap = (mg.loc["leaver"] - mg.loc["stayer"]).reindex(rels)
TOTAL = gap.sum(axis=1)
top15 = list(IMP.feature.head(15))
MVT = gap[top15].T; MVT.columns = [str(c) for c in rels]
MVT.insert(0, "feature", [LABEL[f][:70] for f in top15]); MVT = MVT.reset_index(drop=True)
MVT.loc[len(MVT)] = ["ALL FEATURES (total log-odds gap)"] + list(TOTAL.values)
g6 = TOTAL.get(-6, np.nan); g1 = TOTAL.get(-1, np.nan)
R.update(model_view=MVT, gap6=float(g6), gap1=float(g1), gap_ratio=float(g6 / g1) if g1 else np.nan,
         lead6=[LABEL[f][:60] for f in gap.loc[-6].sort_values(ascending=False).head(5).index],
         lead1=[LABEL[f][:60] for f in gap.loc[-1].sort_values(ascending=False).head(5).index])
B.table(MVT, "9b · Model view: contribution to leavers' score minus stayers' (log-odds) by month before the money left",
        save="m4_9b_model_view")
B.kv([("total gap 6 months out ÷ 1 month out", f"{R['gap_ratio']:.2f} (model 1 was ≈ 0.8: flat)"),
      ("largest drivers 6 months out", " · ".join(R["lead6"])), ("largest drivers 1 month out", " · ".join(R["lead1"]))],
     "9b · Early vs late drivers")
fig, ax = plt.subplots(figsize=(8, 3.6)); ax.plot(rels, TOTAL.values, marker="o", color="#c00000")
ax.set_title("Leavers' score minus stayers' (log-odds), all features"); ax.set_xlabel("months before the money left")
savefig(fig, "m4_9b_total_gap")
figdata("m4_9b_total_gap", pd.DataFrame({"rel": rels, "gap": TOTAL.values}))
figdata("m4_9b_model_view", MVT)

# ── queue time ───────────────────────────────────────────────────────
fq = fa0.copy(); fq["lead"] = (fq.ab_t - fq["tm"]).astype(int)
fq["reason_set"] = fq.r1.map(lambda r: FAMOF.get(r, "?")); fq["reason_kind"] = fq.r1.map(lambda r: "late warning" if r in LATE_FEATS else "early signal")
LEAD = fq.groupby("lead").agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index()
LEAD["share_clients"] = LEAD.clients / LEAD.clients.sum(); LEAD["share_dollars"] = LEAD.capped / LEAD.capped.sum()
RS = fq.groupby(["reason_kind", "reason_set"]).agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index() \
       .sort_values("clients", ascending=False)
RS["share"] = RS.clients / RS.clients.sum()
# sustained alert: first of ≥ 2 consecutive months on the list, before the money left
rkm = f"rank_{MAIN}_{A_MAIN}"
onl = Q[(Q[rkm] <= K_MAIN) & Q.ab_t.notna() & (Q["tm"] < Q.ab_t) & Q.cust_pwr_id.isin(set(fq.cust_pwr_id))][["cust_pwr_id", "tm", "ab_t", "capped"]]
_on = set(zip(onl.cust_pwr_id, onl["tm"]))
onl = onl[[(c_, t_ + 1) in _on for c_, t_ in zip(onl.cust_pwr_id, onl["tm"])]]
SUS = onl.sort_values("tm").drop_duplicates("cust_pwr_id")
SUS = SUS.assign(lead=(SUS.ab_t - SUS["tm"]).astype(int))
SUS["lead_band"] = SUS.lead.map(lambda l: "7+" if l >= 7 else str(l))
LEAD_S = SUS.groupby("lead_band").agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index()
_single = fq[~fq.cust_pwr_id.isin(set(SUS.cust_pwr_id))]
LEAD_S.loc[len(LEAD_S)] = ["one-month blips only", len(_single), float(_single.capped.sum())]
LEAD_S["share_clients"] = LEAD_S.clients / max(len(fq), 1); LEAD_S["share_dollars"] = LEAD_S.capped / max(fq.capped.sum(), 1)
R.update(lead_sus=LEAD_S, lead3_sus=float((SUS.lead >= 3).sum() / max(len(fq), 1)),
         lead3_sus_usd=float(SUS.loc[SUS.lead >= 3, "capped"].sum() / max(fq.capped.sum(), 1)),
         single_only=float(len(_single) / max(len(fq), 1)), sus_ids=SUS)
B.table(LEAD_S, "9c · Sustained alerts (first of ≥ 2 consecutive months on the list): months before the money left",
        pct=("share_clients", "share_dollars"), money=("capped",), save="m4_9c_lead_sustained")
figdata("m4_9c_lead_sustained", LEAD_S)
R.update(lead=LEAD, reason_sets=RS, lead3=float((fq.lead >= 3).mean()) if len(fq) else np.nan,
         lead3_usd=float(fq.loc[fq.lead >= 3, "capped"].sum() / max(fq.capped.sum(), 1)),
         late_share=float((fq.reason_kind == "late warning").mean()) if len(fq) else np.nan)
B.table(LEAD, f"9c · Queue time: months from the first alert to the money leaving ({K_MAIN:,} calls, α = {A_MAIN})",
        pct=("share_clients", "share_dollars"), money=("capped",), save="m4_9c_lead")
B.table(RS, "9c · Top reason at the first alert", pct=("share",), money=("capped",))

# ── leavers vs stayers on change features ────────────────────────────
BEHAV = [f for f in ["dep_bal_vs_norm", "dep_d6", "ntxn_all_out__d6", "ntxn_all_in__d6", "amt_all_out__d6", "amt_all_in__d6",
                     "n_fi_ach_out__d6", "n_fi_out__d6", "ncpty_all_out__d6", "ncpty_all_in__d6", "active_days_out__d6",
                     "active_days_in__d6"] if f in MF.columns][:12]
EB = topd(sub(MAX_EVENT_CUST, SEED + 1).select("grp", "rel", *[F.col(f).cast("double").alias(f) for f in BEHAV]))
BV = EB.groupby(["grp", "rel"])[BEHAV].mean()
fig, axs = plt.subplots(int(np.ceil(len(BEHAV) / 4)), 4, figsize=(16, 3.1 * int(np.ceil(len(BEHAV) / 4))))
for a_, f in zip(np.ravel(axs), BEHAV):
    for g_, col in (("leaver", "#c00000"), ("stayer", "#1f4e79")):
        a_.plot(rels, BV.loc[g_, f].reindex(rels), marker="o", ms=3, color=col, label=g_)
    a_.axhline(0, color="grey", lw=0.6); a_.set_title(label(f)[:50], fontsize=9); a_.tick_params(labelsize=8)
for a_ in np.ravel(axs)[len(BEHAV):]: a_.axis("off")
np.ravel(axs)[0].legend(fontsize=8); fig.suptitle("Mean change (log ratio) by month before the money left — money-still-here months")
savefig(fig, "m4_9d_leavers_vs_stayers")
LS = pd.DataFrame([{"feature": label(f), "month": k, "leavers": BV.loc["leaver", f].get(k), "stayers": BV.loc["stayer", f].get(k)}
                   for f in BEHAV for k in (-12, -6, -3, -1)]).pivot(index="feature", columns="month", values=["leavers", "stayers"])
LS.columns = [f"{a} {b}" for a, b in LS.columns]; LS = LS.reset_index()
R.update(ls=LS)
figdata("m4_9d_leavers_vs_stayers_by_month", BV.reset_index())
figdata("m4_9c_lead", LEAD); figdata("m4_9c_reasons", RS)
B.table(LS, "9d · Leavers vs stayers: mean change 12, 6, 3 and 1 months before", save="m4_9d_leavers_vs_stayers")
B.kv([("leavers / stayers traced", f"{R['n_ev_leavers']:,} / {R['n_ev_stayers']:,}"),
      ("payment change features firing 6+ months out (size-matched)", R["n_pay_onset6"]),
      ("first alerted 3+ months before the money left", f"{R['lead3']:.1%} of clients, {R['lead3_usd']:.1%} of dollars"),
      ("sustained alert 3+ months before the money left", f"{R['lead3_sus']:.1%} of reached clients, {R['lead3_sus_usd']:.1%} of dollars"),
      ("reached only by one-month blips", f"{R['single_only']:.1%} of reached clients"),
      ("first alerts whose top reason is a drain already in progress", f"{R['late_share']:.1%}")], "9e · Summary")
B.show(t0)

## §10 Example clients for RM managers

Chosen **by rule** from the test months (1,000 calls, α = 0.5, primary model). The alert date is the **sustained
alert**: the first of ≥ 2 consecutive months on the list (earlier one-month blips are mentioned, not counted).
Departing examples must pass the §7 check (verdict *departed* or *moved to another bank*) and have **12 steady
months** before the alert (balance ≥ 10% of normal every month); a reached client whose money arrived recently gets
its own example, *new money that left*.

For each client: the **balance in dollars** against its normal balance, the months on the list, the model's reasons,
and a panel for each **change signal that fired** (§9a) with the stayers' threshold; ▼/▲ in the table marks a month
beyond it. The data behind every example is in `figdata/m4_10_examples_long.csv`. Ids are one-way hashes; full ids
are only in `m4_10_examples_FULL_IDS.csv`.

In [ ]:
# =====================================================================
# 10 · EXAMPLE CLIENTS FOR RM MANAGERS                             [OUTPUT BLOCK 10]
# =====================================================================
t0 = time.time(); B = Block("§10 · Example clients")
def play(feat):
    f = feat.replace("miss·", "")
    if re.search(r"(n_fi|^fi_|_fi_|selfpay|top_fi|^share_|n_rails)", f):
        return ("Competitive threat: money appears to be moving to another bank. Compare pricing, earnings credit "
                "and services with the likely competitor; offer a retention package.")
    if f.startswith("dep_"):
        return ("Balance drawdown: deposits are falling ahead of the payments. Ask about cash plans; offer sweep, "
                "money-market or liquidity options to keep the operating balance here.")
    return ("Business activity is shrinking: fewer payments received and fewer trading partners. Check the client's "
            "business; offer credit, working-capital or receivables support.")

OKV = {"departed", "moved to another bank"}
rk = f"rank_{MAIN}_{A_MAIN}"
Qx = Q[["cust_pwr_id", "tm", rk, "p", "bal", "bal_norm", "capped", "ab_t", "exit_type", "y", "r1", "r2", "r3"]]
onl = Qx[(Qx[rk] <= K_MAIN) & ((Qx.ab_t.isna()) | (Qx["tm"] < Qx.ab_t))]
_on = set(zip(onl.cust_pwr_id, onl["tm"]))
sus = onl[[(c_, t_ + 1) in _on for c_, t_ in zip(onl.cust_pwr_id, onl["tm"])]].sort_values("tm").drop_duplicates("cust_pwr_id")
lst = sus[["cust_pwr_id", "tm", "bal", "bal_norm", "r1"]].rename(columns={"tm": "T_alert", "bal": "bal_at_alert", "bal_norm": "norm_at_alert"})
dp = pool[["cust_pwr_id", "ab_t", "exit_type", "capped", "bal_norm"]].merge(lst, on="cust_pwr_id", how="left")
dp["lead"] = dp.ab_t - dp.T_alert; dp["pct_at_alert"] = dp.bal_at_alert / dp.norm_at_alert
hit = dp[dp.T_alert.notna()]

HIST = {}
def steady(ids_):
    """True if the client's balance stayed ≥ 10% of normal in each of the STEADY_M months before its alert."""
    new = [c for c in ids_ if c not in HIST]
    if new:
        h = topd(FEAT.filter(F.col("cust_pwr_id").isin(new)).select("cust_pwr_id", "t", "bal"))
        for c in new: HIST[c] = h[h.cust_pwr_id == c].set_index("t").bal
    out = {}
    for c in ids_:
        ta = dp.loc[dp.cust_pwr_id == c, "T_alert"]; nm = dp.loc[dp.cust_pwr_id == c, "norm_at_alert"]
        if not len(ta) or pd.isna(ta.iloc[0]): out[c] = True; continue
        w = HIST[c].loc[int(ta.iloc[0]) - STEADY_M:int(ta.iloc[0]) - 1]
        out[c] = len(w) >= STEADY_M and bool((w >= 0.10 * float(nm.iloc[0])).all())
    return out
chosen, EXS = set(), []
def pick(df, kind, sort="bal_norm", need_steady=True, want_steady=True):
    d = df[~df.cust_pwr_id.isin(chosen)].sort_values(sort, ascending=False).head(25)
    if not len(d): return
    v = audit(list(d.cust_pwr_id)).set_index("cust_pwr_id").verdict
    ok = [c for c in d.cust_pwr_id if v.get(c) in OKV]
    if need_steady and ok:
        st = steady(ok); ok = [c for c in ok if st[c] == want_steady]
    if ok: chosen.add(ok[0]); EXS.append((kind, ok[0]))
pick(hit[hit.lead >= 4], "Early catch")
pick(hit[(hit.pct_at_alert >= 0.9) & hit.r1.map(lambda r: isinstance(r, str) and not r.startswith("dep_"))], "Payments moved before the balance")
pick(hit[hit.exit_type == "balance exit"], "Drained without closing")
pick(hit[hit.lead <= 2], "Late catch")
pick(hit, "New money that left", want_steady=False)
never = set(SC.loc[SC.ab_t.isna(), "cust_pwr_id"]) - set(SC.loc[SC.y == 1, "cust_pwr_id"])
fal = sus[sus.cust_pwr_id.isin(never)].merge(Qx.groupby("cust_pwr_id").p.max().rename("pmax"), on="cust_pwr_id").sort_values("pmax", ascending=False)
if len(fal): chosen.add(fal.cust_pwr_id.iloc[0]); EXS.append(("False alarm (most confident wrong call)", fal.cust_pwr_id.iloc[0]))
pick(dp[dp.T_alert.isna() & ~dp.cust_pwr_id.isin(set(onl.cust_pwr_id))], "Missed", sort="capped")
ids = [c for _, c in EXS]

# ── change signals with a stayers' threshold (from §9a) ─────────────
tim_on = TIM.dropna(subset=["onset_month"])
THR = {f: (st_rows[f].quantile(0.10) if d_ == "falls" else st_rows[f].quantile(0.90), d_)
       for f, d_ in zip(tim_on.feature, tim_on.leavers) if f in FEAT.columns}
PAN = topd(FEAT.filter(F.col("cust_pwr_id").isin(ids)).select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t",
                                                              *[F.col(f).cast("double").alias(f) for f in THR]))
fmtv = lambda v: "—" if pd.isna(v) else f"{v:+.2f}"
def beyond(v, f):
    th, d_ = THR[f]; return pd.notna(v) and ((v < th) if d_ == "falls" else (v > th))
EXOUT, full_ids, LONG = [], [], []
for n, (kind, c) in enumerate(EXS, 1):
    pan = PAN[PAN.cust_pwr_id == c].sort_values("t"); sc = Qx[Qx.cust_pwr_id == c].sort_values("tm")
    e = int(pan.ab_t.dropna().iloc[0]) if pan.ab_t.notna().any() else None
    s_row = sus[sus.cust_pwr_id == c]
    anchor = s_row.iloc[0] if len(s_row) else (sc.sort_values(rk).iloc[0] if len(sc) else None)
    ta = int(anchor["tm"]) if anchor is not None else None
    lo_t = int(min(sc["tm"].min() if len(sc) else pan.t.max(), (e - 12) if e is not None else 10 ** 6, (ta - 6) if ta is not None else 10 ** 6))
    hi_t = int(e + 1) if e is not None else int(sc["tm"].max() + 1 if len(sc) else pan.t.max())
    pan = pan[(pan.t >= max(lo_t, 0)) & (pan.t <= hi_t)].set_index("t")
    ref = float(anchor.bal_norm) if anchor is not None else float(pan.bal_norm.dropna().median())
    alerts = sc[sc[rk] <= K_MAIN]
    blips = [ym_of(t_) for t_ in alerts["tm"] if ta is not None and t_ < ta]
    fired = []
    for f in THR:
        v = pan[f]; m_ = v[[beyond(x, f) for x in v]].index
        m_ = m_[m_ < e] if e is not None else m_
        if len(m_): fired.append((int(m_.min()), f))
    fired.sort()
    reasons = [r for r in (anchor[["r1", "r2", "r3"]].tolist() if anchor is not None and "r1" in anchor else []) if isinstance(r, str)]
    if anchor is not None and not reasons:
        rr = sc[sc["tm"] == ta]; reasons = [r for r in rr[["r1", "r2", "r3"]].iloc[0].tolist() if isinstance(r, str)] if len(rr) else []
    trig = list(dict.fromkeys([f for f in reasons if f in THR] + [f for _, f in fired]))[:N_TRIG]
    rows = []
    for t_, r in pan.iterrows():
        s_ = sc[sc["tm"] == t_]
        row = {"month": ym_of(t_), "vs_exit": (int(t_ - e) if e is not None else None), "balance": r.bal, "normal": ref,
               "balance_pct_normal": r.bal / ref if ref else np.nan, "score": float(s_.p.iloc[0]) if len(s_) else np.nan,
               "rank": f"{int(s_[rk].iloc[0]):,}" if len(s_) else "—", "on_list": bool(len(s_) and s_[rk].iloc[0] <= K_MAIN)}
        for f in trig:
            v = r.get(f, np.nan); h_ = beyond(v, f)
            row[LABEL[f][:45]] = fmtv(v) + ((" ▼" if THR[f][1] == "falls" else " ▲") if h_ else "")
            LONG.append({"example": n, "kind": kind, "client": cid(c), "month": ym_of(t_), "vs_exit": row["vs_exit"], "variable": f,
                         "label": LABEL[f], "value": v, "threshold": THR[f][0], "direction": THR[f][1], "fired": bool(h_),
                         "reason_at_alert": f in reasons})
        for var, val in (("balance", r.bal), ("normal", ref), ("score", row["score"])):
            LONG.append({"example": n, "kind": kind, "client": cid(c), "month": ym_of(t_), "vs_exit": row["vs_exit"], "variable": var,
                         "label": var, "value": val, "threshold": np.nan, "direction": None,
                         "fired": row["on_list"] if var == "score" else None, "reason_at_alert": False})
        row["reasons"] = " · ".join(LABEL.get(x, x)[:50] for x in s_[["r1", "r2", "r3"]].iloc[0]) if len(s_) else ""
        rows.append(row)
    TAB = pd.DataFrame(rows)
    trig_at = "; ".join(f"{LABEL[f][:55]} = {fmtv(pan[f].get(ta))} (stayers' threshold {THR[f][0]:+.2f})" for f in trig) if ta is not None else ""
    fired_txt = "; ".join(f"{LABEL[f][:55]} (from {m - e:+d})" for m, f in fired[:5]) if e is not None else \
                "; ".join(f"{LABEL[f][:55]} (from {ym_of(m)})" for m, f in fired[:5])
    rsn = " · ".join(LABEL.get(x, x)[:55] for x in reasons)
    ex_type = dp.loc[dp.cust_pwr_id == c, "exit_type"].iloc[0] if (dp.cust_pwr_id == c).any() else None
    blip_txt = f" (earlier one-month appearances: {', '.join(blips)})" if blips else ""
    if kind.startswith("False"):
        text = (f"On the list from {ym_of(ta)} for {len(alerts)} month(s), score up to {sc.p.max():.1%}, and stayed. Reasons: {rsn}. "
                + (f"Change signals beyond the stayers' threshold: {fired_txt}. " if fired_txt else "")
                + f"At {K_MAIN:,} calls about {1 - float(ACC[(ACC.model == MAIN) & (ACC.alpha == A_MAIN) & (ACC.calls == K_MAIN)].precision_mean.iloc[0]):.0%} "
                "of calls look like this one.")
    elif kind == "Missed":
        text = (f"The money left in {ym_of(e)} ({ex_type}); the client never reached the top {K_MAIN:,}"
                + (f" (best rank {int(anchor[rk]):,} in {ym_of(ta)}; reasons then: {rsn})." if anchor is not None else ".")
                + (f" Change signals that fired: {fired_txt}." if fired_txt else " No change signal crossed the stayers' threshold."))
    else:
        text = (f"Sustained alert from {ym_of(ta)}, {int(e - ta)} month(s) before the money left ({ex_type}, {ym_of(e)}){blip_txt}. "
                f"Balance then ${anchor.bal:,.0f} against a ${anchor.bal_norm:,.0f} normal ({anchor.bal / anchor.bal_norm:.0%}). "
                f"Model's reasons: {rsn}. " + (f"Change signals at the alert: {trig_at}. " if trig_at else "")
                + (f"Signals that fired, in order: {fired_txt}. " if fired_txt else "")
                + (f"How an RM could have intervened: {play(reasons[0])}" if reasons else ""))
        if kind == "New money that left":
            text = "Balance arrived within the 12 months before the alert. " + text
    xs = [ym_of(t_) for t_ in pan.index]
    fig, axs = plt.subplots(1 + len(trig), 1, figsize=(11, 2.6 + 1.7 * len(trig)), sharex=True, squeeze=False)
    axs = axs[:, 0]
    axs[0].plot(xs, pan.bal / 1e6, marker="o", color="#1f4e79", label="balance ($m)")
    axs[0].axhline(ref / 1e6, color="grey", ls="--", lw=1, label="normal balance")
    for t_ in alerts["tm"]:
        if t_ in pan.index:
            for ax_ in axs: ax_.axvline(ym_of(t_), color="#c55a11", alpha=0.25, lw=9)
    if e is not None and e in pan.index:
        for ax_ in axs: ax_.axvline(ym_of(e), color="#c00000", ls="--", lw=1.2)
    axs[0].set_title(f"Example {n} · {kind} · client {cid(c)}  (orange = on the list, red dashed = money left)")
    axs[0].legend(loc="lower left", fontsize=8); axs[0].set_ylabel("$m")
    for ax_, f in zip(axs[1:], trig):
        v = pan[f]; th, d_ = THR[f]
        ax_.plot(xs, v.values, marker="o", ms=3, color="#333"); ax_.axhline(th, color="#c00000", lw=0.8, ls=":")
        hm = v[[beyond(x, f) for x in v]]
        ax_.plot([ym_of(t_) for t_ in hm.index], hm.values, "o", color="#c00000", ms=6)
        ax_.set_title(LABEL[f][:80] + f" — leavers {d_}; red dots = beyond the stayers' threshold", fontsize=9)
    axs[-1].tick_params(axis="x", rotation=90)
    savefig(fig, f"m4_10_example_{n}")
    EXOUT.append(dict(n=n, kind=kind, client=cid(c), normal=ref, text=text, table=TAB, chart=f"m4_10_example_{n}"))
    full_ids.append({"example": n, "kind": kind, "display_id": cid(c), "cust_pwr_id": c})
    B.note(f"<b>Example {n} · {kind} · client {cid(c)}</b> — {_html.escape(text)}")
    B.table(TAB, f"Example {n}: month by month (▼/▲ = beyond the stayers' threshold)", pct=("balance_pct_normal", "score"),
            money=("balance", "normal"))
pd.DataFrame(full_ids).to_csv(OUT / "m4_10_examples_FULL_IDS.csv", index=False)
figdata("m4_10_examples_long", pd.DataFrame(LONG))
figdata("m4_10_thresholds", pd.DataFrame([{"feature": f, "label": LABEL[f], "threshold": th, "leavers": d_} for f, (th, d_) in THR.items()]))
R.update(examples=EXOUT)
if len(EXS) < 7: B.check("examples", "INFO", f"only {len(EXS)} of 7 rules found a client")
B.show(t0)

## §13 Data for the Treasury Management briefing

Everything on the briefing page is computed here from this run, on one target definition: the balance trajectory (future leavers vs
comparable stayers, aligned on the event month, weighted to the leavers' size mix), the six business signal groups, first-alert lead time per
unique departing relationship, list performance at sizes 25 – 1,000, and three anonymised examples taken from §10.

In [ ]:
# =====================================================================
# 13 · BRIEFING DATA — trajectory, signal groups, lead time, list sizes, examples   [OUTPUT BLOCK 13]
# =====================================================================
t0 = time.time(); B = Block("§13 · Data for the Treasury Management briefing")
BR = {}                                                     # everything the briefing uses (pickled with it)
rk = f"rank_{MAIN}_{A_MAIN}"

# ── plain-English names ───────────────────────────────────────────────
_PB = {"amt_all_in": "dollars received", "amt_all_out": "dollars paid", "amt_ext_in": "dollars received from outside PNC",
       "amt_ext_out": "dollars paid outside PNC", "amt_int_in": "dollars received from other PNC clients", "amt_int_out": "dollars paid to other PNC clients",
       "ntxn_all_in": "number of payments received", "ntxn_all_out": "number of payments made", "ntxn_ext_in": "payments received from outside PNC",
       "ntxn_ext_out": "payments made outside PNC", "ntxn_int_in": "payments received from other PNC clients", "ntxn_int_out": "payments made to other PNC clients",
       "active_days_in": "days with money received", "active_days_out": "days with payments made", "ncpty_all_in": "number of payers",
       "ncpty_all_out": "number of payees", "ncpty_ext_in": "number of outside payers", "ncpty_ext_out": "number of outside payees",
       "selfpay_amt_out": "transfers to the client's own accounts at other banks", "selfpay_share_out": "share of payments to the client's own accounts at other banks",
       "n_fi_out": "number of other banks paid", "n_fi_ach_out": "number of other banks paid by ACH", "n_fi_in": "number of other banks paying in",
       "top_fi_share_out": "share of outside payments going to one bank", "passthrough_ratio": "share of money received that is paid straight out",
       "net_flow_all": "net payment flow (received minus paid)", "net_flow_ext": "net flow with outside parties", "avg_ticket_in": "average payment size received",
       "avg_ticket_out": "average payment size paid", "top1_share_in": "share received from the largest payer", "top1_share_out": "share paid to the largest payee",
       "hhi_in": "concentration of payers", "hhi_out": "concentration of payees", "retention_in": "share of last month's payers still paying",
       "retention_out": "share of last month's payees still paid", "pay_log_flow": "size of total payment flow", "pay_silent": "no payment activity this month",
       "pay_months_since_act": "months since the last payment", "pay_seen": "has payment history", "dep_log_bal": "size of balance",
       "dep_log_norm": "size of normal balance"}
_SIDE = {"in": ("received", "payers"), "out": ("paid", "payees")}
_PB.update({"earnings_credit_rate": "earnings credit rate", "curr_int_rate": "interest rate", "dep_earnings_credit_rate": "earnings credit rate",
            "dep_curr_int_rate": "interest rate", "dep_at_start": "client already present when the data start", "dep_tenure": "length of relationship",
            "dep_n_accts": "number of accounts", "dep_n_live": "number of live accounts", "dep_siblings": "other entities in the same relationship",
            "dep_opened_3m": "accounts opened in the last 3 months", "dep_closed_3m": "accounts closed in the last 3 months"})
_DEPP = {"dep_d1": ("balance", "vs last month"), "dep_d3": ("balance", "vs the previous 3 months"), "dep_d6": ("balance", "vs 4–6 months earlier"),
         "dep_d12": ("balance", "vs a year earlier"), "dep_drawdown": ("balance", "vs its 12-month peak"), "dep_bal_vs_norm": ("balance", "vs normal"),
         "dep_eom_vs_avg": ("month-end balance", "vs the monthly average"), "dep_live_d6": ("number of live accounts", "vs 4–6 months earlier"),
         "dep_sib_d6": ("other balances in the same relationship", "vs 4–6 months earlier"), "dep_vol6": ("balance volatility over 6 months", ""),
         "dep_range": ("swing of the daily balance within the month", ""), "dep_siblings": ("other entities in the same relationship", ""),
         "dep_earnings_credit_rate_d6": ("earnings credit rate", "vs 4–6 months earlier"), "dep_curr_int_rate_d6": ("interest rate", "vs 4–6 months earlier")}
def plain_base(b):
    if b in _PB: return _PB[b]
    if b in _DEPP: return (_DEPP[b][0] + " " + _DEPP[b][1]).strip()
    if b in DEP: return re.sub(r"\s*\((log|log ratio)\)", "", DEP[b][1])
    m = re.match(r"same_naics2_share_(in|out)$", b)
    if m: return f"share of money {_SIDE[m.group(1)][0]} with clients in the same industry"
    m = re.match(r"n_hub_t(\d)_(in|out)$", b)
    if m: return f"number of very large network {_SIDE[m.group(2)][1]} (hub tier {m.group(1)}+)"
    m = re.match(r"hub_t(\d)_share_(in|out)$", b)
    if m: return f"share of money {_SIDE[m.group(2)][0]} with very large network counterparties (hub tier {m.group(1)}+)"
    m = re.match(r"fi_coverage_(in|out)$", b)
    if m: return f"share of outside money {_SIDE[m.group(1)][0]} whose bank is known"
    m = re.match(r"ext_(business|person)_share_(in|out)$", b)
    if m: return f"share of outside money {_SIDE[m.group(2)][0]} with {'businesses' if m.group(1) == 'business' else 'individuals'}"
    m = re.match(r"int_(tm|org)_share_(in|out)$", b)
    if m: return f"share of PNC money {_SIDE[m.group(2)][0]} with {'Treasury Management clients' if m.group(1) == 'tm' else 'organisations'}"
    m = re.match(r"(n_cpty_elig|elig_share)_(in|out)$", b)
    if m: return ("number of established outside " + _SIDE[m.group(2)][1]) if m.group(1) == "n_cpty_elig" else f"share of outside money {_SIDE[m.group(2)][0]} with established counterparties"
    m = re.match(r"pnc_fi_share_(in|out)$", b)
    if m: return f"share of money {_SIDE[m.group(1)][0]} that stays at PNC"
    m = re.match(r"(n_naics2|naics2_top_share|naics2_entropy)_(in|out)$", b)
    if m: return {"n_naics2": "number of industries ", "naics2_top_share": "share with the main industry ", "naics2_entropy": "spread across industries "}[m.group(1)] + _SIDE[m.group(2)][0]
    m = re.match(r"(fi_)?rec_broken_(amt_|net_)?share_(in|out)$", b)
    if m: return (f"share of regular {'other-bank links' if m.group(1) else _SIDE[m.group(3)][1]} that stopped"
                  + (" (by dollars)" if m.group(2) == "amt_" else ""))
    m = re.match(r"(lost|new)_amt_share_(in|out)$", b)
    if m: return f"share of dollars {_SIDE[m.group(2)][0]} with {'lost' if m.group(1) == 'lost' else 'new'} {_SIDE[m.group(2)][1]}"
    m = re.match(r"fi_retention_(in|out)$", b)
    if m: return f"share of other banks still {_SIDE[m.group(1)][0]}"
    m = re.match(r"share_(ACH|WIRE|RTP_PRT|RTP_P2P|CHECK|PCARD)_(in|out)$", b)
    if m: return f"share of money {_SIDE[m.group(2)][0]} by {m.group(1).replace('_PRT', '').replace('_P2P', ' (P2P)')}"
    m = re.match(r"(new|lost|reactivated|retained)_cpty_(int|ext|all)_(in|out)$", b)
    if m: return f"{ {'new': 'new', 'lost': 'lost', 'reactivated': 'returning', 'retained': 'continuing'}[m.group(1)]} {_SIDE[m.group(3)][1]}" + \
                 {"int": " at PNC", "ext": " outside PNC", "all": ""}[m.group(2)]
    m = re.match(r"(new|lost|reactivated|retained)_fi_(in|out)$", b)
    if m: return f"{m.group(1)} other banks ({_SIDE[m.group(2)][0]})"
    m = re.match(r"rec_(standing|broken)_(in|out)$", b)
    if m: return f"regular {_SIDE[m.group(2)][1]}" + (" that stopped" if m.group(1) == "broken" else "")
    s = b
    for a_, z_ in (("amt", "dollars"), ("ntxn", "payments"), ("ncpty", "counterparties"), ("cpty", "counterparties"), ("fi", "banks"),
                   ("ext", "outside PNC"), ("int", "at PNC"), ("in", "received"), ("out", "paid"), ("n", "number of"), ("share", "share")):
        s = re.sub(rf"(^|_){a_}(?=_|$)", rf"\1{z_}", s)
    return s.replace("_", " ")
_CTX = {"d6": "vs 4–6 months earlier", "d3": "vs the previous 3 months", "peer": "vs similar-size clients", "": ""}
def PL(f):
    b, _, tr = f.replace("miss·", "").partition("__")
    return plain_base(b) + ({"d6": " — change vs 4–6 months earlier", "d3": " — change vs the previous 3 months",
                             "peer": " — compared with similar-size clients", "": ""}[tr])
def PLD(f, d):
    """Direction in plain words: 'dollars received fall vs 4–6 months earlier'."""
    b, _, tr = f.replace("miss·", "").partition("__")
    subj, ctx = (_DEPP[b] if b in _DEPP else (plain_base(b), ""))
    ctx = _CTX[tr] or ctx
    return f"{subj} {'drops' if d == 'falls' else 'rises'} {ctx}".strip()
mo = lambda n: f"{int(n)} month{'s' if abs(int(n)) != 1 else ''}"
GROUPS = ["Balance behavior", "Incoming payment activity", "Outgoing and external-bank activity", "Counterparty relationships",
          "Related entities and relationship context", "Pricing and client context"]
def sgroup(f):
    b = f.split("__")[0].replace("miss·", "")
    if b in ("dep_tenure", "dep_at_start", "dep_log_norm", "dep_log_bal", "pay_log_flow") or b.startswith(("dep_earnings", "dep_curr_int", "dep_share_")):
        return "Pricing and client context"
    if b.startswith(("dep_sib", "dep_n_", "dep_live", "dep_opened", "dep_closed")): return "Related entities and relationship context"
    if b.startswith("dep_"): return "Balance behavior"
    if re.search(r"(n_fi|fi_|_fi$|selfpay|top_fi|pnc_fi|cpty_bank)", b): return "Outgoing and external-bank activity"
    if re.search(r"(new_|reactivated_|retained_|lost_|retention|rec_|ncpty|top1_share|hhi|hub|elig|business_share|person_share|int_tm|int_org|naics)", b):
        return "Counterparty relationships"
    if b.endswith("_out") or "_out_" in b: return "Outgoing and external-bank activity"
    return "Incoming payment activity"

# ── base rate and scoreable population (test months) ────────────────
BASE_N, BASE_D = int(SC.y.sum()), len(SC)
base_rate = BASE_N / BASE_D
cov_ = FEAT.filter((F.col("live") == 1) & F.col("t").isin(*TEST_T)).agg(
    F.count("*").alias("n"), F.sum("bal").alias("b"), F.countDistinct("cust_pwr_id").alias("c")).first()
BR.update(base_n=BASE_N, base_d=BASE_D, base_rate=base_rate, n_test=len(TEST_T), test_first=ym_of(TEST_T[0]), test_last=ym_of(TEST_T[-1]),
          eligible_share=BASE_D / max(cov_["n"], 1), eligible_bal_share=float(SC.bal.sum() / max(cov_["b"], 1.0)),
          eligible_clients=int(SC.cust_pwr_id.nunique()), live_clients=int(cov_["c"]), pay_seen=R["pay_seen"], pay_seen_usd=R["pay_seen_usd"],
          n_departing=len(pool), departing_capped=float(pool.capped.sum()), data_through=END_M, last_label=ym_of(LAST_LABEL_T))

# ── performance at several monthly review-list sizes (unique relationships) ──
def first_alerts(K):
    on = Q[Q[rk] <= K]
    fa = on[on.y == 1].sort_values("tm").drop_duplicates("cust_pwr_id").copy()
    fa["lead"] = (fa.ab_t - fa["tm"]).astype(int)
    _on = set(zip(on.cust_pwr_id, on["tm"]))
    l2 = on[(on.y == 1) & [(c_, t_ + 1) in _on for c_, t_ in zip(on.cust_pwr_id, on["tm"])]]
    return on, fa, set(l2.cust_pwr_id)
rows, FA = [], {}
for K in BRIEF_SIZES:
    on, fa, sus2 = first_alerts(K); FA[K] = (fa, sus2)
    pm_ = on.groupby("tm").y.mean(); cap = fa.capped.sort_values(ascending=False)
    rows.append({"list_size": K, "relationships_surfaced": int(on.cust_pwr_id.nunique()), "hit_rate": float(pm_.mean()),
                 "hit_rate_range": f"{pm_.min():.0%}–{pm_.max():.0%}", "lift": float(pm_.mean() / base_rate),
                 "departing_surfaced": len(fa), "recall": len(fa) / max(len(pool), 1),
                 "median_lead_months": float(fa.lead.median()) if len(fa) else np.nan, "surfaced_3plus": float((fa.lead >= 3).mean()) if len(fa) else np.nan,
                 "balance_surfaced": float(cap.sum()), "balance_share": float(cap.sum() / max(pool.capped.sum(), 1.0)),
                 "top5_share": float(cap.head(5).sum() / max(cap.sum(), 1.0)), "two_months_running": len(sus2 & set(fa.cust_pwr_id)) / max(len(fa), 1)})
CURVE = pd.DataFrame(rows); BR["curve"] = CURVE

# ── lead time at the reference list sizes ────────────────────────────
lb = []
for K in BRIEF_LEAD_SIZES:
    fa, sus2 = FA[K] if K in FA else first_alerts(K)[1:]
    band = lambda l: "6 months (the longest the model looks ahead)" if l >= 6 else "3–5 months" if l >= 3 else "1–2 months"
    g = fa.assign(b=fa.lead.map(band)).groupby("b").agg(relationships=("cust_pwr_id", "size"), balance=("capped", "sum"))
    for nm in ["6 months (the longest the model looks ahead)", "3–5 months", "1–2 months"]:
        r_ = g.loc[nm] if nm in g.index else pd.Series({"relationships": 0, "balance": 0.0})
        lb.append({"list_size": K, "first surfaced": nm, "relationships": int(r_.relationships), "share_of_departing": r_.relationships / max(len(pool), 1),
                   "share_of_balance": r_.balance / max(pool.capped.sum(), 1.0)})
    lb.append({"list_size": K, "first surfaced": "in the event month", "relationships": 0, "share_of_departing": 0.0, "share_of_balance": 0.0})
    nr = pool[~pool.cust_pwr_id.isin(set(fa.cust_pwr_id))]
    lb.append({"list_size": K, "first surfaced": "not surfaced", "relationships": len(nr), "share_of_departing": len(nr) / max(len(pool), 1),
               "share_of_balance": nr.capped.sum() / max(pool.capped.sum(), 1.0)})
LEADT = pd.DataFrame(lb)
# by size of relationship, at the first reference size
K0 = BRIEF_LEAD_SIZES[0]; fa0b = FA[K0][0] if K0 in FA else first_alerts(K0)[1]
pb = pool.assign(band=pd.cut(pool.bal_norm, [b_[0] for b_ in BANDS] + [BANDS[-1][1]], labels=[b_[2] for b_ in BANDS], right=False))
pb = pb.merge(fa0b[["cust_pwr_id", "lead"]], on="cust_pwr_id", how="left")
SIZE = (pb.groupby("band", observed=False).agg(departing=("cust_pwr_id", "size"), surfaced=("lead", "count"), median_lead=("lead", "median"),
                                              surfaced_3plus=("lead", lambda s: float((s >= 3).sum())), balance=("capped", "sum")).reset_index())
SIZE["surfaced_share"] = SIZE.surfaced / SIZE.departing.clip(lower=1); SIZE["surfaced_3plus"] = SIZE.surfaced_3plus / SIZE.departing.clip(lower=1)
SIZE["share_of_departing_balance"] = SIZE.balance / max(pb.capped.sum(), 1.0)
BR.update(lead=LEADT, size=SIZE, lead_ref=K0, top1_share=float(pool.capped.max() / max(pool.capped.sum(), 1.0)),
          top5_pool_share=float(pool.capped.nlargest(5).sum() / max(pool.capped.sum(), 1.0)))

# ── balance and incoming-payment trajectory: future leavers vs comparable stayers ──
pay_col = "amt_all_in" if "amt_all_in" in FEAT.columns else ("ntxn_all_in" if "ntxn_all_in" in FEAT.columns else None)
e_ = (F.when(F.col("ab_t").isNotNull(), F.col("ab_t"))
       .when(F.col("last_live_t") == END_T, F.lit(TRJ_PRE) + F.expr(f"pmod(hash(cust_pwr_id, {SEED + 7}), {END_T - TRJ_PRE + 1})")))
P0 = (FEAT.select("cust_pwr_id", "t", "bal", "bal_norm", "msh", "ab_t", "last_live_t", *([pay_col] if pay_col else []))
          .withColumn("e", e_).filter(F.col("e").isNotNull() & ~F.col("cust_pwr_id").isin(list(EXCL) or ["__none__"]))
          .withColumn("grp", F.when(F.col("ab_t").isNotNull(), F.lit("leaver")).otherwise(F.lit("stayer"))))
bandx = F.when(F.col("norm0") < 1e5, 0).when(F.col("norm0") < 1e6, 1).when(F.col("norm0") < 1e7, 2).otherwise(3)
anc = (P0.filter((F.col("t") == F.col("e") - TRJ_PRE) & (F.col("msh") == 1) & F.col("bal_norm").isNotNull())
         .select("cust_pwr_id", F.col("bal_norm").alias("norm0")).withColumn("band", bandx))
TJ = (P0.join(anc, "cust_pwr_id").withColumn("rel", F.col("t") - F.col("e"))
        .filter(F.col("rel").between(-TRJ_PRE, TRJ_POST)).withColumn("bidx", F.col("bal") / F.col("norm0")))
if pay_col:
    wb = Window.partitionBy("cust_pwr_id")
    TJ = (TJ.withColumn("_pb", F.avg(F.when(F.col("rel") <= -TRJ_PRE + 2, F.col(pay_col))).over(wb))
            .withColumn("pidx", F.exp(F.col(pay_col) - F.col("_pb"))))
TA = topd(TJ.groupBy("grp", "band", "rel").agg(
    F.count("*").alias("n"), F.expr("percentile_approx(bidx, 0.5)").alias("bal_med"),
    F.expr("percentile_approx(bidx, 0.25)").alias("bal_p25"), F.expr("percentile_approx(bidx, 0.75)").alias("bal_p75"),
    *([F.expr("percentile_approx(pidx, 0.5)").alias("pay_med")] if pay_col else [])))
NCL = {r_["grp"]: r_["n"] for r_ in TJ.filter(F.col("rel") == -TRJ_PRE).groupBy("grp").agg(F.count("*").alias("n")).collect()}
wmix = TA[(TA.grp == "leaver") & (TA.rel == -TRJ_PRE)].set_index("band").n
wmix = wmix / wmix.sum()
def comb(g, col):
    d_ = TA[TA.grp == g].pivot(index="rel", columns="band", values=col)
    w_ = wmix.reindex(d_.columns).fillna(0.0)
    return (d_.fillna(0.0) * w_).sum(axis=1) / (d_.notna() * w_).sum(axis=1).replace(0, np.nan)
rels_t = list(range(-TRJ_PRE, TRJ_POST + 1))
TRJ = pd.DataFrame({"rel": rels_t})
for g in ("leaver", "stayer"):
    for col in (["bal_med", "bal_p25", "bal_p75"] + (["pay_med"] if pay_col else [])):
        TRJ[f"{g}_{col}"] = comb(g, col).reindex(rels_t).values
def sep_onset(a, b, frac):
    on_ = None
    for r_ in range(-1, -TRJ_PRE - 1, -1):
        la, st = TRJ.set_index("rel").loc[r_, a], TRJ.set_index("rel").loc[r_, b]
        if pd.notna(la) and pd.notna(st) and la < st * (1 - frac): on_ = r_
        else: break
    return on_
bal_sep = sep_onset("leaver_bal_med", "stayer_bal_med", TRJ_GAP)          # median balance ≥ 5% below stayers from here to the event
pay_sep = sep_onset("leaver_pay_med", "stayer_pay_med", TRJ_GAP) if pay_col else None
# when each kind of change first stands out from comparable stayers' normal variation (size-matched lift ≥ 1.5, §9)
_tg = TIM[TIM.onset_month.notna()].assign(group=lambda d: d.feature.map(sgroup))
_on = lambda gs: (int(_tg[_tg.group.isin(gs)].onset_month.min()) if _tg.group.isin(gs).any() else None)
sig_on = _on(["Incoming payment activity", "Outgoing and external-bank activity", "Counterparty relationships"])
bal_sig_on = _on(["Balance behavior"])
_lead_ref = FA[K0][0].lead if K0 in FA else first_alerts(K0)[1].lead
BR.update(trj=TRJ, trj_n=NCL, bal_sep=bal_sep, pay_sep=pay_sep, sig_on=sig_on, bal_sig_on=bal_sig_on,
          pay_at_m3=float(TRJ.set_index("rel").loc[-3, "leaver_pay_med"]) if pay_col else np.nan,
          pay_at_m1=float(TRJ.set_index("rel").loc[-1, "leaver_pay_med"]) if pay_col else np.nan,
          bal_at_m6=float(TRJ.set_index("rel").loc[-6, "leaver_bal_med"]), stay_at_m6=float(TRJ.set_index("rel").loc[-6, "stayer_bal_med"]), med_lead=float(_lead_ref.median()) if len(_lead_ref) else np.nan,
          pay_col=pay_col, bal_at_m1=float(TRJ.set_index("rel").loc[-1, "leaver_bal_med"]),
          bal_at_m3=float(TRJ.set_index("rel").loc[-3, "leaver_bal_med"]), stay_at_m1=float(TRJ.set_index("rel").loc[-1, "stayer_bal_med"]))
fig, ax = plt.subplots(1, 2 if pay_col else 1, figsize=(13.5 if pay_col else 7.5, 4.4), squeeze=False); ax = ax[0]
ax[0].fill_between(TRJ.rel, TRJ.leaver_bal_p25 * 100, TRJ.leaver_bal_p75 * 100, color="#c00000", alpha=0.10, lw=0)
ax[0].plot(TRJ.rel, TRJ.leaver_bal_med * 100, color="#c00000", lw=2.4, marker="o", ms=4, label="clients whose money later left")
ax[0].plot(TRJ.rel, TRJ.stayer_bal_med * 100, color="#1f4e79", lw=2.0, ls="--", label="comparable clients who stayed")
ax[0].set_ylabel("balance, % of normal balance 12 months before"); ax[0].set_title("Deposit balance")
if pay_col:
    ax[1].plot(TRJ.rel, TRJ.leaver_pay_med * 100, color="#c00000", lw=2.4, marker="o", ms=4, label="clients whose money later left")
    ax[1].plot(TRJ.rel, TRJ.stayer_pay_med * 100, color="#1f4e79", lw=2.0, ls="--", label="comparable clients who stayed")
    ax[1].set_ylabel("incoming payments, % of their starting level"); ax[1].set_title("Money received from payments")
for a_ in ax:
    a_.axvline(0, color="#555", lw=1); a_.text(0.15, a_.get_ylim()[1] * 0.97, "money left", fontsize=8, va="top", color="#555")
    if sig_on is not None:
        a_.axvspan(sig_on - 0.5, -0.5, color="#f4b183", alpha=0.25, lw=0)
    a_.set_xticks(rels_t); a_.set_xlabel("months before the money left (0 = event month)"); a_.grid(alpha=0.25)
ax[0].legend(fontsize=8, loc="lower left")
if sig_on is not None:
    ax[0].text(sig_on - 0.4, ax[0].get_ylim()[0] + 3, f"payment changes stand out\nfrom month {sig_on}", fontsize=8, color="#8a4a00")
fig.tight_layout(); savefig(fig, "m4_13_trajectory"); figdata("m4_13_trajectory", TRJ)

# ── six business signal groups ───────────────────────────────────────
GX = {"Balance behavior": "Unusual balance swings: some clients first build their balance up, then it falls below normal in the last month or two.",
      "Incoming payment activity": "Fewer payments arrive, on fewer days and from fewer payers, while the balance still looks normal.",
      "Outgoing and external-bank activity": "More money is sent to other banks, including transfers to accounts in the client's own name.",
      "Counterparty relationships": "Regular payers stop paying and are not replaced; the client's set of counterparties shrinks.",
      "Related entities and relationship context": "Other entities in the same relationship move balances or close accounts around the same time.",
      "Pricing and client context": "Earnings credit or interest-rate changes, length of relationship and size relative to similar clients."}
imp_g = IMP.assign(group=IMP.feature.map(sgroup)).groupby("group").share.sum()
tim_g = TIM.assign(group=TIM.feature.map(sgroup))
rows = []
for gname in GROUPS:
    tg = tim_g[(tim_g.group == gname)]
    ton = tg[tg.onset_month.notna()].sort_values(["onset_month", "lift_1m"], ascending=[True, False])
    on_ = int(ton.onset_month.min()) if len(ton) else None
    what = "; ".join(PLD(f_, d_) for f_, d_ in zip(ton.feature.head(2), ton.leavers.head(2))) or \
           ("no clear change before the event; used for context: " + " and ".join(PL(f_) for f_ in IMP[IMP.feature.map(sgroup) == gname].feature.head(2)))
    timing = ("early warning (3+ months ahead)" if on_ is not None and on_ <= -3 else
              "close to the event (1–2 months ahead)" if on_ is not None else "no consistent change before the event")
    rows.append({"signal group": gname, "share of the model's evidence": float(imp_g.get(gname, 0.0)), "what changes": what,
                 "first visible": (f"{mo(-on_)} before" if on_ is not None else "—"), "timing": timing, "plain example": GX[gname]})
SIG = pd.DataFrame(rows); BR["signals"] = SIG
# heatmap of §9 with plain names (appendix)
_tf = list(TIM.feature)
fig, ax_ = plt.subplots(figsize=(11, 0.3 * len(_tf) + 1.6))
M_ = np.array([LIFT[f].reindex(rels).values for f in _tf], dtype=float)
im = ax_.imshow(np.clip(M_, 1, 5), aspect="auto", cmap="Oranges", vmin=1, vmax=5)
ax_.set_yticks(range(len(_tf))); ax_.set_yticklabels([PLD(f, d)[:75] for f, d in zip(_tf, TIM.leavers)], fontsize=8)
ax_.set_xticks(range(len(rels))); ax_.set_xticklabels(rels); ax_.set_xlabel("months before the money left")
ax_.set_title("How much more often leavers are outside stayers' normal range (1 = no difference, capped at 5)", fontsize=10)
fig.colorbar(im, ax=ax_, shrink=0.5); fig.tight_layout(); savefig(fig, "m4_13_onset_heatmap")

# ── three anonymised examples, chosen by rule ───────────────────────
is_pay_change = lambda f: isinstance(f, str) and ("__d6" in f or "__d3" in f) and not f.startswith("dep_") and \
                          sgroup(f) in ("Incoming payment activity", "Outgoing and external-bank activity", "Counterparty relationships")
is_change = lambda f: isinstance(f, str) and (("__d6" in f or "__d3" in f) or f in ("dep_d1", "dep_d3", "dep_d6", "dep_drawdown", "dep_bal_vs_norm", "dep_live_d6", "dep_sib_d6"))
fa_main = first_alert[(MAIN, A_MAIN, K_MAIN)].set_index("cust_pwr_id")
Qk = Q[["cust_pwr_id", "tm", rk, "p", "bal", "bal_norm", "ab_t", "r1", "r2", "r3"]].copy(); Qk["on"] = Qk[rk] <= K_MAIN
# monthly balances of the departing relationships, to check when the money actually went
_pool_ids = spark.createDataFrame([(str(x),) for x in fa_main.index], "cust_pwr_id string")
BALD = topd(FEAT.join(F.broadcast(_pool_ids), "cust_pwr_id").select("cust_pwr_id", "t", "bal"))
BALD = {k: g.set_index("t").bal for k, g in BALD.groupby("cust_pwr_id")}
rows = []
for c, g in Qk[Qk.cust_pwr_id.isin(set(fa_main.index) - set(EXCL))].groupby("cust_pwr_id"):
    e = int(fa_main.loc[c, "ab_t"]); ta = int(fa_main.loc[c, "tm"]); g = g.set_index("tm")
    before = g.loc[g.index < ta]; between = g.loc[(g.index >= ta) & (g.index < e)]
    a_ = g.loc[ta]
    b_ = BALD.get(c, pd.Series(dtype=float)).loc[ta:e - 2]
    rows.append({"c": c, "e": e, "ta": ta, "lead": e - ta, "norm": float(a_.bal_norm), "pct": float(a_.bal / a_.bal_norm),
                 "min_before": float(b_.min() / a_.bal_norm) if len(b_) else np.nan,
                 "off_before": int((~before.on).sum()), "on_before": int(before.on.sum()), "on_share": float(between.on.mean()),
                 "on_n": int(between.on.sum()), "r1": a_.r1, "chg": any(is_change(x) for x in (a_.r1, a_.r2, a_.r3)), "pay1": is_pay_change(a_.r1)})
CAND = pd.DataFrame(rows)
def pick_ex(df, sort, n=30):
    """First candidate (by sort) whose departure passed the §7 check."""
    d = df.sort_values(sort, ascending=False).head(n)
    if not len(d): return None
    v = audit(list(d.c)).set_index("cust_pwr_id").verdict
    ok = [c for c in d.c if v.get(c) in OKV]
    return ok[0] if ok else None
# A: first appeared 3+ months ahead (scored but off the list before), balance still ≥ 90% of normal, stayed on the list
cA = pick_ex(CAND[(CAND.lead >= 3) & (CAND.pct >= 0.9) & (CAND.min_before >= 0.5) & (CAND.off_before >= 2) & (CAND.on_before == 0)
                  & (CAND.on_share >= 0.6) & CAND.chg],
             ["lead", "on_share", "norm"]) if len(CAND) else None
# B: the strongest evidence at the first alert is a payment change, balance still ≥ 90% of normal
cB = pick_ex(CAND[(CAND.lead >= 2) & (CAND.pct >= 0.9) & (CAND.min_before >= 0.15) & CAND.pay1 & (CAND.on_share >= 0.5) & (CAND.c != cA)],
             ["on_share", "lead", "norm"]) if len(CAND) else None
# C: on the list for 3+ months, money still here at the end of the data
never = set(Q.loc[Q.ab_t.isna(), "cust_pwr_id"])
gC = Qk[Qk.cust_pwr_id.isin(never)].groupby("cust_pwr_id").agg(n_on=("on", "sum"), pmax=("p", "max")).reset_index()
gC = gC[gC.n_on >= 3].sort_values(["pmax", "cust_pwr_id"], ascending=[False, True]).head(20)
firstC = Qk[Qk.cust_pwr_id.isin(set(gC.cust_pwr_id)) & Qk.on].sort_values("tm").drop_duplicates("cust_pwr_id").set_index("cust_pwr_id")
hC = topd(FEAT.filter(F.col("cust_pwr_id").isin(list(gC.cust_pwr_id) or ["__none__"])).select("cust_pwr_id", "t", "bal")) if len(gC) else \
     pd.DataFrame(columns=["cust_pwr_id", "t", "bal"])
okC = set()
for c_, g_ in hC.groupby("cust_pwr_id"):
    ta_, ref_ = int(firstC.loc[c_, "tm"]), float(firstC.loc[c_, "bal_norm"])
    after = g_[g_.t >= ta_].set_index("t").bal
    if len(after) and after.min() >= 0.5 * ref_ and after.get(END_T, 0.0) >= 0.8 * ref_: okC.add(c_)
cC = next((c for c in gC.cust_pwr_id if c in okC), None)
CHOSEN = [(t_, ti_, c_) for t_, ti_, c_ in (("A", "Early warning while balances still looked healthy", cA),
                                             ("B", "Payment behavior changed before the final balance decline", cB),
                                             ("C", "Surfaced, but the money did not leave", cC)) if c_ is not None]
ids_ = [c_ for _, _, c_ in CHOSEN]
TF = [f for f in THR if f in FEAT.columns]
PAN_ = topd(FEAT.filter(F.col("cust_pwr_id").isin(ids_ or ["__none__"])).select("cust_pwr_id", "t", "bal", "bal_norm", "first_t",
                                                                                 *[F.col(f).cast("double").alias(f) for f in TF]))
EXB = []
for tag, title, c in CHOSEN:
    h = PAN_[PAN_.cust_pwr_id == c].sort_values("t").set_index("t")
    sc_ = Qk[Qk.cust_pwr_id == c].sort_values("tm"); on_l = sc_[sc_.on]
    leaver = c in fa_main.index
    if leaver:
        e = int(fa_main.loc[c, "ab_t"]); ta = int(fa_main.loc[c, "tm"]); on_l = on_l[on_l["tm"] < e]
    else:
        e = None; ta = int(on_l["tm"].min())
    at = sc_[sc_["tm"] == ta].iloc[0]; ref = float(at.bal_norm); anchor = e if e is not None else ta
    reasons = [PL(x) for x in at[["r1", "r2", "r3"]] if isinstance(x, str)]
    lo_, hi_ = (anchor - 12, anchor + 2) if e is not None else (anchor - 9, min(anchor + 9, END_T))
    fired = []
    for f in TF:
        v_ = h[f].loc[ta - 3:(e - 1 if e is not None else ta + 6)]          # around the alert only
        m_ = [t_ for t_, x in v_.items() if beyond(x, f)]
        if m_: fired.append((min(m_), f))
    fired.sort()
    def when_(t_):
        if e is not None: return f"{mo(e - t_)} before the money left"
        return "at the first alert" if t_ == ta else (f"{mo(ta - t_)} before the first alert" if t_ < ta else f"{mo(t_ - ta)} after the first alert")
    ev_txt = "; ".join(f"{PLD(f, THR[f][1])} ({when_(t_)})" for t_, f in fired[:3])
    norm_band = next(b_[2] for b_ in BANDS if b_[0] <= ref < b_[1])
    pct_alert = float(h.bal.get(ta, np.nan) / ref) if ref else np.nan
    ver = AUD_CACHE.get(c, {}).get("verdict")
    if leaver:
        et = dp.loc[dp.cust_pwr_id == c, "exit_type"].iloc[0] if (dp.cust_pwr_id == c).any() else ""
        n_on = len(on_l); span = int(((sc_["tm"] >= ta) & (sc_["tm"] < e)).sum())   # months it was still eligible and scored
        situation = f"A relationship with a normal balance of {norm_band}, observed for {int(e - h.first_t.iloc[0])} months before the event."
        first = (f"First surfaced {mo(e - ta)} before the money left, with the balance at {pct_alert:.0%} of normal; on the list in "
                 f"{n_on} of the {span} months it was scored before the money left.")
        outcome = ("The balance fell below 5% of normal and stayed there while the accounts remained open (balance exit)." if et == "balance exit"
                   else "All deposit accounts became inactive or closed (full exit).")
        unc = {"moved to another bank": "Payments to accounts in the client's own name at other banks rose around the exit, which points to a move, "
                                        "but the client's reasons are not visible in the data."
               }.get(ver, "The data show the money leaving but not why: a move to another bank, a business change or a planned use of cash are all possible.")
    else:
        situation = f"A relationship with a normal balance of {norm_band} that ranked high on the list."
        first = f"On the list for {mo(len(on_l))}; the balance was at {pct_alert:.0%} of normal when it first appeared."
        outcome = (f"The money did not leave by {END_M}: the balance stayed at or above half of its normal level at the first alert, and was "
                   f"{float(h.bal.get(END_T, np.nan) / ref):.0%} of it at the end of the data.")
        unc = ("The changes that raised it also occur in clients who reorganise, move money temporarily or shift activity between their own "
               "entities. Only a business review can tell these apart, and the money may still leave after the data end.")
    evidence = ("Strongest evidence at the first alert: " + "; ".join(reasons[:3]) + "."
                + (f" Changes outside the usual range of comparable clients who stayed: {ev_txt}." if ev_txt else ""))
    hb = h.loc[lo_:hi_]
    fig, a_ = plt.subplots(figsize=(7.2, 2.6))
    a_.plot(hb.index - anchor, hb.bal / ref * 100, color="#1f4e79", marker="o", ms=3); a_.axhline(100, color="grey", ls="--", lw=0.8)
    for x_ in on_l["tm"]:
        if lo_ <= x_ <= hi_: a_.axvspan(x_ - anchor - 0.5, x_ - anchor + 0.5, color="#f4b183", alpha=0.35, lw=0)
    if e is not None: a_.axvline(0, color="#c00000", ls="--", lw=1.2)
    a_.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    a_.set_xlabel("months before the money left" if e is not None else "months from the first alert")
    a_.set_ylabel("% of normal"); a_.set_title(f"Client {tag}: balance (orange = on the list)", fontsize=9)
    fig.tight_layout(); savefig(fig, f"m4_13_example_{tag}")
    EXB.append(dict(tag=tag, title=title, kind=title, situation=situation, first=first, evidence=evidence,
                    lead=(mo(e - ta) if e is not None else "not applicable (no departure)"), outcome=outcome, uncertain=unc,
                    chart=f"m4_13_example_{tag}"))
pd.DataFrame([{"tag": t_, "cust_pwr_id": c_} for t_, _, c_ in CHOSEN]).to_csv(OUT / "m4_13_examples_FULL_IDS.csv", index=False)
BR["examples"] = EXB
pd.DataFrame([{k_: v_ for k_, v_ in x.items() if k_ != "chart"} for x in EXB]).to_csv(OUT / "m4_13_examples_text.csv", index=False)
for nm_, df_ in (("m4_13_curve", CURVE), ("m4_13_lead", LEADT), ("m4_13_size", SIZE), ("m4_13_signals", SIG)): figdata(nm_, df_)

_gap = []
for c_, r_ in fa_main.iterrows():
    b_ = BALD.get(c_)
    if b_ is None: continue
    e_ = int(r_.ab_t); ref_ = float(r_.bal_norm)
    low = [t_ for t_ in range(e_ - 6, e_) if b_.get(t_, np.nan) < 0.15 * ref_]
    first_low = None
    for t_ in range(e_ - 6, e_):                    # first month from which the balance stays below 15% until the exit
        if all(b_.get(u, np.inf) < 0.15 * ref_ for u in range(t_, e_)): first_low = t_; break
    _gap.append({"c": c_, "e": e_, "ta": int(r_["tm"]), "gone_early": first_low is not None and e_ - first_low >= 2,
                 "months_early": (e_ - first_low) if first_low is not None else 0, "capped": float(r_.capped)})
GAP = pd.DataFrame(_gap)
BR.update(gone_early=float(GAP.gone_early.mean()) if len(GAP) else np.nan,
          gone_early_usd=float(GAP.loc[GAP.gone_early, "capped"].sum() / max(GAP.capped.sum(), 1.0)) if len(GAP) else np.nan,
          gone_early_med=float(GAP.loc[GAP.gone_early, "months_early"].median()) if GAP.gone_early.any() else np.nan,
          lead_adj_med=float((GAP.e - GAP.months_early.where(GAP.gone_early, 0) - GAP.ta).median()) if len(GAP) else np.nan)
B.kv([("base rate (test months, client-months)", f"{BASE_N:,} / {BASE_D:,} = {base_rate:.2%}"),
      ("eligible to be scored (share of live client-months · of balance)", f"{BR['eligible_share']:.1%} · {BR['eligible_bal_share']:.1%}"),
      ("departing relationships with money still here (test months)", f"{len(pool):,}"),
      ("trajectory: leavers / stayers traced", f"{NCL.get('leaver', 0):,} / {NCL.get('stayer', 0):,}"),
      ("balances separate from stayers / payment signals visible / median first alert",
       f"{bal_sep} / {sig_on} / {BR['med_lead']:.0f} months before (list of {K0:,})"),
      ("surfaced leavers whose balance was below 15% of normal 2+ months before the recorded exit month",
       f"{BR['gone_early']:.1%} (of balance {BR['gone_early_usd']:.1%}; median {BR['gone_early_med']} months early; median lead counted from "
       f"the real drain {BR['lead_adj_med']})"),
      ("examples", ", ".join(f"{x['tag']} = {x['kind']}" for x in EXB) + (" (missing: no client met the rule)" if len(EXB) < 3 else ""))], "13a · Briefing numbers")
B.table(CURVE, "13b · Review-list sizes (unique relationships; hit rate averaged over test months)",
        pct=("hit_rate", "recall", "surfaced_3plus", "balance_share", "top5_share", "two_months_running"), money=("balance_surfaced",))
B.table(LEADT, "13c · When departing relationships were first surfaced", pct=("share_of_departing", "share_of_balance"))
B.table(SIZE, f"13d · By size (list of {K0:,})", pct=("surfaced_share", "surfaced_3plus", "share_of_departing_balance"), money=("balance",))
B.table(SIG, "13e · Signal groups", pct=("share of the model's evidence",))
B.show(t0)

## §14 The briefing page

A business-facing page for Treasury Management: summary, balance trajectory, definition, signal groups, lead time, list sizes, examples,
limitations, the questions for Treasury Management, and a collapsed technical appendix. No ids, names or institutions; examples use size bands
and balances as a share of normal. The technical report (§12) is unchanged.

In [ ]:
# =====================================================================
# 14 · TREASURY MANAGEMENT BRIEFING (HTML)                          [OUTPUT BLOCK 14]
# =====================================================================
t0 = time.time(); B = Block("§14 · Treasury Management briefing")
esc = _html.escape
def pc(v, d=0): return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{d}%}"
def usd_(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "—"
    return f"${v / 1e9:,.1f}bn" if abs(v) >= 1e9 else f"${v / 1e6:,.0f}m" if abs(v) >= 1e6 else f"${v:,.0f}"
def img_(name, w="100%"):
    return (f"<img src='data:image/png;base64,{base64.b64encode(open(CHARTS[name], 'rb').read()).decode()}' style='max-width:{w}'>"
            if name in CHARTS else "")
def ht(df, fmt=None, cls="t"):
    fmt = fmt or {}
    h = [f"<table class='{cls}'><tr>" + "".join(f"<th>{esc(str(c))}</th>" for c in df.columns) + "</tr>"]
    for _, r_ in df.iterrows():
        cells = []
        for c in df.columns:
            v = r_[c]; f_ = fmt.get(c)
            s_ = (f_(v) if f_ else ("—" if v is None or (isinstance(v, float) and np.isnan(v)) else
                                    f"{v:,}" if isinstance(v, (int, np.integer)) else f"{v:,.2f}" if isinstance(v, float) else str(v)))
            cells.append(f"<td>{esc(s_)}</td>")
        h.append("<tr>" + "".join(cells) + "</tr>")
    return "".join(h) + "</table>"
CV = BR["curve"].set_index("list_size")
k_lo = 250 if 250 in CV.index else CV.index[len(CV) // 2]; k_hi = CV.index.max(); K0 = BR["lead_ref"]
L0 = BR["lead"][BR["lead"].list_size == K0].set_index("first surfaced")
early3 = float(L0.loc[["6 months (the longest the model looks ahead)", "3–5 months"], "share_of_departing"].sum())
early3_usd = float(L0.loc[["6 months (the longest the model looks ahead)", "3–5 months"], "share_of_balance"].sum())
bs, ps, so, bso = BR.get("bal_sep"), BR.get("pay_sep"), BR.get("sig_on"), BR.get("bal_sig_on")
pay_first = so is not None and (bso is None or so < bso)
TRJ_TITLE = ("Payment changes stand out before balance changes" if pay_first else "Balances and payments both change before the money leaves")
_bal_txt = (f"Leavers' median balance starts to drift below that of comparable stayers about {mo(-bs)} before the money leaves "
            f"({pc(BR['bal_at_m6'])} of normal six months before against {pc(BR['stay_at_m6'])} for stayers, {pc(BR['bal_at_m3'])} three months before, "
            f"{pc(BR['bal_at_m1'])} one month before)" if bs else
            f"Leavers' median balance stays close to stayers' until the final month ({pc(BR['bal_at_m1'])} of normal one month before)")
_pay_txt = (f"; their incoming payments hold steady until about {mo(-ps)} before and then fall faster ({pc(BR['pay_at_m3'])} of their earlier level "
            f"three months before, {pc(BR['pay_at_m1'])} one month before)" if ps else "")
_std_txt = ((f" Measured against how much clients normally vary from month to month, payment changes stand out first: about {mo(-so)} before"
             + (f", against {mo(-bso)} for balance changes." if bso else ", while balance changes do not stand out consistently."))
            if so is not None else "")
META = (f"Data through {BR['data_through']} · model: v13 model 4 on customer metrics v3 · target: money leaves within 6 months "
        f"(full exit or balance exit, v13 definition) · back-test months {BR['test_first']} – {BR['test_last']}")

S = [f"""<header><h1>Deposit Attrition Early Warning: Initial Findings</h1>
<p class='sub'>What internal deposit and payment behavior may tell us before client money leaves.</p><p class='meta'>{esc(META)}</p></header>"""]
# ── 1 · executive summary ─────────────────────────────────────────────
S.append(f"""<section><h2>Summary</h2><ul class='summary'>
<li><b>A client can keep its accounts open while moving most of its money out.</b> About {R['exits_per_year']:,.0f} Treasury Management
relationships a year moved their money out, holding {usd_(R['at_risk_year'])} of normal balance; {R['bal_exit_share']:.0%} of those dollars
left through <i>balance exits</i>, where the accounts stayed open. A measure based only on account closures does not see them.</li>
<li><b>Departing clients' behavior changes months before the money leaves.</b> {_bal_txt}{_pay_txt}.{_std_txt} The changes show up in
payments received, payments to other banks, counterparties and related entities, not only in the balance.</li>
<li><b>The model turns these patterns into a ranked monthly early-warning review list.</b> In a list of {k_lo:,} relationships a month,
{pc(CV.loc[k_lo, 'hit_rate'])} of those surfaced were relationships whose money left within six months, {CV.loc[k_lo, 'lift']:.1f} times the
{pc(BR['base_rate'], 1)} rate among all eligible relationships. In a list of {k_hi:,}, {pc(CV.loc[k_hi, 'recall'])} of departing relationships were surfaced, holding
{pc(CV.loc[k_hi, 'balance_share'])} of the departing balance; {pc(CV.loc[k_hi, 'surfaced_3plus'])} of those surfaced were surfaced at least three months ahead.</li>
<li><b>Treasury Management's input is needed</b> to decide whether this definition matches how the business thinks about a lost relationship,
who should review the alerts, how many can realistically be reviewed, and how a pilot should be designed and measured.</li></ul></section>""")
# ── 2 · balance trajectory ────────────────────────────────────────────
n_ = BR["trj_n"]
S.append(f"""<section><h2>{TRJ_TITLE}</h2>
<p>Median balance and incoming payments of clients whose money later left, against comparable clients who stayed, over the 12 months before the
event. The shaded band starts where payment changes first stand out from comparable stayers' normal month-to-month variation.</p>{img_('m4_13_trajectory')}
<p class='read'>Twelve months out the two groups look the same. {_bal_txt}{_pay_txt}.{_std_txt} The typical first alert comes
{mo(BR['med_lead'])} before the money leaves (list of {K0:,}).</p>
<p class='foot'>Population: Treasury Management deposit relationships whose money was still here 12 months before the event (balance ≥ half of
normal, normal ≥ $10k): {n_.get('leaver', 0):,} relationships whose money left between 2024 and {BR['data_through']}, and {n_.get('stayer', 0):,} that
stayed, each aligned on a randomly drawn month and weighted to the leavers' size mix. Balance = monthly average, as % of the normal balance 12 months
before; incoming payments = dollars received, against the first three months shown. Lines are medians; the red band is the leavers' middle half.
Relationships later found to have moved within PNC or come back are excluded.</p></section>""")
# ── 3 · definition
FX_NOTE = (f"""<p class='note'><b>A definition question for Treasury Management:</b> {R['p_return']:.0%} of relationships whose accounts were all inactive or
closed for {CONFIRM_M} months in a row later became active again. Most of the departing dollars ({R['bal_exit_share']:.0%}) leave through balance exits,
which this does not affect, but it suggests that a full exit may need a longer confirmation period, or that some status codes mark temporary
inactivity rather than a closed relationship.</p>""" if R['p_return'] > 0.05 else "")
# ────────────────────────────────────────────────────
S.append(f"""<section><h2>What counts as attrition</h2><div class='defs'>
<div><b>Full exit</b><br>All of the relationship's deposit accounts become closed or inactive, confirmed by {CONFIRM_M} months in a row.</div>
<div><b>Balance exit</b><br>The client moves nearly all of its normal balance out and keeps it out for {BAL_EXIT_HOLD} months, while the accounts may stay open.</div>
<div><b>Money leaves</b><br>The earlier of the two. The model asks whether this happens within the next {H} months.</div></div>
{FX_NOTE}
<details><summary>Exact thresholds</summary><ul>
<li>Normal balance: median monthly average balance over up to {NORM_WIN} prior months with live accounts (at least {NORM_MIN_OBS} needed).</li>
<li>Balance exit: balance below {BAL_EXIT_FRAC:.0%} of normal for {BAL_EXIT_HOLD} consecutive months while at least one account is live.</li>
<li>Full exit: every account non-live; confirmed after {CONFIRM_M} observed non-live months.</li>
<li>Eligible relationships ("money still here"): live, at least {MIN_HIST} months observed, normal balance ≥ ${MSH_FLOOR:,.0f}, current balance ≥ {MSH_FRAC:.0%} of normal.</li>
<li>The largest departures were checked one by one; those whose balance came back or moved to another PNC customer in the same relationship are not counted as losses.</li></ul></details></section>""")
# ── 4 · signal groups ─────────────────────────────────────────────────
sg = BR["signals"].copy()
S.append(f"""<section><h2>What changes before the money leaves</h2>
<p>The model's evidence grouped into six business signal groups. "First visible" is the earliest month from which departing clients cross the
normal range of comparable stayers of the same size at least 1.5 times as often, and keep doing so until the event.</p>
{ht(sg, {"share of the model's evidence": lambda v: pc(v)})}
<p class='note'>These are pieces of evidence that contributed to an alert, not confirmed reasons a client left. The same pattern can come from a move
to another bank, a business that is shrinking or reorganising, or a planned use of cash.</p></section>""")
# ── 5 · lead time ─────────────────────────────────────────────────────
LT = BR["lead"].pivot(index="first surfaced", columns="list_size", values="relationships").reindex(L0.index)
LS_ = BR["lead"].pivot(index="first surfaced", columns="list_size", values="share_of_departing").reindex(L0.index)
lt = pd.DataFrame({"first surfaced": LT.index})
for K in LT.columns:
    lt[f"list of {K:,}: relationships"] = LT[K].values; lt[f"list of {K:,}: share"] = LS_[K].values
GONE_NOTE = (f"<li>The exit month can lag the real drain: for {pc(BR['gone_early'])} of the departing relationships surfaced "
             f"({pc(BR['gone_early_usd'])} of their balance) the balance was already below 15% of normal two or more months before the recorded exit "
             f"month (median {mo(BR['gone_early_med'])}), because a balance exit needs three months below 5% of a normal balance that itself adjusts "
             f"downwards. Counted from the real drain, the median lead is {mo(BR['lead_adj_med'])}.</li>"
             if pd.notna(BR.get("gone_early")) and BR["gone_early"] >= 0.05 else "")
sz = BR["size"][["band", "departing", "surfaced_share", "median_lead", "surfaced_3plus", "share_of_departing_balance"]].rename(columns={
    "band": "normal balance", "departing": "departing relationships", "surfaced_share": "surfaced", "median_lead": "median lead (months)",
    "surfaced_3plus": "surfaced 3+ months ahead", "share_of_departing_balance": "share of departing balance"})
S.append(f"""<section><h2>How much advance warning</h2>
<p>Each departing relationship is counted once, at the first month it appeared on the list within the six months before its money left.
Denominator: the {BR['n_departing']:,} relationships whose money left within six months of a back-test month while it was still here.</p>
{ht(lt, {c: (lambda v: pc(v)) for c in lt.columns if c.endswith('share')})}
<ul><li>Median first-alert lead: {mo(BR['med_lead'])} (list of {K0:,}); {pc(early3)} of departing relationships, holding {pc(early3_usd)} of
the departing balance, were first surfaced at least three months ahead.</li>
<li>Persistence: {pc(CV.loc[K0, 'two_months_running'] if K0 in CV.index else np.nan)} of the departing relationships surfaced were on the list in at least
two consecutive months before their money left; the rest appeared in single months.</li>
{GONE_NOTE}<li>No relationship is surfaced in the event month itself: once the money has gone the relationship is no longer eligible. Six months is the
longest lead counted because the model looks six months ahead.</li></ul>
<h3>By size of relationship (list of {K0:,})</h3><p>The list ranks by probability weighted towards
larger balances, so larger relationships are surfaced more often and earlier; most small relationships that leave are not surfaced.</p>{ht(sz, {"surfaced": pc, "surfaced 3+ months ahead": pc, "share of departing balance": pc,
                                                     "median lead (months)": lambda v: "—" if pd.isna(v) else f"{v:.0f}"})}
<p class='note'>Dollar results are concentrated: the largest departing relationship is {pc(BR['top1_share'])} of the departing balance and the five
largest are {pc(BR['top5_pool_share'])}. A few very large relationships can move the dollar figures by themselves.</p></section>""")
# ── 6 · review-list sizes ─────────────────────────────────────────────
cv = BR["curve"][["list_size", "relationships_surfaced", "hit_rate", "hit_rate_range", "lift", "departing_surfaced", "recall", "median_lead_months",
                  "surfaced_3plus", "balance_surfaced", "balance_share", "top5_share"]].rename(columns={
    "list_size": "monthly list size", "relationships_surfaced": "unique relationships surfaced", "hit_rate": "hit rate",
    "hit_rate_range": "hit rate, range across months", "lift": "lift over base rate", "departing_surfaced": "departing relationships surfaced",
    "recall": "share of departing relationships surfaced", "median_lead_months": "median lead (months)", "surfaced_3plus": "surfaced 3+ months ahead",
    "balance_surfaced": "departing balance surfaced", "balance_share": "share of departing balance", "top5_share": "…from the 5 largest"})
fig, a1 = plt.subplots(figsize=(8, 3.6)); xs = list(range(len(cv)))
a1.plot(xs, cv["hit rate"] * 100, marker="o", color="#1f4e79", label="hit rate")
a1.plot(xs, cv["share of departing relationships surfaced"] * 100, marker="s", color="#c55a11", label="share of departing relationships surfaced")
a1.axhline(BR["base_rate"] * 100, color="grey", ls=":", lw=1); a1.text(len(cv) - 1, BR["base_rate"] * 100 + 1, "base rate", ha="right", fontsize=8, color="grey")
a1.set_xticks(xs); a1.set_xticklabels([f"{k:,}" for k in cv["monthly list size"]]); a1.set_xlabel("relationships on the monthly review list")
a1.set_ylabel("%"); a1.legend(fontsize=8); a1.grid(alpha=0.25); fig.tight_layout(); savefig(fig, "m4_14_curve")
S.append(f"""<section><h2>Performance at different review-list sizes</h2>
<p>Relationships ranked each month by the model's probability that the money leaves within six months, weighted towards larger balances. The table
shows what lists of different sizes would have contained in the back-test. It describes list performance; it is not a recommendation to review any
particular number of relationships.</p>{img_('m4_14_curve', '760px')}
{ht(cv, {"hit rate": pc, "lift over base rate": lambda v: f"{v:.1f}×", "share of departing relationships surfaced": pc,
         "median lead (months)": lambda v: "—" if pd.isna(v) else f"{v:.0f}", "surfaced 3+ months ahead": pc, "departing balance surfaced": usd_,
         "share of departing balance": pc, "…from the 5 largest": pc})}
<details><summary>How these are calculated</summary><ul>
<li><b>Base rate</b> = {BR['base_n']:,} eligible client-months followed by the money leaving within six months ÷ {BR['base_d']:,} eligible client-months
= {pc(BR['base_rate'], 2)}; grain: client-month, pooled across the {BR['n_test']} back-test months ({BR['test_first']} – {BR['test_last']}).</li>
<li><b>Hit rate</b>: share of the list whose money left within six months, computed per month and averaged across the back-test months; the range shows
the lowest and highest month. <b>Lift</b> = hit rate ÷ base rate.</li>
<li><b>Unique relationships surfaced</b>: distinct relationships appearing on the list in any back-test month. <b>Share of departing relationships surfaced</b>:
distinct departing relationships surfaced at least once in the six months before their money left ÷ {BR['n_departing']:,}.</li>
<li><b>Departing balance surfaced</b>: the lesser of current and normal balance at the first alert, summed over the departing relationships surfaced, over the
whole back-test period (not annualised).</li>
<li><b>Coverage</b>: {pc(BR['eligible_share'])} of live Treasury Management deposit client-months ({pc(BR['eligible_bal_share'])} of the balance) were eligible to be
scored; {pc(BR['pay_seen'])} of live client-months have payment history.</li>
<li>Each month's model was fitted only on outcomes known at that month, so these are honest back-tests.</li></ul></details></section>""")
# ── 7 · examples ──────────────────────────────────────────────────────
ex_html = "".join(f"""<div class='ex'><h3>Client {x['tag']}: {esc(x['title'])}</h3>{img_(x['chart'], '640px')}<dl>
<dt>Situation</dt><dd>{esc(x['situation'])}</dd><dt>First warning</dt><dd>{esc(x['first'])}</dd><dt>Evidence</dt><dd>{esc(x['evidence'])}</dd>
<dt>Lead time</dt><dd>{esc(x['lead'])}</dd><dt>Outcome</dt><dd>{esc(x['outcome'])}</dd><dt>What remains uncertain</dt><dd>{esc(x['uncertain'])}</dd></dl></div>"""
                  for x in BR["examples"])
S.append(f"""<section><h2>Three anonymised examples</h2><p>Chosen by rule from the back-test months; amounts are shown as a size band and balances as a
percentage of normal. Months on the ranked list are shaded.</p>{ex_html}</section>""")
# ── 8 · limitations ───────────────────────────────────────────────────
S.append("""<section><h2>Limitations</h2><ul>
<li>The results are historical back-tests, not proof of future retention impact.</li>
<li>The appropriate review capacity and routing process have not been established.</li>
<li>Not every relationship has complete payment visibility; payments outside PNC are seen only from PNC's side.</li>
<li>A small number of very large relationships can materially affect dollar results.</li>
<li>The recorded exit month of a balance exit can come after most of the money has already gone, which makes some lead times look longer than they are.</li>
<li>Behavioral evidence does not by itself show whether a client is moving banks, shrinking, reorganising or making an internal transfer.</li>
<li>Business review is needed to create ground truth and to measure outcomes.</li></ul></section>""")
# ── 9 · input needed ──────────────────────────────────────────────────
S.append("""<section class='ask'><h2>Where we need Treasury Management's input</h2><ol>
<li>Validating whether the attrition definition reflects how the business thinks about a lost relationship.</li>
<li>Understanding any current review or outreach process.</li>
<li>Determining who should review the alerts first.</li>
<li>Selecting a realistic monthly review volume.</li>
<li>Reviewing a small sample of cases and recording business conclusions.</li>
<li>Defining a controlled pilot and the outcomes that should be measured.</li></ol>
<p><b>Proposed next step:</b> a working session to walk through a sample of recent alerts together and agree on the definition, the reviewers and the
pilot design.</p></section>""")
# ── 10 · technical appendix ───────────────────────────────────────────
fo = R["folds"][["test_month", "test_rows", "test_leaving", "auc_simple", "auc_tree"]].rename(columns={
    "test_month": "back-test month", "test_rows": "eligible relationships", "test_leaving": "money left within 6 months",
    "auc_simple": "AUC, simple model", "auc_tree": "AUC, tree model"})
CALT = R['cal'][['bin', 'mean_predicted', 'actual', 'rows']].copy()
CALT["bin"] = (CALT.bin.astype(int) + 1).astype(str); CALT["rows"] = CALT.rows.astype(int)
CALT = CALT.rename(columns={'bin': 'decile (10 = highest)', 'mean_predicted': 'mean predicted'})
top = (IMP.head(20).assign(evidence=lambda d: d.feature.map(PL), group=lambda d: d.feature.map(sgroup))[["evidence", "group", "share", "alone_auc"]]
         .rename(columns={"share": "share of the model's evidence", "alone_auc": "AUC on its own"}))
tim_ = (TIM.assign(signal=TIM.feature.map(PL), group=TIM.feature.map(sgroup))[["signal", "group", "leavers", "onset_month", "lift_6m", "lift_3m", "lift_1m"]]
          .rename(columns={"leavers": "leavers' direction", "onset_month": "first stands out (month)", "lift_6m": "lift 6 months before",
                           "lift_3m": "lift 3 months before", "lift_1m": "lift 1 month before"}))
S.append(f"""<section><h2>Technical appendix</h2>
<details><summary>Method in brief</summary><ul>
<li>Monthly panel of Treasury Management deposit relationships from daily balances; payment behavior from the PKG customer metrics (v3), joined on the power id.</li>
<li>Rows: eligible relationships ("money still here"). Features look backwards only: balance levels and changes, payment levels compared with
similar-size clients, changes against the client's own months −6 to −4 and the previous three months.</li>
<li>Gradient-boosted trees (primary) and a ridge logistic model, refitted for each back-test month on outcomes known at that month; 20% of clients
held out for tuning and calibration. Ranking = probability × normal balance^{A_MAIN}.</li>
<li>Pooled AUC: tree {R['auc_gbm']:.3f}, simple model {R['auc_lin']:.3f}.</li></ul></details>
<details><summary>Discrimination by back-test month</summary>{ht(fo, {"AUC, simple model": lambda v: f"{v:.3f}", "AUC, tree model": lambda v: f"{v:.3f}"})}</details>
<details><summary>Strongest individual pieces of evidence</summary>{ht(top, {"share of the model's evidence": pc, "AUC on its own": lambda v: f"{v:.3f}"})}</details>
<details><summary>When each change stands out (how much more often leavers are outside comparable stayers' normal range)</summary>{img_('m4_13_onset_heatmap')}
{ht(tim_, {"first stands out (month)": lambda v: "—" if pd.isna(v) else f"{v:.0f}", "lift 6 months before": lambda v: "—" if pd.isna(v) else f"{v:.2f}",
           "lift 3 months before": lambda v: "—" if pd.isna(v) else f"{v:.2f}", "lift 1 month before": lambda v: "—" if pd.isna(v) else f"{v:.2f}"})}</details>
<details><summary>Calibration (predicted vs actual, by decile)</summary>{ht(CALT, {"mean predicted": pc, "actual": pc})}</details>
</section>""")
CSS_B = """body{font-family:Arial,Helvetica,sans-serif;max-width:1060px;margin:28px auto;padding:0 18px;color:#1d2733;line-height:1.5}
header{border-bottom:3px solid #1f4e79;margin-bottom:8px}h1{font-size:27px;margin:0 0 4px}.sub{font-size:16px;color:#44515f;margin:0 0 4px}
.meta{font-size:12px;color:#6b7785;margin:4px 0 10px}h2{color:#1f4e79;font-size:20px;margin:30px 0 8px}h3{font-size:15px;margin:18px 0 6px}
ul.summary li{margin:8px 0}.read{background:#f3f6f9;padding:10px 14px;border-left:4px solid #1f4e79}.foot,.note{font-size:12px;color:#5b6773}
.defs{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}.defs div{background:#f3f6f9;padding:10px 12px;border-radius:4px;font-size:14px}
table.t{border-collapse:collapse;font-size:12.5px;margin:8px 0;display:block;overflow-x:auto}table.t th{background:#1f4e79;color:#fff;padding:5px 8px;text-align:left}
table.t td{padding:4px 8px;border-bottom:1px solid #e1e6eb;vertical-align:top}details{margin:8px 0;font-size:13px}summary{cursor:pointer;color:#1f4e79;font-weight:bold}
.ex{border:1px solid #dde3ea;border-radius:6px;padding:4px 16px 10px;margin:14px 0}dl{display:grid;grid-template-columns:170px 1fr;gap:4px 12px;font-size:13.5px}
dt{font-weight:bold;color:#44515f}dd{margin:0}.ask{background:#fff7ec;border:1px solid #f0c48a;border-radius:6px;padding:4px 18px 10px}
@media(max-width:700px){.defs{grid-template-columns:1fr}dl{grid-template-columns:1fr}}"""
BRIEF = OUT / "PKG_Deposit_Attrition_Early_Warning_Initial_Findings.html"
BRIEF.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>Deposit Attrition Early Warning: Initial Findings</title><style>{CSS_B}</style></head>"
                 f"<body>{''.join(S)}<p class='foot'>Internal — PNC Treasury Management · DSI · {esc(META)}</p></body></html>", encoding="utf-8")
import pickle
with open(OUT / "m4_13_briefing.pkl", "wb") as fh:
    pickle.dump({"BR": BR, "charts": {k: str(v) for k, v in CHARTS.items() if k.startswith(("m4_13", "m4_14", "m4_9a"))}}, fh)
txt = re.sub(r"<img[^>]*>", "", "".join(S))
bad = [w for w in (" RM", "RMs", " call", "calls", "save rate", "ROI") if w in txt]
B.kv([("briefing", str(BRIEF.resolve())), ("size", f"{BRIEF.stat().st_size / 1e6:,.1f} MB"),
      ("wording check (RM / call / save rate / ROI)", "clean" if not bad else f"found: {bad}")], "14a · Written")
B.show(t0)

## §11–12 Prediction scorecard and the HTML report

Same sections as before, plus the model comparison and the check of the largest departures. Section 1 ("What we
found") stays a placeholder with the headline numbers filled in; the findings are written after the results are read.

In [ ]:
# =====================================================================
# 12 · PREDICTIONS · HTML REPORT                                    [OUTPUT BLOCK 12]
# =====================================================================
t0 = time.time(); B = Block("§12 · Predictions and report")
_v = lambda mn, a, K: float(VAL[(VAL.model == mn) & (VAL.alpha == a) & (VAL.calls == K)].dollars_reached_year.iloc[0])
PRED = [
    ("P1", "Calibrated top decile predicted ÷ actual in 0.8–1.25", 0.55, lambda: 0.8 <= R["top_dec_ratio"] <= 1.25, f"{R['top_dec_ratio']:.2f}"),
    ("P2", "Tree AUC ≥ simple + 0.01", 0.80, lambda: R["auc_gbm"] - R["auc_lin"] >= 0.01, f"{R['auc_gbm'] - R['auc_lin']:+.3f}"),
    ("P3", "Fewer than 12 checked departures are 'moved to another bank'", 0.60,
     lambda: int((R["audit"].verdict == "moved to another bank").sum()) < 12, f"{int((R['audit'].verdict == 'moved to another bank').sum())}"),
    ("P4", "≥ 30% of reached clients have a sustained alert 3+ months ahead", 0.50, lambda: R["lead3_sus"] >= 0.30, f"{R['lead3_sus']:.1%}"),
    ("P5", "≥ 15% of reached clients reached only by one-month blips", 0.40, lambda: R["single_only"] >= 0.15, f"{R['single_only']:.1%}"),
    ("P6", "Dollars reached (1,000 calls, α=0.5) within ±5% of $11.82bn", 0.70, lambda: abs(R["main_reached"] / 11.82e9 - 1) <= 0.05,
     f"${R['main_reached'] / 1e9:,.2f}bn"),
    ("P7", "Best ÷ worst month precision ≤ 2×", 0.65, lambda: R["prec_ratio"] <= 2, f"{R['prec_ratio']:.2f}×")]
rows = []
for pid, txt, conf, fn, obs in PRED:
    try: hit = bool(fn())
    except Exception: hit = None
    rows.append({"id": pid, "prediction": txt, "confidence": conf, "observed": obs,
                 "result": "HELD" if hit else "FAILED" if hit is False else "n/a",
                 "brier": (conf - float(hit)) ** 2 if hit is not None else np.nan})
PS = pd.DataFrame(rows); R["preds"] = PS
B.table(PS, "12a · Pre-registered predictions", pct=("confidence",), save="m4_12a_predictions")

def img(name):
    return f"<img src='data:image/png;base64,{base64.b64encode(open(CHARTS[name], 'rb').read()).decode()}' style='max-width:100%'>"
def tab(df, pct=(), money=(), rows=60):
    return fmt_df(df.head(rows), pct, money).to_html(index=False, border=0, classes="t", escape=True)
usd = lambda v: f"${v / 1e9:,.2f}bn" if abs(v) >= 1e9 else f"${v / 1e6:,.1f}m" if abs(v) >= 1e6 else f"${v:,.0f}"
HVd = R["headline"]
S = [f"""<h1>Deposit attrition early warning — v13 model 4</h1>
<p class='sub'>Payment Knowledge Graph (PKG) · Treasury Management · DSI · test months {R['test_months'][0]} … {R['test_months'][-1]} ·
primary model: {R['primary']}{f' ({R["engine"]})' if R['engine'] else ''}</p>"""]
def _fnum(df, col, row_match, key="dollars"):
    try: return float(df.loc[df[key] == row_match, col].iloc[0])
    except Exception: return np.nan
_early = R["tim"][(R["tim"].kind == "early signal") & R["tim"].onset_month.notna() & R["tim"].set.str.startswith("payment")].sort_values("onset_month")
_early_txt = "; ".join(f"{l.split(' — ')[0][:60]} {d} (from {int(m)} months)" for l, d, m in
                       zip(_early.label.head(3), _early.leavers.head(3), _early.onset_month.head(3))) or "no payment change crossed the bar"
_n_in = int(_early.feature.head(3).str.contains("_in_|_in$|_in__").sum()) if len(_early) else 0
_side = "mostly on the receiving side" if _n_in >= 2 else "mostly on the paying side" if len(_early) and _n_in <= 1 else ""
def _ls(feat, col):
    try: return float(R["ls"].loc[R["ls"].feature == label(feat), col].iloc[0])
    except Exception: return np.nan
_bn3 = _ls("dep_bal_vs_norm", "leavers -3") - _ls("dep_bal_vs_norm", "stayers -3")
_out1 = _ls("amt_all_out__d6", "leavers -1")
_bal_txt = ("The balance itself stays close to stayers' level until the final month" if pd.notna(_bn3) and abs(_bn3) < 0.1
            else "The balance falls below stayers' level in the months before the exit" if pd.notna(_bn3) else "")
_bal_txt += (", when the money goes out as a payment." if pd.notna(_out1) and _out1 > 0.5 else "." if _bal_txt else "")
_fo = R["folds"]; _ahead = int((_fo.auc_tree > _fo.auc_simple).sum()) if R["engine"] else 0
_better = R["engine"] and R["auc_gbm"] > R["auc_lin"]
_bv = R.get("ls")
_cap = float(HVd.reached_per_year.iloc[-1]); _top5 = float(HVd.reached_per_year.iloc[2]) if len(HVd) > 2 else np.nan
_lo = min(_cap, _top5) if pd.notna(_top5) else _cap
_ac = R["acc"]; _pr = float(_ac[(_ac.model == MAIN) & (_ac.alpha == A_MAIN) & (_ac.calls == K_MAIN)].precision_mean.iloc[0])
_prs = float(_ac[(_ac.model == "simple") & (_ac.alpha == A_MAIN) & (_ac.calls == K_MAIN)].precision_mean.iloc[0])
_vc = R["audit"].verdict.value_counts()
FINDINGS = f"""
<ol class='find'>
<li><b>About {R['exits_per_year']:,.0f} Treasury Management clients a year move their money out, holding {usd(R['at_risk_year'])} of normal balance.</b>
{R['bal_exit_share']:.0%} of those dollars leave through <i>balance exits</i>: the client drains its balance and leaves the accounts open, which a
closure-based measure never counts. The dollars are concentrated: the largest single client is {R['top_client_share']:.0%} of the total.</li>
<li><b>Payment behaviour changes before the balance moves.</b> Against stayers of the same size, the first changes to cross the bar are
{_side + " — " if _side else ""}{_early_txt}. {_bal_txt}
Leavers' scores rise towards the exit (gap to stayers 6 months out = {R['gap_ratio']:.0%} of the gap 1 month out).</li>
<li><b>The tree model {"ranks leavers better than" if _better else "does not beat"} the simple model</b>: AUC {R['auc_gbm']:.3f} against {R['auc_lin']:.3f},
ahead in {_ahead} of {len(_fo)} test months; precision at {K_MAIN:,} calls a month {_pr:.1%} against {_prs:.1%}. The dollar difference
({usd(R['diff'])} a year, 95% {usd(R['diff_ci'][0])} to {usd(R['diff_ci'][1])}) is {"significant" if R['diff_ci'][0] > 0 else "not yet proven"}.</li>
<li><b>{K_MAIN:,} calls a month reach {usd(R['main_reached'])} a year of departing balance ({R['main_share']:.0%} of it)</b>, with {usd(_lo)} as the
cautious figure (largest five clients removed, or each client's credit capped at ${CAP_CREDIT / 1e6:,.0f}m). What that saves depends on the save
rate, which only a pilot can measure: at 5% it is {usd(_lo * 0.05)}–{usd(R['main_reached'] * 0.05)} a year, at 20% {usd(_lo * 0.20)}–{usd(R['main_reached'] * 0.20)}.</li>
<li><b>How early the list warns.</b> {R['lead3_sus_usd']:.0%} of the dollars reached had a sustained alert (two
consecutive months on the list) three or more months before the money left ({R['lead3_sus']:.0%} of the clients reached);
{R['single_only']:.0%} of reached clients appeared on the list only in one-month blips.</li>
<li><b>The largest departures were checked in the data.</b> Of {R['n_audited']} checked: {_vc.get('departed', 0)} departed,
{_vc.get('moved to another bank', 0)} moved their money to their own accounts at another bank, {_vc.get('still transacting', 0)} still transact here,
{_vc.get('moved within PNC', 0)} moved within PNC and {_vc.get('came back', 0)} came back. The last two groups are not losses; all need an RM's confirmation.</li>
</ol>"""
S.append(f"""<section><h2>1 · What we found</h2><div class='box warn'><b>Draft, generated from this run.</b> Review the wording and the
example clients with the RM leads before circulating.</div>{FINDINGS}
<h3>Headline numbers</h3>{tab(pd.DataFrame([
 ["clients whose money left, per year (after the §7 check)", f"{R['exits_per_year']:,.0f}"],
 ["their normal balance, per year", usd(R['at_risk_year'])], ["…share leaving through balance exits (accounts left open)", f"{R['bal_exit_share']:.1%}"],
 ["pooled AUC, simple / tree", f"{R['auc_lin']:.3f} / {R['auc_gbm']:.3f}" if R['engine'] else f"{R['auc_lin']:.3f} / —"],
 [f"dollars reached per year, {K_MAIN:,} calls, α = {A_MAIN}", f"{usd(R['main_reached'])} (95% {usd(R['main_ci'][0])} – {usd(R['main_ci'][1])})"],
 ["…without the largest 5 departing clients", usd(_top5) if pd.notna(_top5) else "—"],
 [f"…with each client's credit capped at ${CAP_CREDIT / 1e6:,.0f}m", usd(_cap)],
 ["tree − simple, dollars reached per year", f"{usd(R['diff'])} (95% {usd(R['diff_ci'][0])} – {usd(R['diff_ci'][1])})" if R['engine'] else "—"],
 ["sustained alert 3+ months before the money left", f"{R['lead3_sus']:.1%} of reached clients, {R['lead3_sus_usd']:.1%} of dollars"],
 ["reached only by one-month blips", f"{R['single_only']:.1%} of reached clients"],
 ["leavers' score gap 6 months out ÷ 1 month out", f"{R['gap_ratio']:.2f}"]], columns=["measure", "value"]))}</section>""")
S.append("""<section><h2>2 · The question</h2><p>Which Treasury Management deposit clients will move their money out in the
next six months, early enough for a relationship manager to act; how much of that money a fixed number of calls can
reach; and how much a stronger model adds. Model 4 corrects model 1, which had mostly learned which <i>kind</i> of
client leaves (small, quiet) instead of what <i>changes</i> before a client leaves.</p></section>""")
S.append(f"""<section><h2>3 · Data and preparation</h2>
<h3>Sources</h3>{tab(pd.DataFrame([[DEP_TABLE, "daily balances and status per account × leg"], [MET_TABLE, f"{R['n_metrics']} monthly payment metrics per customer"]], columns=["table", "content"]))}
<h3>Pipeline</h3><ol><li>Legs summed per account-day; balances rolled to customer-day, then month; both status vocabularies mapped to live / closed.</li>
<li>A complete month grid per customer: a month with no rows is non-live with zero balance.</li>
<li>Payment metrics joined on the power id ({R['key_match']:.1%} of deposit customers found); the largest entity per power id and month.</li>
<li>Rows kept: <b>money still here</b> only. Payment levels become gaps to size peers (same month, same decile of normal
balance); changes vs months −6…−4 and vs the last 3 months; three size anchors.</li>
<li>{R['n_dupes']} near-duplicate features dropped (|r| &gt; {CORR_MAX}); one missing-share column per feature set in the simple model;
the tree model reads missing values natively.</li></ol>
<h3>Silent months</h3><p>No payment row after the client's first row = zero activity (dollars and counts 0; shares empty).
{R['pay_seen']:.1%} of live client-months have payment history; {R['pay_silent']:.1%} are silent.</p>
<h3>Checks</h3>{tab(pd.DataFrame(CHECKS))}</section>""")
S.append(f"""<section><h2>4 · Definitions</h2><h3>The target, and why</h3>
<div class='box'><b>The money leaves within the next 6 months</b>: the earlier of a <b>full exit</b> (every account non-live,
confirmed by {CONFIRM_M} observed non-live months) and a <b>balance exit</b> (balance below {BAL_EXIT_FRAC:.0%} of normal for
{BAL_EXIT_HOLD} months while accounts stay open).</div><ul>
<li><b>It is what costs money:</b> {R['bal_exit_share']:.0%} of departing normal balance leaves through balance exits, which a closure-only target never sees.</li>
<li><b>It is earlier:</b> the drain usually precedes the formal closure.</li>
<li><b>Full exits are absorbing:</b> {R['p_return']:.2%} of clients non-live for {CONFIRM_M}+ months came back.</li>
<li><b>Six months</b> leaves time to act; months without six confirmed months ahead ({R['last_label']} onwards) are unlabelled.</li></ul>
<h3>Terms</h3>{tab(pd.DataFrame([
 ["normal balance", f"median balance over up to {NORM_WIN} prior live months (needs {NORM_MIN_OBS})"],
 ["money still here", f"live, normal ≥ ${MSH_FLOOR:,.0f}, current ≥ {MSH_FRAC:.0%} of normal — the only rows trained on, scored and called"],
 ["size peers", "same month, same decile of normal balance"], ["capped credit", "lesser of current and normal balance at the alert"],
 ["α", "score = probability × normal balance^α"], ["dollars reached", "capped balance of departing clients at first alert, counted once, annualised"],
 ["savings", "dollars reached × assumed save rate (1–20%); no call cost"]], columns=["term", "meaning"]))}</section>""")
S.append(f"""<section><h2>5 · Training and test sets</h2><p>Rolling monthly folds: each test month's models see only outcomes known then
(origins ≤ test month − 6); {HOLD_FRAC:.0%} of clients are held out (all their months) to choose the ridge penalty
and tree depth on the first fold, stop the trees early and calibrate their probabilities on every fold. Stayers are sampled at {NEG_RATE:.0%} and weighted back.</p>
{tab(R['tune'])}{tab(R['folds'], pct=('mean_p_simple', 'mean_p_tree', 'actual_rate'))}</section>""")
S.append(f"""<section><h2>6 · Feature sets</h2><p>{R['n_feats']} features after de-duplication; no identity columns.</p>
{tab(R['feature_sets'], pct=('median_coverage',))}{tab(R['forms'])}</section>""")
S.append(f"""<section><h2>7 · Methods</h2><h3>Simple model</h3><p>Ridge logistic regression (L2 = {R['L2']:g}, chosen on validation) on
standardised, clipped features plus one missing-share per feature set; damped Newton.</p>
<h3>Tree model</h3><p>{('Gradient-boosted trees (' + R['engine'] + f'), depth {R["depth"]}, learning rate {GBM_ETA}, early stopping on the validation months; exact TreeSHAP gives the reason codes.') if R['engine'] else 'Not run: no tree library installed.'}</p>
<h3>Validation</h3><p>Backward-looking features; labels need six confirmed months; each fold trains only on outcomes known before its test month;
{HOLD_FRAC:.0%} of clients are held out inside the training window for tuning and early stopping; held-out clients from the {CAL_RECENT} most recent
labelled months calibrate the tree (isotonic, capped at {P_CAP:.0%}); test months are scored in full.</p>
<h3>From a score to dollars</h3><p>Money-still-here clients ranked by p × normal^α; top K called; each departing client credited once, at its first alert, with its capped balance.</p>
<h3>Uncertainty and concentration</h3><p>Per-month accuracy; bootstrap over departing clients for the dollars and for the tree-vs-simple difference; dollars shown without the largest clients and after the §7 check.</p></section>""")
S.append(f"""<section><h2>8 · Results</h2>
<h3>8.1 How much is at risk</h3><p>{R['exits_per_year']:,.0f} clients a year move their money out, with {usd(R['at_risk_year'])} of normal balance
({R['bal_exit_share']:.0%} through balance exits). The largest single client is {R['top_client_share']:.1%} of it.</p>{tab(R['at_risk_tab'], money=('normal_balance', 'per_year'))}
<h3>8.2 Checking the largest departures</h3>{tab(R['audit_counts'], money=('normal',))}{tab(R['audit'], pct=('balance_after_vs_normal', 'siblings_absorbed', 'activity_after_vs_before', 'own_name_out_vs_drop'), money=('normal', 'test_pool_capped'))}
<h3>8.3 Which features predict</h3>{img('m4_6_importance')}{tab(R['form_imp'], pct=('share', 'share_simple'))}
{tab(R['imp'].head(25), pct=('share', 'share_simple'))}{tab(R['fam_imp'], pct=('share', 'share_simple'))}
<h3>8.4 Simple vs tree model</h3>{tab(R['cmp'], pct=tuple(c for c in R['cmp'].columns if c.startswith('precision_')), money=tuple(c for c in R['cmp'].columns if c in ('simple', 'tree')))}
<h3>8.5 Accuracy against calls</h3>{img('m4_8_accuracy_vs_calls')}{tab(R['acc'], pct=('precision_mean', 'precision_min', 'precision_max', 'recall_clients', 'recall_dollars'))}
<h3>8.6 Accuracy by test month</h3>{img('m4_8_by_month')}{tab(R['bym'], pct=tuple(c for c in R['bym'].columns if c.startswith('precision_')))}
<h3>8.7 Calibration</h3>{img('m4_8_calibration')}{tab(R['cal'], pct=('mean_predicted', 'actual'))}<h4>Simple model</h4>{tab(R['cal_simple'], pct=('mean_predicted', 'actual'))}
<h3>8.8 When signals fire</h3><p>Event time describes leavers after the fact; queue time is what an RM experiences.</p>
{img('m4_9a_onset_heatmap')}{tab(R['tim'], pct=('coverage',))}<h4>Model view</h4>{img('m4_9b_total_gap')}{tab(R['model_view'])}
<p>Largest drivers 6 months out: {', '.join(R['lead6'])}. 1 month out: {', '.join(R['lead1'])}.</p>
<h4>Queue time</h4><p>First alert of any kind:</p>{tab(R['lead'], pct=('share_clients', 'share_dollars'), money=('capped',))}
<p>Sustained alert (first of ≥ 2 consecutive months on the list):</p>{tab(R['lead_sus'], pct=('share_clients', 'share_dollars'), money=('capped',))}{tab(R['reason_sets'], pct=('share',), money=('capped',))}
<h3>8.9 Savings against calls</h3>{img('m4_8_savings_vs_calls')}{tab(HVd, pct=('share',), money=('reached_per_year', 'departing_per_year'))}
{tab(R['val'], pct=('share_of_departing', 'share_of_dollars'), money=tuple(['dollars_reached_year'] + [f'save_{int(s * 100)}pct_year' for s in SAVE_RATES]))}
<h3>8.10 Leavers and stayers</h3>{img('m4_9d_leavers_vs_stayers')}{tab(R['ls'])}
<h3>8.11 Examples for RM managers</h3><p>Chosen by rule; alert = sustained alert; departing examples passed the §7 check and had 12 steady
months before the alert (except <i>new money that left</i>). Balances in dollars; change signals with ▼/▲ where beyond the stayers' threshold.
Review with the client's RM before wider use.</p>""" +
"".join(f"<h4>Example {e['n']} · {e['kind']} · client {e['client']}</h4><p>{_html.escape(e['text'])}</p>{img(e['chart'])}"
        f"{tab(e['table'], pct=('balance_pct_normal', 'score'), money=('balance', 'normal'))}" for e in R["examples"]) + "</section>")
S.append(f"""<section><h2>9 · Predictions written before the run</h2>{tab(R['preds'], pct=('confidence',))}</section>
<section><h2>10 · Limitations</h2><ul><li><b>The save rate is unmeasured</b>; savings are illustrations until a pilot with a holdout.</li>
<li><b>The §7 check is a rule, not a review</b>; flagged clients need an RM's confirmation, so the headline keeps them and the exclusion is shown as a sensitivity row.</li>
<li><b>Identity is current-state:</b> a customer's power id and relationship are today's.</li>
<li><b>Few test months</b> ({R['n_test']}): the data start in 2024 and every label needs six confirmed months.</li>
<li><b>Event-time onsets describe leavers after the fact</b>; they are not detection leads.</li></ul></section>
<section><h2>11 · Next steps</h2><ul><li>RM review of the largest departures and the example clients.</li>
<li>A pilot with a randomised holdout to measure the save rate.</li><li>Tune the tree model further only if the dollars-reached interval says it pays.</li></ul></section>
<section><h2>Appendix · Parameters</h2>{tab(pd.DataFrame([[k, str(v)] for k, v in [("H", H), ("CONFIRM_M", CONFIRM_M), ("NORM_WIN / MIN_OBS", f"{NORM_WIN} / {NORM_MIN_OBS}"),
 ("money still here", f"{MSH_FRAC} × normal, ≥ ${MSH_FLOOR:,}"), ("balance exit", f"< {BAL_EXIT_FRAC} × normal for {BAL_EXIT_HOLD} months"),
 ("NEG_RATE", NEG_RATE), ("L2 chosen", R['L2']), ("tree depth chosen", R['depth']), ("CORR_MAX", CORR_MAX), ("CALLS", CALLS),
 ("ALPHAS", ALPHAS), ("SAVE_RATES", SAVE_RATES), ("LIFT_BAR", LIFT_BAR), ("N_AUDIT", N_AUDIT)]], columns=["parameter", "value"]))}</section>""")
CSS = """body{font-family:Arial,Helvetica,sans-serif;max-width:1180px;margin:24px auto;padding:0 18px;color:#222;line-height:1.45}
h1{font-size:26px;margin-bottom:4px}.sub{color:#666;margin-top:0}h2{border-bottom:2px solid #1f4e79;padding-bottom:4px;margin-top:34px}
h3{color:#1f4e79;margin-top:22px}table.t{border-collapse:collapse;font-size:12px;margin:8px 0;display:block;overflow-x:auto}
table.t th{background:#1f4e79;color:#fff;padding:4px 8px;text-align:left}table.t td{padding:3px 8px;border-bottom:1px solid #e3e3e3}
ol.find li{margin:8px 0}.box{background:#eef4fa;border-left:4px solid #1f4e79;padding:10px 14px;margin:10px 0}.box.warn{background:#fff4e5;border-color:#e65100}"""
REPORT = OUT / "PKG_Attrition_v13_m4_report.html"
REPORT.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>PKG attrition v13 · model 4</title><style>{CSS}</style></head>"
                  f"<body>{''.join(S)}<p style='color:#999;font-size:11px'>Internal — PNC Treasury Management, DSI</p></body></html>",
                  encoding="utf-8")
B.kv([("report", str(REPORT.resolve())), ("size", f"{REPORT.stat().st_size / 1e6:,.1f} MB"), ("CSVs and charts", str(OUT.resolve())), ("figure data", str(FIG.resolve())), ("results bundle", "m4_results.pkl"),
      ("total wall time", f"{(time.time() - T0) / 60:,.1f} min")], "12b · Written")
import pickle
R["sus_ids"] = None
with open(OUT / "m4_results.pkl", "wb") as fh:          # everything the report uses, for rebuilding it later
    pickle.dump({"R": {k: v for k, v in R.items() if k != "excl_ids"}, "checks": CHECKS,
                 "charts": {k: str(v) for k, v in CHARTS.items()}}, fh)
for d_ in (X, FEAT, MF, AMP): d_.unpersist()
B.show(t0)